# Notebook 08: Autoencoders - Learning to Compress and Reconstruct

---

## What This Notebook Covers

This notebook introduces **Autoencoders**, a type of neural network that learns to compress data into a smaller representation and then reconstruct it. We will learn:

1. **What is an Autoencoder?** - The concept and architecture
2. **Why use Autoencoders?** - Applications and use cases
3. **Fashion MNIST Dataset** - Loading and exploring the data
4. **CNN Classification (Warmup)** - Review of convolutional networks
5. **Building an Autoencoder** - Encoder and Decoder design
6. **Deconvolution (Upsampling)** - How to increase image size
7. **Training and Evaluation** - MSE loss for reconstruction

---

## What is an Autoencoder?

An **Autoencoder** is a neural network that learns to:
1. **Encode**: Compress input data into a smaller representation (called the **latent space** or **bottleneck**)
2. **Decode**: Reconstruct the original input from this compressed representation

```
Input Image          Encoder          Latent Space         Decoder         Reconstructed Image
   28×28      →    [Compress]    →      Small       →    [Expand]     →        28×28
   784 pixels      (CNN layers)      (e.g., 64 values)   (Deconv layers)      784 pixels
```

**The key insight:** The network is forced to learn the most important features of the data because it must squeeze everything through a small bottleneck!

---

## Why Use Autoencoders?

| Application | How It Works |
|-------------|-------------|
| **Dimensionality Reduction** | The latent space is a compressed representation |
| **Denoising** | Train on noisy inputs, reconstruct clean outputs |
| **Anomaly Detection** | Unusual inputs have high reconstruction error |
| **Feature Learning** | The encoder learns useful representations |
| **Generative Models** | Variational Autoencoders (VAEs) can generate new data |

---

---

## 🎬 Interactive Visualizations in This Notebook

This notebook now includes **six hands-on interactive visualizations** that you can click, drag, step through, and play. Each one targets a concept that is much easier to *see* than to read about:

| # | Visualization | What it makes clear |
|---|---------------|---------------------|
| 1 | **Autoencoder Pipeline Explorer** | The whole squeeze-then-rebuild story end to end, with the 784 → 256 → 784 value counts |
| 2 | **Convolution Stride Lab** | How `stride=2` halves the image and how filters create channels |
| 3 | **Deconv Upsample Lab** | The two-step `upsample → conv` recipe, plus the checkerboard problem |
| 4 | **"Same" Padding Playground** | Why `padding = ks//2` keeps the spatial size identical |
| 5 | **3D Architecture Hourglass** | The tensor volumes shrinking and expanding in 3D (drag to rotate) |
| 6 | **Latent Space Explorer** | Dragging through the compressed code and watching reconstructions morph |

Most have **clickable stages**, a **speed slider**, **Play/Step** controls, and a **code panel** that updates to show the exact line running at each stage. Just run the cell beneath each section. *(They are self-contained and work offline.)*

---

### 🔍 Visualize It: The Full Autoencoder Pipeline

Before any code, here is the big picture. Step through the five stages (or press **Play**) to watch an image get squeezed through the bottleneck and rebuilt. Click any stage on the rail to jump to it and see the code for that stage.

In [ ]:
# ============================================================================
# INTERACTIVE VISUALIZATION (self-contained) -- run this cell.
# End-to-end: Input -> Encoder -> Bottleneck -> Decoder -> Output, with live value counts.
# The widget lives in the separate file `ae_pipeline_explorer.html`; it is embedded here as a
# base64 data-URI iframe, so the notebook stays fully self-contained and works
# offline in Jupyter, Colab, VS Code, and the exported HTML. The iframe
# broadcasts its own content height, and the listener below resizes it to fit
# exactly -- full width, wrapped to content, with no empty space below.
# ============================================================================
from IPython.display import HTML
import base64

_html_b64 = "..."

HTML('''
<iframe id="ae-pipeline-frame"
        src="data:text/html;base64,''' + _html_b64 + '''"
        style="width:100%; height:780px; border:1px solid #DDE5F2;
               border-radius:16px; box-shadow:0 8px 24px rgba(108,92,231,.12);
               display:block;"
        loading="lazy" title="Autoencoder pipeline explorer"></iframe>
<script>
(function(){
  function onMsg(e){
    if (e.data && e.data.type === "ae-frame-height") {
      var f = document.getElementById("ae-pipeline-frame");
      if (f) f.style.height = (e.data.height) + "px";
    }
  }
  window.addEventListener("message", onMsg);
})();
</script>
''')


# Part 1: Setup and Imports

---

First, let's import all the libraries we need.

In [ ]:
# =====================================================
# STANDARD PYTHON LIBRARIES
# =====================================================

import pickle          # For loading/saving Python objects
import gzip            # For compressed files
import math            # Mathematical functions
import os              # Operating system interface
import time            # Timing operations
import shutil          # File operations

# =====================================================
# DATA SCIENCE LIBRARIES
# =====================================================

import numpy as np                    # Numerical computing
import matplotlib as mpl              # Plotting configuration
import matplotlib.pyplot as plt       # Creating plots

# =====================================================
# PYTORCH LIBRARIES
# =====================================================

import torch                          # Main PyTorch library
from torch import tensor, nn, optim   # Tensors, neural networks, optimizers
from torch.utils.data import DataLoader, default_collate  # Data loading utilities
import torch.nn.functional as F       # Functional operations (loss functions, etc.)
import torchvision.transforms.functional as TF  # Image transformations

# =====================================================
# HUGGING FACE DATASETS
# =====================================================

# datasets is a library from Hugging Face that provides easy access
# to many popular datasets, including Fashion MNIST
from datasets import load_dataset, load_dataset_builder

# =====================================================
# UTILITY LIBRARIES
# =====================================================

from pathlib import Path              # Modern path handling
from operator import attrgetter, itemgetter  # Efficient attribute/item access
from functools import partial         # Partial function application
from collections.abc import Mapping   # For type checking dictionaries

# =====================================================
# FASTAI UTILITIES (from previous notebooks)
# =====================================================

# These are helper functions from the fastai library and our miniai module
import fastcore.all as fc
from fastcore.test import test_close  # Testing utility
from fastprogress import progress_bar, master_bar  # Progress bars

In [ ]:
# =====================================================
# CONFIGURATION AND SETTINGS
# =====================================================

# Set how PyTorch displays tensors:
#   precision=2     - Show 2 decimal places
#   linewidth=140   - Wider lines before wrapping
#   sci_mode=False  - Don't use scientific notation
torch.set_printoptions(precision=2, linewidth=140, sci_mode=False)

# Set random seed for reproducibility
# This ensures we get the same "random" results each time
torch.manual_seed(1)

# Use grayscale colormap for images (Fashion MNIST is black and white)
mpl.rcParams['image.cmap'] = 'gray'

# Disable warning messages from the datasets library
import logging
logging.disable(logging.WARNING)

In [ ]:
# =====================================================
# DEVICE SELECTION (GPU IF AVAILABLE)
# =====================================================

# Choose the best available device:
#   - MPS: Apple Silicon GPU (M1/M2 Macs)
#   - CUDA: NVIDIA GPU
#   - CPU: Fallback if no GPU available

if torch.backends.mps.is_available():
    def_device = 'mps'
elif torch.cuda.is_available():
    def_device = 'cuda'
else:
    def_device = 'cpu'

print(f"Using device: {def_device}")

In [ ]:
# =====================================================
# HELPER FUNCTIONS (from previous notebooks)
# =====================================================

def to_device(x, device=def_device):
    """
    Move a tensor or collection of tensors to the specified device.
    
    Arguments:
        x      - A tensor, dictionary, or iterable of tensors
        device - The target device ('cuda', 'mps', or 'cpu')
    
    Returns:
        The same structure with all tensors moved to the device
    """
    if isinstance(x, torch.Tensor):
        return x.to(device)
    if isinstance(x, Mapping):
        return {k: v.to(device) for k, v in x.items()}
    return type(x)(to_device(o, device) for o in x)


def conv(ni, nf, ks=3, stride=2, act=True):
    """
    Create a convolution layer with optional ReLU activation.
    
    This is the same helper function from the convolutions notebook.
    
    Arguments:
        ni     - Number of input channels
        nf     - Number of output channels (filters)
        ks     - Kernel size (default 3 for 3x3)
        stride - Stride (default 2 to halve spatial size)
        act    - Whether to add ReLU activation (default True)
    
    Returns:
        A Conv2d layer, optionally wrapped with ReLU
    """
    res = nn.Conv2d(ni, nf, stride=stride, kernel_size=ks, padding=ks//2)
    if act:
        res = nn.Sequential(res, nn.ReLU())
    return res


def show_image(im, ax=None, figsize=(3,3), title=None, noframe=True, cmap=None):
    """Display a single image."""
    if ax is None:
        fig, ax = plt.subplots(figsize=figsize)
    if hasattr(im, 'cpu'):
        im = im.detach().cpu()
    if hasattr(im, 'numpy'):
        im = im.numpy()
    if im.ndim == 3 and im.shape[0] in [1, 3]:
        im = im.transpose(1, 2, 0)
    if im.ndim == 3 and im.shape[2] == 1:
        im = im[:, :, 0]
    ax.imshow(im, cmap=cmap)
    if title is not None:
        ax.set_title(title)
    if noframe:
        ax.axis('off')
    return ax


def show_images(ims, nrows=1, ncols=None, titles=None, figsize=None, imsize=3):
    """Display multiple images in a grid."""
    if ncols is None:
        ncols = len(ims)
    if figsize is None:
        figsize = (ncols * imsize, nrows * imsize)
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
    if nrows * ncols == 1:
        axes = [axes]
    else:
        axes = axes.flatten()
    for i, (im, ax) in enumerate(zip(ims, axes)):
        title = titles[i] if titles else None
        show_image(im, ax=ax, title=title)
    plt.tight_layout()
    return fig

---

# Part 2: Loading the Fashion MNIST Dataset

---

## What is Fashion MNIST?

**Fashion MNIST** is a dataset of clothing images, designed as a drop-in replacement for the classic MNIST digit dataset. It's more challenging while having the same format:

- **60,000 training images** and **10,000 test images**
- **28 × 28 pixels** each, grayscale
- **10 classes** of clothing items

| Label | Description |
|-------|-------------|
| 0 | T-shirt/top |
| 1 | Trouser |
| 2 | Pullover |
| 3 | Dress |
| 4 | Coat |
| 5 | Sandal |
| 6 | Shirt |
| 7 | Sneaker |
| 8 | Bag |
| 9 | Ankle boot |

---

In [ ]:
# =====================================================
# LOADING FASHION MNIST WITH HUGGING FACE DATASETS
# =====================================================

# Define the column names in the dataset
x = 'image'   # The column containing images
y = 'label'   # The column containing labels (0-9)

# Name of the dataset on Hugging Face
name = "fashion_mnist"

# load_dataset downloads and caches the dataset
# ignore_verifications=True skips hash verification (faster loading)
#
# Returns a DatasetDict with 'train' and 'test' splits
dsd = load_dataset(name, ignore_verifications=True)

print(f"Dataset loaded: {name}")
print(f"Splits available: {list(dsd.keys())}")
print(f"Training samples: {len(dsd['train'])}")
print(f"Test samples: {len(dsd['test'])}")

In [ ]:
# Let's look at what one sample looks like
sample = dsd['train'][0]
print("Sample structure:")
print(f"  Keys: {sample.keys()}")
print(f"  Image type: {type(sample['image'])}")
print(f"  Label: {sample['label']}")

---

## Transforming the Data

The images come as PIL Image objects. We need to convert them to PyTorch tensors for training.

In [ ]:
# =====================================================
# DEFINING THE TRANSFORMATION
# =====================================================

# The @inplace decorator modifies the input dictionary directly
# instead of returning a new one. This saves memory.
#
# If you don't have the inplace decorator, you can define it:
def inplace(f):
    """Decorator that makes a function modify its argument in place."""
    def wrapper(b):
        f(b)
        return b
    return wrapper

@inplace
def transformi(b):
    """
    Transform a batch of data by converting images to tensors.
    
    Arguments:
        b - A dictionary with 'image' key containing PIL Images
    
    Modifies b in place:
        - Converts each PIL Image to a PyTorch tensor
        - Tensor values are scaled to [0, 1] range
        - Shape becomes (1, 28, 28) - 1 channel, 28x28 pixels
    
    TF.to_tensor does several things:
        1. Converts PIL Image to PyTorch tensor
        2. Rearranges from (H, W, C) to (C, H, W)
        3. Scales pixel values from [0, 255] to [0.0, 1.0]
    """
    b[x] = [TF.to_tensor(o) for o in b[x]]

In [ ]:
# =====================================================
# APPLYING THE TRANSFORMATION
# =====================================================

# Batch size - how many samples to process at once
bs = 256

# with_transform applies our transformation lazily (when data is accessed)
# This is memory-efficient: images are converted only when needed
tds = dsd.with_transform(transformi)

print("Transformed dataset created.")
print("Transformation will be applied when data is accessed.")

In [ ]:
# =====================================================
# EXAMINING A TRANSFORMED SAMPLE
# =====================================================

# Get the training split
ds = tds['train']

# Get the first image (now it's a tensor!)
img = ds[0]['image']

print(f"Image type: {type(img)}")
print(f"Image shape: {img.shape}")
print(f"  - {img.shape[0]} channel (grayscale)")
print(f"  - {img.shape[1]}x{img.shape[2]} pixels")
print(f"Pixel value range: {img.min():.2f} to {img.max():.2f}")

# Display the image
show_image(img, figsize=(2, 2))

---

## Creating Data Loaders

Data loaders batch the data and handle shuffling for training.

In [ ]:
# =====================================================
# COLLATE FUNCTION
# =====================================================

# A collate function takes a list of samples and combines them into a batch.
# 
# The Hugging Face dataset returns dictionaries like:
#   {'image': tensor, 'label': int}
#
# We need to:
#   1. Stack all images into a single tensor
#   2. Stack all labels into a single tensor
#   3. Move everything to the GPU

def collate_dict(ds):
    """
    Create a collate function for dictionary-based datasets.
    
    Returns a function that:
        1. Extracts 'image' and 'label' from each sample
        2. Stacks them into batch tensors
    """
    def _collate(batch):
        # Stack all images: list of (1,28,28) -> (batch_size, 1, 28, 28)
        images = torch.stack([b['image'] for b in batch])
        # Stack all labels: list of ints -> (batch_size,)
        labels = torch.tensor([b['label'] for b in batch])
        return images, labels
    return _collate

# Create the collate function for our dataset
cf = collate_dict(ds)

### Understanding the Collate Function: Closure Pattern

#### What happens with `cf = collate_dict(ds)`?

`collate_dict` is **already defined** — calling it doesn't *create* the function. Instead, it **calls** `collate_dict`, which **returns the inner function `_collate`**.

So after this line, `cf` **is** the `_collate` function itself — it's a **function object**, not a computed result.

```python
cf = collate_dict(ds)
# cf IS _collate — it's a callable function, not a data result
```

> **Note:** In the current implementation, the `ds` argument isn't actually used inside `_collate`. It's likely there for future flexibility or was used in an earlier version of the code.

---

#### What happens with `cf(b)` inside `collate_`?

Since `cf` *is* the `_collate` function, calling `cf(b)` is identical to calling `_collate(b)`:

1. It takes a **batch** `b` — a list of dictionaries like `[{'image': tensor, 'label': int}, ...]`
2. It **stacks** all images into a single tensor of shape `(batch_size, 1, 28, 28)`
3. It **stacks** all labels into a single tensor of shape `(batch_size,)`
4. It returns the tuple `(images, labels)`

The full chain inside `collate_(b)` works like this:

```
b (list of dicts)
    │
    ▼
cf(b)  →  _collate(b)  →  (stacked_images, stacked_labels)
    │
    ▼
to_device(...)  →  moves both tensors to GPU
    │
    ▼
returns (images_on_gpu, labels_on_gpu)
```

---

#### Why This Pattern? — The Factory Function

This is a **factory function** (also called a **closure**) — a function that **builds and returns another function**.

**Why is it useful here?**

PyTorch's `DataLoader` expects a collate function with the signature `collate(batch)` — just **one argument**. By wrapping the logic inside a factory:

- You can **bake in configuration** (like dataset structure, column names, etc.) at creation time
- You still hand the `DataLoader` a **clean, single-argument function** that it knows how to call

```python
# Factory creates a configured function
cf = collate_dict(ds)        # returns _collate, configured for dict-based datasets

# That function is then used inside another function
def collate_(b):
    return to_device(cf(b))  # stack into batch → move to GPU

# DataLoader only sees a simple collate_(batch) signature
dl = DataLoader(dataset, collate_fn=collate_)
```

This is a common and elegant pattern in Python for **composing behavior** while keeping interfaces clean.

In [ ]:
# =====================================================
# DATA LOADERS WITH DEVICE TRANSFER
# =====================================================

def collate_(b):
    """
    Collate function that also moves data to the device (GPU).
    
    Combines cf (which stacks samples) with to_device (which moves to GPU).
    """
    return to_device(cf(b))


def data_loaders(dsd, bs, **kwargs):
    """
    Create DataLoaders for all splits in a dataset dict.
    
    Arguments:
        dsd    - A DatasetDict with 'train', 'test', etc.
        bs     - Batch size
        kwargs - Additional arguments for DataLoader
    
    Returns:
        A dictionary of DataLoaders, one per split
    """
    return {k: DataLoader(v, bs, **kwargs) for k, v in dsd.items()}

# Understanding `data_loaders`: Creating DataLoaders from a DatasetDict

## What does this function do?

A Hugging Face `DatasetDict` contains **multiple splits** of your data — typically `'train'` and `'test'`. This function takes that entire dict and creates a **PyTorch `DataLoader`** for each split in one go.

```python
def data_loaders(dsd, bs, **kwargs):
    return {k: DataLoader(v, bs, **kwargs) for k, v in dsd.items()}
```

---

## Breaking it down step by step

### 1. `dsd.items()` — Iterating over the DatasetDict

`dsd` is a Hugging Face `DatasetDict`, which behaves like a Python dictionary:

```python
dsd = {
    'train': <Dataset with 60000 samples>,
    'test':  <Dataset with 10000 samples>
}
```

Calling `dsd.items()` gives you key-value pairs:

| `k` (key) | `v` (value) |
|---|---|
| `'train'` | The training `Dataset` object |
| `'test'` | The test `Dataset` object |

### 2. `DataLoader(v, bs, **kwargs)` — Wrapping each dataset

For each split, a PyTorch `DataLoader` is created with:

- `v` — the dataset for that split
- `bs` — the batch size
- `**kwargs` — any additional arguments (e.g., `collate_fn=collate_`, `shuffle=True`, `num_workers=4`)

### 3. Dictionary comprehension — Collecting the results

The `{k: ... for k, v in ...}` syntax builds a **new dictionary** that maps each split name to its corresponding `DataLoader`.

---

## What the output looks like

```python
dls = data_loaders(dsd, bs=64, collate_fn=collate_)

# dls is now:
# {
#     'train': DataLoader(train_dataset, batch_size=64, collate_fn=collate_),
#     'test':  DataLoader(test_dataset,  batch_size=64, collate_fn=collate_)
# }

# Access them like:
train_dl = dls['train']
test_dl  = dls['test']
```

---

## Why `**kwargs`?

The `**kwargs` pattern lets you **pass through any extra arguments** to `DataLoader` without hardcoding them. This keeps the function flexible:

```python
# Minimal usage
dls = data_loaders(dsd, bs=64)

# With extra options
dls = data_loaders(dsd, bs=64, collate_fn=collate_, num_workers=4, shuffle=True)
```

All the keyword arguments after `bs` get collected into `kwargs` and forwarded directly to each `DataLoader` constructor via `**kwargs`.

---

## Summary

This is a **convenience wrapper** that avoids writing repetitive code like:

```python
# Without the helper — repetitive
train_dl = DataLoader(dsd['train'], 64, collate_fn=collate_)
test_dl  = DataLoader(dsd['test'],  64, collate_fn=collate_)
```

Instead, one line handles all splits:

```python
# With the helper — clean and scalable
dls = data_loaders(dsd, 64, collate_fn=collate_)
```

If your `DatasetDict` had more splits (e.g., `'train'`, `'validation'`, `'test'`), this function would automatically create a `DataLoader` for each one — no extra code needed.

In [ ]:
# =====================================================
# CREATE THE DATA LOADERS
# =====================================================

# Create loaders for train and test splits
dls = data_loaders(tds, bs, collate_fn=collate_)

print(f"Data loaders created: {list(dls.keys())}")
print(f"Batch size: {bs}")

In [ ]:
# =====================================================
# GET CONVENIENT REFERENCES
# =====================================================

# Shortcuts for training and validation (test) loaders
dt = dls['train']   # Training data loader
dv = dls['test']    # Validation/test data loader

# Get one batch to inspect
# next(iter(dt)) gets the first batch from the training loader
xb, yb = next(iter(dt))

print(f"Batch shapes:")
print(f"  Images (xb): {xb.shape}")
print(f"    - {xb.shape[0]} images in batch")
print(f"    - {xb.shape[1]} channel")
print(f"    - {xb.shape[2]}x{xb.shape[3]} pixels")
print(f"  Labels (yb): {yb.shape}")
print(f"  Device: {xb.device}")

In [ ]:
# =====================================================
# CLASS LABELS
# =====================================================

# The dataset has human-readable names for each class
# ds.features[y] contains metadata about the label column
# .names gives us the list of class names

labels = ds.features[y].names

print("Fashion MNIST Classes:")
for i, label in enumerate(labels):
    print(f"  {i}: {label}")

In [ ]:
# =====================================================
# VISUALIZING A BATCH
# =====================================================

# itemgetter creates a function that extracts items by index
# itemgetter(0, 3, 5) returns a function that returns (seq[0], seq[3], seq[5])
#
# Here we use it to get the label names for our batch
# yb[:16] are the first 16 label indices (e.g., [9, 0, 0, 3, ...])
# We want to convert these to names like ['Ankle boot', 'T-shirt/top', ...]

# *yb[:16] unpacks the tensor to individual arguments
lbl_getter = itemgetter(*yb[:16].tolist())
titles = lbl_getter(labels)

print(f"First 16 labels: {yb[:16].tolist()}")
print(f"Label names: {titles}")

#### Understanding `itemgetter` for Label Lookup

#### The Problem We're Solving

We have:

- `yb[:16]` — a tensor of **16 label indices**, e.g., `[9, 0, 0, 3, 7, ...]`
- `labels` — a list/tuple of **label names**, e.g., `['T-shirt/top', 'Trouser', 'Pullover', ..., 'Ankle boot']`

We want to convert those indices into human-readable names like `['Ankle boot', 'T-shirt/top', 'T-shirt/top', 'Dress', ...]`.

---

#### Line 1: `lbl_getter = itemgetter(*yb[:16].tolist())`

This line does **three things** — let's unpack them from the inside out.

#### Step 1: `yb[:16].tolist()`

Converts the first 16 label indices from a PyTorch tensor to a plain Python list:

```python
yb[:16]           # tensor([9, 0, 0, 3, 7, ...])
yb[:16].tolist()  # [9, 0, 0, 3, 7, ...]
```

#### Step 2: `*` (unpacking operator)

The `*` **unpacks** the list into separate arguments:

```python
itemgetter(*[9, 0, 0, 3, 7])
# is equivalent to:
itemgetter(9, 0, 0, 3, 7)
```

Without `*`, you'd pass a single list as one argument — `itemgetter` would not know what to do with it. With `*`, each index becomes a **separate argument**.

#### Step 3: `itemgetter(9, 0, 0, 3, 7, ...)`

`itemgetter` from Python's `operator` module **creates a function** that, when called on a sequence, extracts items at the specified indices.

```python
lbl_getter = itemgetter(9, 0, 0, 3, 7)
# lbl_getter is now a FUNCTION that does:
#   given a sequence s → return (s[9], s[0], s[0], s[3], s[7])
```

> **Key insight:** `itemgetter` doesn't fetch anything yet — it **builds a reusable lookup function**.

---

#### Line 2: `titles = lbl_getter(labels)`

Now we **call** that function on `labels` (the list of label names):

```python
labels = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
          'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

titles = lbl_getter(labels)
# Executes: (labels[9], labels[0], labels[0], labels[3], labels[7], ...)
# Returns:  ('Ankle boot', 'T-shirt/top', 'T-shirt/top', 'Dress', 'Sneaker', ...)
```

---

#### The Full Flow Visualized

```
yb[:16]                    →  tensor([9, 0, 0, 3, 7, ...])
    │
    ▼  .tolist()
[9, 0, 0, 3, 7, ...]      →  plain Python list
    │
    ▼  * (unpack)
itemgetter(9, 0, 0, 3, 7)  →  creates a FUNCTION
    │
    ▼  call with labels
(labels[9], labels[0], ...) →  ('Ankle boot', 'T-shirt/top', ...)
```

---

## Why not just use a list comprehension?

You absolutely could write this as:

```python
titles = [labels[i] for i in yb[:16].tolist()]
```

Both approaches work. The `itemgetter` version is:

- **Slightly faster** for large sequences (implemented in C)
- **Reusable** — you can call `lbl_getter` on different label lists without rebuilding
- **A common pattern** in fast.ai code, which favors functional programming style

The list comprehension is arguably more readable if you're not used to `itemgetter`, but both produce the same result.

In [ ]:
# =====================================================
# DISPLAY SAMPLE IMAGES
# =====================================================

mpl.rcParams['figure.dpi'] = 70

# Show the first 16 images with their labels
# Need to move to CPU for matplotlib
show_images(xb[:16].cpu(), nrows=2, ncols=8, titles=titles, imsize=1.7)

---

# Part 3: Warmup - CNN Classification

---

Before building our autoencoder, let's verify our setup works by training a simple CNN classifier. This is the same architecture from the convolutions notebook.

**Why do this warmup?**
- Confirms our data pipeline is working
- Verifies GPU is being used correctly
- Reviews the CNN concepts before we modify them for autoencoders

In [ ]:
# =====================================================
# TRAINING HYPERPARAMETERS
# =====================================================

bs = 256    # Batch size (already set, but good to be explicit)
lr = 0.4    # Learning rate for SGD optimizer

In [ ]:
# =====================================================
# CNN CLASSIFIER ARCHITECTURE
# =====================================================

# This is the same CNN from the convolutions notebook:
#   - 5 convolutional layers with stride=2
#   - Each layer halves the spatial dimensions
#   - Final layer outputs 10 values (one per class)
#
# Size progression:
#   28x28 -> 14x14 -> 7x7 -> 4x4 -> 2x2 -> 1x1

cnn = nn.Sequential(
    conv(1, 4),              # 28x28 -> 14x14, 4 channels
    conv(4, 8),              # 14x14 -> 7x7,   8 channels
    conv(8, 16),             # 7x7   -> 4x4,   16 channels
    conv(16, 16),            # 4x4   -> 2x2,   16 channels
    conv(16, 10, act=False), # 2x2   -> 1x1,   10 channels (no ReLU!)
    nn.Flatten()             # (batch, 10, 1, 1) -> (batch, 10)
).to(def_device)

print("CNN Classifier Architecture:")
print(cnn)

# How `conv(1, 4)` Transforms 28×28 → 14×14 with 4 Channels

## The `conv` Function

Here's the `conv` function from the notebook:

```python
def conv(ni, nf, ks=3, stride=2, act=True):
    res = nn.Conv2d(ni, nf, stride=stride, kernel_size=ks, padding=ks//2)
    if act:
        res = nn.Sequential(res, nn.ReLU())
    return res
```

When we call `conv(1, 4)`, we get these parameters:

| Parameter | Value | Meaning |
|-----------|-------|---------|
| `ni` | 1 | Input channels (grayscale image) |
| `nf` | 4 | Output channels (4 filters) |
| `ks` | 3 | Kernel size (3×3) |
| `stride` | 2 | Step size (default) |
| `padding` | 1 | Computed as `ks//2 = 3//2 = 1` |

---

## Two Independent Transformations

The convolution performs two separate operations simultaneously:

1. **Spatial reduction** (28 → 14) — controlled by `stride`
2. **Channel expansion** (1 → 4) — controlled by `nf`

---

## 1. Spatial Reduction: 28 → 14

### Why Does `stride=2` Halve the Size?

With `stride=2`, the kernel jumps 2 pixels at a time instead of 1:

```
stride=1: ● ● ● ● ● ● ● ●  →  kernel visits every position
stride=2: ●   ●   ●   ●    →  kernel skips every other position
```

### The Output Size Formula

$$\text{output size} = \left\lfloor \frac{\text{input} + 2 \times \text{padding} - \text{kernel}}{\text{stride}} \right\rfloor + 1$$

Plugging in our values:

$$\text{output size} = \left\lfloor \frac{28 + 2(1) - 3}{2} \right\rfloor + 1 = \left\lfloor \frac{27}{2} \right\rfloor + 1 = 13 + 1 = 14$$

### Visual Intuition

Think of it like sampling every other pixel:

```
Input row (28 pixels):
[0][1][2][3][4][5][6][7][8][9]...[27]
 ↓     ↓     ↓     ↓     ↓
[0]   [2]   [4]   [6]   [8]  ...  → 14 output positions
```

---

## 2. Channel Expansion: 1 → 4

### What Are Channels?

- **Input channel**: The grayscale image has 1 channel (pixel intensity)
- **Output channels**: Each filter produces one output channel (feature map)

### How Filters Create Channels

Each filter is a learnable 3×3 weight matrix. With 4 filters, we get 4 output channels:

```
                    Filter 0 (3×3) ──→ Channel 0 (14×14)
                         │
Input (1×28×28) ──→ Filter 1 (3×3) ──→ Channel 1 (14×14)
                         │
                    Filter 2 (3×3) ──→ Channel 2 (14×14)
                         │
                    Filter 3 (3×3) ──→ Channel 3 (14×14)
```

### What Do Filters Learn?

During training, each filter learns to detect different features:

| Filter | Might Learn to Detect |
|--------|----------------------|
| Filter 0 | Horizontal edges |
| Filter 1 | Vertical edges |
| Filter 2 | Corners or curves |
| Filter 3 | Textures or patterns |

The network discovers these features automatically through backpropagation.

---

## Visual Summary

```
        INPUT                          OUTPUT
     ┌─────────┐                    ┌───────┐
     │         │                    │ Ch 0  │ 14×14
     │  28×28  │     conv(1,4)      ├───────┤
     │         │  ─────────────→    │ Ch 1  │ 14×14
     │ 1 chan  │   stride=2         ├───────┤
     │         │   4 filters        │ Ch 2  │ 14×14
     └─────────┘                    ├───────┤
                                    │ Ch 3  │ 14×14
                                    └───────┘

      Shape: (1, 28, 28)            Shape: (4, 14, 14)
      Values: 784                   Values: 784
```

Interestingly, the total number of values stays the same (784) in this first layer — we've traded spatial resolution for richer feature representation.

---

## The Full CNN Progression

Here's how the entire CNN classifier transforms the input:

| Layer | Operation | Output Shape | Spatial Size | Channels | Total Values |
|-------|-----------|--------------|--------------|----------|--------------|
| Input | — | (1, 28, 28) | 28×28 | 1 | 784 |
| `conv(1, 4)` | stride=2, 4 filters | (4, 14, 14) | 14×14 | 4 | 784 |
| `conv(4, 8)` | stride=2, 8 filters | (8, 7, 7) | 7×7 | 8 | 392 |
| `conv(8, 16)` | stride=2, 16 filters | (16, 4, 4) | 4×4 | 16 | 256 |
| `conv(16, 16)` | stride=2, 16 filters | (16, 2, 2) | 2×2 | 16 | 64 |
| `conv(16, 10)` | stride=2, 10 filters | (10, 1, 1) | 1×1 | 10 | 10 |
| `Flatten()` | reshape | (10,) | — | — | 10 |

### The Pattern

As we go deeper:
- **Spatial size decreases**: 28 → 14 → 7 → 4 → 2 → 1
- **Channels increase**: 1 → 4 → 8 → 16 → 16 → 10
- **Information compresses**: 784 → 784 → 392 → 256 → 64 → 10

The network learns to extract increasingly abstract features while compressing the representation down to 10 values — one score per Fashion MNIST class.

---

## Key Takeaways

1. **Stride controls spatial reduction**: `stride=2` halves the height and width
2. **Number of filters controls output channels**: `nf=4` means 4 output channels
3. **These are independent**: You can have any combination of stride and filter count
4. **Padding preserves information**: `padding=ks//2` prevents excessive shrinking at borders
5. **CNNs trade space for depth**: Smaller spatial size, more channels = richer features

### 🔍 Visualize It: Stride and Filters

Drag the orange kernel over the input, toggle **stride = 1 vs 2**, and watch the output grid fill in. The formula on the right resolves live, and the lower panel shows how 4 filters produce 4 output channels.

In [ ]:
# ============================================================================
# INTERACTIVE VISUALIZATION (self-contained) -- run this cell.
# Stride controls output size; number of filters controls output channels.
# The widget lives in the separate file `conv_stride_lab.html`; it is embedded here as a
# base64 data-URI iframe, so the notebook stays fully self-contained and works
# offline in Jupyter, Colab, VS Code, and the exported HTML. The iframe
# broadcasts its own content height, and the listener below resizes it to fit
# exactly -- full width, wrapped to content, with no empty space below.
# ============================================================================
from IPython.display import HTML
import base64

_html_b64 = "PCFET0NUWVBFIGh0bWw+CjxodG1sIGxhbmc9ImVuIj4KPGhlYWQ+CjxtZXRhIGNoYXJzZXQ9IlVURi04Ij4KPG1ldGEgbmFtZT0idmlld3BvcnQiIGNvbnRlbnQ9IndpZHRoPWRldmljZS13aWR0aCwgaW5pdGlhbC1zY2FsZT0xLjAiPgo8dGl0bGU+Q29udm9sdXRpb24gU3RyaWRlIExhYjwvdGl0bGU+CjxzdHlsZT4KICA6cm9vdHsKICAgIC0tYmc6I0VERjJGQjsgLS1jYXJkOiNGRkZGRkY7IC0taW5rOiMxRjJBNDQ7IC0tbXV0ZWQ6IzY0NzQ4QjsKICAgIC0tcHVycGxlOiM2QzVDRTc7IC0tcHVycGxlLXNvZnQ6I0VGRUJGRjsKICAgIC0tdGVhbDojMEU5QzhGOyAtLXRlYWwtc29mdDojRTBGNUYyOwogICAgLS1vcmFuZ2U6I0U4ODIxRjsgLS1vcmFuZ2Utc29mdDojRkRFRURDOwogICAgLS1saW5lOiNEREU1RjI7CiAgICAtLXNoYWRvdzowIDEwcHggMjhweCByZ2JhKDEwOCw5MiwyMzEsLjE0KTsKICAgIC0tc2hhZG93LXNtOjAgNHB4IDE0cHggcmdiYSgxMDgsOTIsMjMxLC4xMCk7CiAgICAtLXJhZGl1czoxNnB4OwogICAgLS1tb25vOiJTRiBNb25vIix1aS1tb25vc3BhY2UsTWVubG8sQ29uc29sYXMsbW9ub3NwYWNlOwogIH0KICAqe2JveC1zaXppbmc6Ym9yZGVyLWJveDttYXJnaW46MDtwYWRkaW5nOjB9CiAgYm9keXtiYWNrZ3JvdW5kOnZhcigtLWJnKTtjb2xvcjp2YXIoLS1pbmspOwogICAgZm9udDoxNXB4LzEuNTUgLWFwcGxlLXN5c3RlbSwiU2Vnb2UgVUkiLEludGVyLFJvYm90byxzYW5zLXNlcmlmOwogICAgcGFkZGluZzoyMnB4IDE2cHggMThweDtvdmVyZmxvdy14OmhpZGRlbjt9CiAgLndyYXB7bWF4LXdpZHRoOjExMjBweDttYXJnaW46MCBhdXRvfQogIGhlYWRlciBoMXtmb250LXNpemU6MjNweDtmb250LXdlaWdodDo4MDA7bGV0dGVyLXNwYWNpbmc6LS4wMmVtfQogIGhlYWRlciBoMSAuaGx7Y29sb3I6dmFyKC0tb3JhbmdlKX0KICBoZWFkZXIgcC5zdWJ7Y29sb3I6dmFyKC0tbXV0ZWQpO21hcmdpbi10b3A6NnB4O21heC13aWR0aDo4MjBweH0KICBoZWFkZXIgcC5zdWIgY29kZXtmb250LWZhbWlseTp2YXIoLS1tb25vKTtmb250LXNpemU6LjkyZW07YmFja2dyb3VuZDojZmZmO2JvcmRlci1yYWRpdXM6NnB4O3BhZGRpbmc6MXB4IDZweDtib3gtc2hhZG93OnZhcigtLXNoYWRvdy1zbSl9CgogIC5jb250cm9sc3tkaXNwbGF5OmZsZXg7YWxpZ24taXRlbXM6Y2VudGVyO2dhcDoxNHB4O2ZsZXgtd3JhcDp3cmFwOwogICAgYmFja2dyb3VuZDp2YXIoLS1jYXJkKTtib3JkZXItcmFkaXVzOnZhcigtLXJhZGl1cyk7Ym94LXNoYWRvdzp2YXIoLS1zaGFkb3cpOwogICAgcGFkZGluZzoxM3B4IDE2cHg7bWFyZ2luOjE4cHggMCAxNnB4O30KICAuc2Vne2Rpc3BsYXk6aW5saW5lLWZsZXg7YmFja2dyb3VuZDojRUZGM0ZCO2JvcmRlci1yYWRpdXM6MTFweDtwYWRkaW5nOjNweDtnYXA6M3B4fQogIC5zZWcgYnV0dG9ue2JvcmRlcjpub25lO2JhY2tncm91bmQ6dHJhbnNwYXJlbnQ7Zm9udDppbmhlcml0O2ZvbnQtd2VpZ2h0OjcwMDtib3JkZXItcmFkaXVzOjlweDtwYWRkaW5nOjdweCAxNHB4O2N1cnNvcjpwb2ludGVyO2NvbG9yOnZhcigtLW11dGVkKX0KICAuc2VnIGJ1dHRvbi5vbntiYWNrZ3JvdW5kOnZhcigtLW9yYW5nZSk7Y29sb3I6I2ZmZjtib3gtc2hhZG93OnZhcigtLXNoYWRvdy1zbSl9CiAgYnV0dG9uLmFjdHtmb250OmluaGVyaXQ7Zm9udC13ZWlnaHQ6NzAwO2JvcmRlcjpub25lO2JvcmRlci1yYWRpdXM6MTJweDtjdXJzb3I6cG9pbnRlcjtwYWRkaW5nOjlweCAxOHB4O3RyYW5zaXRpb246dHJhbnNmb3JtIC4xMnN9CiAgI3BsYXl7YmFja2dyb3VuZDp2YXIoLS1wdXJwbGUpO2NvbG9yOiNmZmY7Ym94LXNoYWRvdzowIDZweCAxNnB4IHJnYmEoMTA4LDkyLDIzMSwuMzUpfQogICNwbGF5OmhvdmVye3RyYW5zZm9ybTp0cmFuc2xhdGVZKC0xcHgpfQogICNzdGVwe2JhY2tncm91bmQ6dmFyKC0tcHVycGxlLXNvZnQpO2NvbG9yOnZhcigtLXB1cnBsZSl9CiAgI3Jlc2V0e2JhY2tncm91bmQ6dHJhbnNwYXJlbnQ7Y29sb3I6dmFyKC0tbXV0ZWQpO3RleHQtZGVjb3JhdGlvbjp1bmRlcmxpbmU7cGFkZGluZzo5cHggOHB4fQogIC5zcGR7ZGlzcGxheTpmbGV4O2FsaWduLWl0ZW1zOmNlbnRlcjtnYXA6OHB4O21hcmdpbi1sZWZ0OmF1dG87Zm9udC1zaXplOjEzcHg7Y29sb3I6dmFyKC0tbXV0ZWQpO2ZvbnQtd2VpZ2h0OjYwMH0KICAuc3BkIGlucHV0e2FjY2VudC1jb2xvcjp2YXIoLS1wdXJwbGUpO3dpZHRoOjExMHB4fQogIGxhYmVsLmxibHtmb250LXNpemU6MTNweDtmb250LXdlaWdodDo3MDA7Y29sb3I6dmFyKC0tbXV0ZWQpfQoKICAuZ3JpZHtkaXNwbGF5OmdyaWQ7Z3JpZC10ZW1wbGF0ZS1jb2x1bW5zOjFmciAxZnI7Z2FwOjE4cHg7YWxpZ24taXRlbXM6c3RhcnR9CiAgLmdyaWQuaXMtbmFycm93e2dyaWQtdGVtcGxhdGUtY29sdW1uczoxZnJ9CiAgLmNhcmR7YmFja2dyb3VuZDp2YXIoLS1jYXJkKTtib3JkZXItcmFkaXVzOnZhcigtLXJhZGl1cyk7Ym94LXNoYWRvdzp2YXIoLS1zaGFkb3cpO3BhZGRpbmc6MTVweCAxNnB4fQogIC5jYXJkIGgye2ZvbnQtc2l6ZToxMnB4O2ZvbnQtd2VpZ2h0OjgwMDtsZXR0ZXItc3BhY2luZzouMDhlbTt0ZXh0LXRyYW5zZm9ybTp1cHBlcmNhc2U7Y29sb3I6dmFyKC0tbXV0ZWQpO21hcmdpbi1ib3R0b206MTBweDtkaXNwbGF5OmZsZXg7anVzdGlmeS1jb250ZW50OnNwYWNlLWJldHdlZW59CiAgLmNhcmQgaDIgLmJhZGdle2ZvbnQtZmFtaWx5OnZhcigtLW1vbm8pO2ZvbnQtc2l6ZToxMXB4O2JhY2tncm91bmQ6dmFyKC0tb3JhbmdlLXNvZnQpO2NvbG9yOnZhcigtLW9yYW5nZSk7Ym9yZGVyLXJhZGl1czo5OTlweDtwYWRkaW5nOjJweCA5cHg7dGV4dC10cmFuc2Zvcm06bm9uZTtsZXR0ZXItc3BhY2luZzowfQogIGNhbnZhc3t3aWR0aDoxMDAlO2hlaWdodDphdXRvO2Rpc3BsYXk6YmxvY2s7YmFja2dyb3VuZDojZmJmZGZmO2JvcmRlcjoxcHggc29saWQgdmFyKC0tbGluZSk7Ym9yZGVyLXJhZGl1czoxMHB4fQogIC5zdGFnZXdyYXB7ZGlzcGxheTpncmlkO2dyaWQtdGVtcGxhdGUtY29sdW1uczoxZnIgYXV0byAxZnI7Z2FwOjEwcHg7YWxpZ24taXRlbXM6Y2VudGVyfQogIC5hcnJvd21pZHtmb250LXNpemU6MjRweDtjb2xvcjp2YXIoLS1vcmFuZ2UpO3RleHQtYWxpZ246Y2VudGVyfQogIC5hcnJvd21pZCBzbWFsbHtkaXNwbGF5OmJsb2NrO2ZvbnQtc2l6ZToxMHB4O2NvbG9yOnZhcigtLW11dGVkKTtmb250LXdlaWdodDo3MDA7bWFyZ2luLXRvcDozcHh9CgogIC5mb3JtdWxhe2JhY2tncm91bmQ6I0Y2RjhGRTtib3JkZXItbGVmdDo0cHggc29saWQgdmFyKC0tb3JhbmdlKTtib3JkZXItcmFkaXVzOjEwcHg7cGFkZGluZzoxMXB4IDE0cHg7bWFyZ2luLXRvcDoxMnB4O2ZvbnQtZmFtaWx5OnZhcigtLW1vbm8pO2ZvbnQtc2l6ZToxM3B4O2xpbmUtaGVpZ2h0OjEuNztvdmVyZmxvdy14OmF1dG99CiAgLmZvcm11bGEgYntjb2xvcjp2YXIoLS1vcmFuZ2UpfQogIC5mb3JtdWxhIC5yZXN7Y29sb3I6dmFyKC0tdGVhbCk7Zm9udC13ZWlnaHQ6ODAwfQogIC5ub3Rle2NvbG9yOnZhcigtLW11dGVkKTtmb250LXNpemU6MTIuNXB4O21hcmdpbi10b3A6MTBweDtsaW5lLWhlaWdodDoxLjV9CiAgLm5vdGUgYi5ve2NvbG9yOnZhcigtLW9yYW5nZSl9IC5ub3RlIGIudHtjb2xvcjp2YXIoLS10ZWFsKX0gLm5vdGUgYi5we2NvbG9yOnZhcigtLXB1cnBsZSl9CgogIC8qIGNoYW5uZWwgZXhwYW5zaW9uICovCiAgLmNocm93e2Rpc3BsYXk6ZmxleDthbGlnbi1pdGVtczpjZW50ZXI7Z2FwOjEwcHg7anVzdGlmeS1jb250ZW50OmNlbnRlcjtmbGV4LXdyYXA6d3JhcDttYXJnaW4tdG9wOjZweH0KICAuZmlsdGVyc3RhY2t7ZGlzcGxheTpmbGV4O2ZsZXgtZGlyZWN0aW9uOmNvbHVtbjtnYXA6NnB4fQogIC5maWx0e2Rpc3BsYXk6ZmxleDthbGlnbi1pdGVtczpjZW50ZXI7Z2FwOjhweH0KICAuZmt7ZGlzcGxheTpncmlkO2dyaWQtdGVtcGxhdGUtY29sdW1uczpyZXBlYXQoMywxMXB4KTtncmlkLWF1dG8tcm93czoxMXB4O2dhcDoycHh9CiAgLmZrIGl7d2lkdGg6MTFweDtoZWlnaHQ6MTFweDtib3JkZXItcmFkaXVzOjJweH0KICAuZm1hcHt3aWR0aDo0NnB4O2hlaWdodDo0NnB4O2JvcmRlci1yYWRpdXM6NnB4O2JvcmRlcjoxcHggc29saWQgdmFyKC0tbGluZSl9CiAgLmZsYWJlbHtmb250LWZhbWlseTp2YXIoLS1tb25vKTtmb250LXNpemU6MTFweDtjb2xvcjp2YXIoLS1tdXRlZCk7bWluLXdpZHRoOjY0cHh9CiAgZm9vdGVye21hcmdpbi10b3A6MTZweDtjb2xvcjp2YXIoLS1tdXRlZCk7Zm9udC1zaXplOjEyLjVweDt0ZXh0LWFsaWduOmNlbnRlcn0KICBAbWVkaWEgKHByZWZlcnMtcmVkdWNlZC1tb3Rpb246IHJlZHVjZSl7Knt0cmFuc2l0aW9uOm5vbmUhaW1wb3J0YW50fX0KPC9zdHlsZT4KPC9oZWFkPgo8Ym9keT4KPGRpdiBjbGFzcz0id3JhcCI+CjxoZWFkZXI+CiAgPGgxPkNvbnZvbHV0aW9uOiA8c3BhbiBjbGFzcz0iaGwiPnN0cmlkZSBzaHJpbmtzLCBmaWx0ZXJzIG11bHRpcGx5PC9zcGFuPjwvaDE+CiAgPHAgY2xhc3M9InN1YiI+QSBzaW5nbGUgPGNvZGU+Y29udigxLCA0KTwvY29kZT4gZG9lcyB0d28gdW5yZWxhdGVkIHRoaW5ncyBhdCBvbmNlLiA8YiBzdHlsZT0iY29sb3I6dmFyKC0tb3JhbmdlKSI+U3RyaWRlPC9iPgogIGNvbnRyb2xzIGhvdyBmYXIgdGhlIDPDlzMga2VybmVsIGp1bXBzLCB3aGljaCBzZXRzIHRoZSBvdXRwdXQgc2l6ZS4gVGhlIDxiIHN0eWxlPSJjb2xvcjp2YXIoLS1wdXJwbGUpIj5udW1iZXIgb2YgZmlsdGVyczwvYj4KICBjb250cm9scyBob3cgbWFueSBmZWF0dXJlLW1hcCBjaGFubmVscyBjb21lIG91dC4gRHJhZyB0aGUga2VybmVsLCB0b2dnbGUgdGhlIHN0cmlkZSwgYW5kIHdhdGNoIHRoZSBvdXRwdXQgZ3JpZCBmaWxsIGluLjwvcD4KPC9oZWFkZXI+Cgo8ZGl2IGNsYXNzPSJjb250cm9scyI+CiAgPHNwYW4gY2xhc3M9ImxibCI+U3RyaWRlPC9zcGFuPgogIDxkaXYgY2xhc3M9InNlZyIgaWQ9InN0cmlkZVNlZyI+CiAgICA8YnV0dG9uIGRhdGEtdj0iMSI+c3RyaWRlID0gMTwvYnV0dG9uPgogICAgPGJ1dHRvbiBkYXRhLXY9IjIiIGNsYXNzPSJvbiI+c3RyaWRlID0gMjwvYnV0dG9uPgogIDwvZGl2PgogIDxidXR0b24gaWQ9InBsYXkiIGNsYXNzPSJhY3QiPlBsYXkg4pa2PC9idXR0b24+CiAgPGJ1dHRvbiBpZD0ic3RlcCIgY2xhc3M9ImFjdCI+U3RlcCDihpI8L2J1dHRvbj4KICA8YnV0dG9uIGlkPSJyZXNldCIgY2xhc3M9ImFjdCI+UmVzZXQ8L2J1dHRvbj4KICA8bGFiZWwgY2xhc3M9InNwZCI+c3BlZWQgPGlucHV0IHR5cGU9InJhbmdlIiBpZD0ic3BkIiBtaW49IjEiIG1heD0iNiIgc3RlcD0iMSIgdmFsdWU9IjMiPjxzcGFuIGlkPSJzcGR2Ij4xLjDDlzwvc3Bhbj48L2xhYmVsPgo8L2Rpdj4KCjxkaXYgY2xhc3M9ImdyaWQiIGlkPSJncmlkIj4KICA8ZGl2IGNsYXNzPSJjYXJkIj4KICAgIDxoMj5LZXJuZWwgc2xpZGluZyBvdmVyIHRoZSBpbnB1dCA8c3BhbiBjbGFzcz0iYmFkZ2UiIGlkPSJwb3NCYWRnZSI+cG9zIDAsMDwvc3Bhbj48L2gyPgogICAgPGRpdiBjbGFzcz0ic3RhZ2V3cmFwIj4KICAgICAgPGRpdj4KICAgICAgICA8Y2FudmFzIGlkPSJjaW4iIHdpZHRoPSIyODAiIGhlaWdodD0iMjgwIj48L2NhbnZhcz4KICAgICAgICA8ZGl2IHN0eWxlPSJ0ZXh0LWFsaWduOmNlbnRlcjtmb250LXNpemU6MTFweDtmb250LXdlaWdodDo3MDA7Y29sb3I6dmFyKC0tb3JhbmdlKTttYXJnaW4tdG9wOjVweCI+SW5wdXQgMjjDlzI4PC9kaXY+CiAgICAgIDwvZGl2PgogICAgICA8ZGl2IGNsYXNzPSJhcnJvd21pZCI+4pa2PHNtYWxsIGlkPSJtaWRsYWIiPnN0cmlkZSAyPC9zbWFsbD48L2Rpdj4KICAgICAgPGRpdj4KICAgICAgICA8Y2FudmFzIGlkPSJjb3V0IiB3aWR0aD0iMjgwIiBoZWlnaHQ9IjI4MCI+PC9jYW52YXM+CiAgICAgICAgPGRpdiBzdHlsZT0idGV4dC1hbGlnbjpjZW50ZXI7Zm9udC1zaXplOjExcHg7Zm9udC13ZWlnaHQ6NzAwO2NvbG9yOnZhcigtLXRlYWwpO21hcmdpbi10b3A6NXB4IiBpZD0ib3V0bGFiIj5PdXRwdXQgMTTDlzE0PC9kaXY+CiAgICAgIDwvZGl2PgogICAgPC9kaXY+CiAgICA8ZGl2IGNsYXNzPSJub3RlIj5UaGUgb3JhbmdlIHNxdWFyZSBpcyB0aGUgM8OXMyBrZXJuZWwuIEVhY2ggcG9zaXRpb24gaXQgbGFuZHMgb24gd3JpdGVzIDxiIGNsYXNzPSJ0Ij5vbmU8L2I+IG91dHB1dCBwaXhlbC4KICAgIFdpdGggPGIgY2xhc3M9Im8iPnN0cmlkZSA9IDI8L2I+IGl0IHNraXBzIGV2ZXJ5IG90aGVyIHNwb3QsIHNvIGEgMjjDlzI4IGlucHV0IHlpZWxkcyBhIDE0w5cxNCBvdXRwdXQg4oCUIGhhbGYgdGhlIHNpemUuPC9kaXY+CiAgPC9kaXY+CgogIDxkaXYgY2xhc3M9ImNhcmQiPgogICAgPGgyPlRoZSBvdXRwdXQtc2l6ZSBmb3JtdWxhPC9oMj4KICAgIDxkaXYgY2xhc3M9ImZvcm11bGEiIGlkPSJmb3JtdWxhIj48L2Rpdj4KICAgIDxkaXYgY2xhc3M9Im5vdGUiIGlkPSJmbm90ZSI+PC9kaXY+CiAgICA8aDIgc3R5bGU9Im1hcmdpbi10b3A6MTZweCI+Q2hhbm5lbCBleHBhbnNpb246IDEg4oaSIDQgZmlsdGVyczwvaDI+CiAgICA8ZGl2IGNsYXNzPSJjaHJvdyI+CiAgICAgIDxkaXYgY2xhc3M9ImZpbHRlcnN0YWNrIiBpZD0iZmlsdGVyc3RhY2siPjwvZGl2PgogICAgPC9kaXY+CiAgICA8ZGl2IGNsYXNzPSJub3RlIj5FYWNoIGZpbHRlciBpcyBpdHMgb3duIGxlYXJuYWJsZSAzw5czIHdlaWdodCBncmlkLiBSdW4gYWxsIDQgb3ZlciB0aGUgc2FtZSBpbnB1dCBhbmQgeW91IGdldAogICAgPGIgY2xhc3M9InAiPjQgb3V0cHV0IGNoYW5uZWxzPC9iPiDigJQgZWFjaCBhIGRpZmZlcmVudCBmZWF0dXJlIG1hcCAoZWRnZXMsIGN1cnZlcywgdGV4dHVyZXPigKYpLiBTdHJpZGUgc2V0cyB0aGUKICAgIDxpPnNpemU8L2k+IG9mIGVhY2ggbWFwOyB0aGUgZmlsdGVyIGNvdW50IHNldHMgPGk+aG93IG1hbnk8L2k+IG1hcHMuPC9kaXY+CiAgPC9kaXY+CjwvZGl2PgoKPGZvb3Rlcj5Ucnkgc3dpdGNoaW5nIHRvIDxiPnN0cmlkZSA9IDE8L2I+OiB0aGUga2VybmVsIHZpc2l0cyBldmVyeSBwb3NpdGlvbiBhbmQgdGhlIG91dHB1dCBzdGF5cyAyOMOXMjguIFN0cmlkZSBpcyB0aGUgb25seSB0aGluZyB0aGF0IGNoYW5nZWQuPC9mb290ZXI+CjwvZGl2PgoKPHNjcmlwdD4KKGZ1bmN0aW9uKCl7CiJ1c2Ugc3RyaWN0IjsKY29uc3QgJD1pZD0+ZG9jdW1lbnQuZ2V0RWxlbWVudEJ5SWQoaWQpOwpjb25zdCBJTj0yOCwgS1M9MywgUEFEPTE7CmxldCBzdHJpZGU9Miwga3g9MCwga3k9MCwgcGxheWluZz1mYWxzZSwgdGltZXI9bnVsbDsKY29uc3Qgc3BlZWRzPVswLjUsMC43NSwxLjAsMS41LDIuMCwzLjBdOwoKY29uc3QgY2luPSQoJ2NpbicpLCBjb3V0PSQoJ2NvdXQnKTsKY29uc3QgZ2luPWNpbi5nZXRDb250ZXh0KCcyZCcpLCBnb3V0PWNvdXQuZ2V0Q29udGV4dCgnMmQnKTsKY29uc3QgQ0VMTD1jaW4ud2lkdGgvSU47CgovLyBzeW50aGV0aWMgaW5wdXQgaW50ZW5zaXR5IChhIHNvZnQgYmxvYiBzbyB0aGUga2VybmVsIGhhcyBzb21ldGhpbmcgdG8gInNlZSIpCmZ1bmN0aW9uIGludGVuKHgseSl7CiAgY29uc3QgZHg9KHgtMTMpLzksIGR5PSh5LTEzKS85OwogIGxldCB2PU1hdGguZXhwKC0oZHgqZHgrZHkqZHkpKSowLjggKyAwLjA4OwogIGlmKCgoeCt5KSU2KTwyKSB2Kz0wLjA2OwogIHJldHVybiBNYXRoLm1pbigxLHYpOwp9CmZ1bmN0aW9uIG91dFNpemUoKXsgcmV0dXJuIE1hdGguZmxvb3IoKElOKzIqUEFELUtTKS9zdHJpZGUpKzE7IH0KCmZ1bmN0aW9uIGRyYXdJbnB1dCgpewogIGdpbi5jbGVhclJlY3QoMCwwLGNpbi53aWR0aCxjaW4uaGVpZ2h0KTsKICBmb3IobGV0IHk9MDt5PElOO3krKylmb3IobGV0IHg9MDt4PElOO3grKyl7CiAgICBjb25zdCBnPWludGVuKHgseSk7IGNvbnN0IGM9TWF0aC5yb3VuZCgyNTUtKGcqMTUwKSk7CiAgICBnaW4uZmlsbFN0eWxlPWByZ2IoJHtjfSwke2MrMTB9LCR7TWF0aC5taW4oMjU1LGMrMjQpfSlgOwogICAgZ2luLmZpbGxSZWN0KHgqQ0VMTCx5KkNFTEwsQ0VMTCxDRUxMKTsKICB9CiAgLy8gZ3JpZCBsaW5lcyAobGlnaHQpCiAgZ2luLnN0cm9rZVN0eWxlPSdyZ2JhKDEyMCwxNDAsMTgwLC4xMCknOyBnaW4ubGluZVdpZHRoPTE7CiAgZm9yKGxldCBpPTA7aTw9SU47aSsrKXtnaW4uYmVnaW5QYXRoKCk7Z2luLm1vdmVUbyhpKkNFTEwsMCk7Z2luLmxpbmVUbyhpKkNFTEwsY2luLmhlaWdodCk7Z2luLnN0cm9rZSgpOwogICAgZ2luLmJlZ2luUGF0aCgpO2dpbi5tb3ZlVG8oMCxpKkNFTEwpO2dpbi5saW5lVG8oY2luLndpZHRoLGkqQ0VMTCk7Z2luLnN0cm9rZSgpO30KICAvLyBrZXJuZWwgcmVjdGFuZ2xlOiB0b3AtbGVmdCBpbnB1dCBwaXhlbCBmb3IgdGhpcyBvdXRwdXQgcG9zaXRpb24gPSBreCpzdHJpZGUgLSBQQUQKICBjb25zdCBpeD1reCpzdHJpZGUtUEFELCBpeT1reSpzdHJpZGUtUEFEOwogIGdpbi5maWxsU3R5bGU9J3JnYmEoMjMyLDEzMCwzMSwuMjIpJzsKICBnaW4uZmlsbFJlY3QoaXgqQ0VMTCxpeSpDRUxMLEtTKkNFTEwsS1MqQ0VMTCk7CiAgZ2luLnN0cm9rZVN0eWxlPScjRTg4MjFGJzsgZ2luLmxpbmVXaWR0aD0zOwogIGdpbi5zdHJva2VSZWN0KGl4KkNFTEwsaXkqQ0VMTCxLUypDRUxMLEtTKkNFTEwpOwp9CgpsZXQgZmlsbGVkPVtdOwpmdW5jdGlvbiByZXNldE91dCgpeyBjb25zdCBPPW91dFNpemUoKTsgZmlsbGVkPUFycmF5KE8qTykuZmlsbChmYWxzZSk7IH0KZnVuY3Rpb24gZHJhd091dHB1dCgpewogIGNvbnN0IE89b3V0U2l6ZSgpOyBjb25zdCBvYz1jb3V0LndpZHRoL087CiAgZ291dC5jbGVhclJlY3QoMCwwLGNvdXQud2lkdGgsY291dC5oZWlnaHQpOwogIGZvcihsZXQgeT0wO3k8Tzt5KyspZm9yKGxldCB4PTA7eDxPO3grKyl7CiAgICBpZihmaWxsZWRbeSpPK3hdKXsKICAgICAgLy8gdmFsdWUgPSBhdmcgaW50ZW5zaXR5IHVuZGVyIHRoYXQga2VybmVsIHBvcwogICAgICBjb25zdCBpeD14KnN0cmlkZS1QQUQsIGl5PXkqc3RyaWRlLVBBRDsgbGV0IHM9MCxjbnQ9MDsKICAgICAgZm9yKGxldCBhPTA7YTxLUzthKyspZm9yKGxldCBiPTA7YjxLUztiKyspe2NvbnN0IG54PWl4K2Isbnk9aXkrYTtpZihueDwwfHxueTwwfHxueD49SU58fG55Pj1JTiljb250aW51ZTtzKz1pbnRlbihueCxueSk7Y250Kys7fQogICAgICBjb25zdCBnPWNudD9zL2NudDowOyBjb25zdCBjPU1hdGgucm91bmQoMjU1LShnKjE1MCkpOwogICAgICBnb3V0LmZpbGxTdHlsZT1gcmdiKCR7Y30sJHtNYXRoLm1pbigyNTUsYysxOCl9LCR7TWF0aC5taW4oMjU1LGMrMTQpfSlgOwogICAgfSBlbHNlIGdvdXQuZmlsbFN0eWxlPScjZjFmNWZjJzsKICAgIGdvdXQuZmlsbFJlY3QoeCpvYyx5Km9jLG9jLG9jKTsKICB9CiAgZ291dC5zdHJva2VTdHlsZT0ncmdiYSgxMjAsMTQwLDE4MCwuMTYpJzsgZ291dC5saW5lV2lkdGg9MTsKICBmb3IobGV0IGk9MDtpPD1PO2krKyl7Z291dC5iZWdpblBhdGgoKTtnb3V0Lm1vdmVUbyhpKm9jLDApO2dvdXQubGluZVRvKGkqb2MsY291dC5oZWlnaHQpO2dvdXQuc3Ryb2tlKCk7CiAgICBnb3V0LmJlZ2luUGF0aCgpO2dvdXQubW92ZVRvKDAsaSpvYyk7Z291dC5saW5lVG8oY291dC53aWR0aCxpKm9jKTtnb3V0LnN0cm9rZSgpO30KICAvLyBoaWdobGlnaHQgY3VycmVudCB0YXJnZXQgY2VsbAogIGdvdXQuc3Ryb2tlU3R5bGU9JyMwRTlDOEYnOyBnb3V0LmxpbmVXaWR0aD0zOwogIGdvdXQuc3Ryb2tlUmVjdChreCpvYyxreSpvYyxvYyxvYyk7Cn0KCmZ1bmN0aW9uIHVwZGF0ZUZvcm11bGEoKXsKICBjb25zdCBPPW91dFNpemUoKTsKICAkKCdmb3JtdWxhJykuaW5uZXJIVE1MPQogICAgYG91dHB1dCA9IOKMiihpbnB1dCArIDLCt3BhZCDiiJIga2VybmVsKSAvIDxiPnN0cmlkZTwvYj7ijIsgKyAxPGJyPmArCiAgICBgJm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7PSDijIooMjggKyAywrcxIOKIkiAzKSAvIDxiPiR7c3RyaWRlfTwvYj7ijIsgKyAxPGJyPmArCiAgICBgJm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7PSDijIoyNyAvIDxiPiR7c3RyaWRlfTwvYj7ijIsgKyAxID0gPHNwYW4gY2xhc3M9InJlcyI+JHtPfTwvc3Bhbj5gOwogICQoJ2Zub3RlJykuaW5uZXJIVE1MID0gc3RyaWRlPT09MgogICAgPyBgV2l0aCA8YiBjbGFzcz0ibyI+c3RyaWRlID0gMjwvYj4gdGhlIGtlcm5lbCBza2lwcyBldmVyeSBvdGhlciBwb3NpdGlvbiwgc28gd2lkdGggYW5kIGhlaWdodCBhcmUgcm91Z2hseSA8Yj5oYWx2ZWQ8L2I+OiAyOCDihpIgMTQuYAogICAgOiBgV2l0aCA8YiBjbGFzcz0ibyI+c3RyaWRlID0gMTwvYj4gdGhlIGtlcm5lbCB2aXNpdHMgPGk+ZXZlcnk8L2k+IHBvc2l0aW9uLCBzbyA8Y29kZT5wYWRkaW5nID0ga3MvLzI8L2NvZGU+IGtlZXBzIHRoZSBzaXplIHRoZSA8Yj5zYW1lPC9iPjogMjgg4oaSIDI4LmA7CiAgJCgnb3V0bGFiJykudGV4dENvbnRlbnQ9YE91dHB1dCAke099w5cke099YDsKICAkKCdtaWRsYWInKS50ZXh0Q29udGVudD1gc3RyaWRlICR7c3RyaWRlfWA7Cn0KCmZ1bmN0aW9uIHJlbmRlcigpewogICQoJ3Bvc0JhZGdlJykudGV4dENvbnRlbnQ9YHBvcyAke2t5fSwke2t4fWA7CiAgZHJhd0lucHV0KCk7IGRyYXdPdXRwdXQoKTsgdXBkYXRlRm9ybXVsYSgpOyBwb3N0SCgpOwp9CmZ1bmN0aW9uIGFkdmFuY2UoKXsKICBjb25zdCBPPW91dFNpemUoKTsKICBmaWxsZWRba3kqTytreF09dHJ1ZTsKICBreCsrOyBpZihreD49Tyl7a3g9MDtreSsrO30KICBpZihreT49Tyl7IC8vIGZpbmlzaGVkIGEgZnVsbCBwYXNzCiAgICBreT0wO2t4PTA7IC8vIGxvb3AKICAgIGlmKHBsYXlpbmcpeyAvLyBtYXJrIGFsbCB0aGVuIHJlc3RhcnQgdmlzaWJseQogICAgICByZXNldE91dCgpOwogICAgfQogIH0KICByZW5kZXIoKTsKfQpmdW5jdGlvbiBmdWxsUmVzZXQoKXsga3g9MDtreT0wOyByZXNldE91dCgpOyByZW5kZXIoKTsgfQoKZnVuY3Rpb24gc3RvcFBsYXkoKXsgcGxheWluZz1mYWxzZTsgaWYodGltZXIpY2xlYXJJbnRlcnZhbCh0aW1lcik7ICQoJ3BsYXknKS50ZXh0Q29udGVudD0nUGxheSDilrYnOyB9CiQoJ3BsYXknKS5hZGRFdmVudExpc3RlbmVyKCdjbGljaycsKCk9PnsKICBpZihwbGF5aW5nKXtzdG9wUGxheSgpO3JldHVybjt9CiAgcGxheWluZz10cnVlOyAkKCdwbGF5JykudGV4dENvbnRlbnQ9J1BhdXNlIOKPuCc7CiAgY29uc3QgbXM9MTQwL3NwZWVkc1srJCgnc3BkJykudmFsdWUtMV07CiAgdGltZXI9c2V0SW50ZXJ2YWwoKCk9PnsgaWYocGxheWluZylhZHZhbmNlKCk7IH0sIE1hdGgubWF4KDE4LG1zKSk7Cn0pOwokKCdzdGVwJykuYWRkRXZlbnRMaXN0ZW5lcignY2xpY2snLCgpPT57IHN0b3BQbGF5KCk7IGFkdmFuY2UoKTsgfSk7CiQoJ3Jlc2V0JykuYWRkRXZlbnRMaXN0ZW5lcignY2xpY2snLCgpPT57IHN0b3BQbGF5KCk7IGZ1bGxSZXNldCgpOyB9KTsKJCgnc3BkJykuYWRkRXZlbnRMaXN0ZW5lcignaW5wdXQnLCgpPT57CiAgJCgnc3BkdicpLnRleHRDb250ZW50PXNwZWVkc1srJCgnc3BkJykudmFsdWUtMV0udG9GaXhlZCgxKSsnw5cnOwogIGlmKHBsYXlpbmcpeyBjbGVhckludGVydmFsKHRpbWVyKTsgY29uc3QgbXM9MTQwL3NwZWVkc1srJCgnc3BkJykudmFsdWUtMV07IHRpbWVyPXNldEludGVydmFsKCgpPT57aWYocGxheWluZylhZHZhbmNlKCk7fSxNYXRoLm1heCgxOCxtcykpOyB9Cn0pOwokKCdzdHJpZGVTZWcnKS5hZGRFdmVudExpc3RlbmVyKCdjbGljaycsZT0+ewogIGNvbnN0IGI9ZS50YXJnZXQuY2xvc2VzdCgnYnV0dG9uJyk7IGlmKCFiKXJldHVybjsKICBzdHJpZGU9K2IuZGF0YXNldC52OwogIGRvY3VtZW50LnF1ZXJ5U2VsZWN0b3JBbGwoJyNzdHJpZGVTZWcgYnV0dG9uJykuZm9yRWFjaCh4PT54LmNsYXNzTGlzdC50b2dnbGUoJ29uJyx4PT09YikpOwogIHN0b3BQbGF5KCk7IGZ1bGxSZXNldCgpOwp9KTsKLy8gZHJhZyBrZXJuZWwgb24gaW5wdXQKY2luLmFkZEV2ZW50TGlzdGVuZXIoJ3BvaW50ZXJkb3duJyxlPT57IHN0b3BQbGF5KCk7IG1vdmVLKGUpOyBjaW4uc2V0UG9pbnRlckNhcHR1cmUoZS5wb2ludGVySWQpOyBjaW4ub25wb2ludGVybW92ZT1tb3ZlSzsgfSk7CmNpbi5hZGRFdmVudExpc3RlbmVyKCdwb2ludGVydXAnLCgpPT57IGNpbi5vbnBvaW50ZXJtb3ZlPW51bGw7IH0pOwpmdW5jdGlvbiBtb3ZlSyhlKXsKICBjb25zdCByPWNpbi5nZXRCb3VuZGluZ0NsaWVudFJlY3QoKTsKICBjb25zdCBweD0oZS5jbGllbnRYLXIubGVmdCkvci53aWR0aCpJTiwgcHk9KGUuY2xpZW50WS1yLnRvcCkvci5oZWlnaHQqSU47CiAgY29uc3QgTz1vdXRTaXplKCk7CiAga3g9TWF0aC5tYXgoMCxNYXRoLm1pbihPLTEsTWF0aC5yb3VuZCgocHgrUEFEKS9zdHJpZGUpKSk7CiAga3k9TWF0aC5tYXgoMCxNYXRoLm1pbihPLTEsTWF0aC5yb3VuZCgocHkrUEFEKS9zdHJpZGUpKSk7CiAgcmVuZGVyKCk7Cn0KCi8vIGJ1aWxkIDQgZmlsdGVycyB3LyBmZWF0dXJlIG1hcHMKZnVuY3Rpb24gYnVpbGRGaWx0ZXJzKCl7CiAgY29uc3QgZnM9JCgnZmlsdGVyc3RhY2snKTsgZnMuaW5uZXJIVE1MPScnOwogIGNvbnN0IHBhbGV0dGVzPVtbJyNFODgyMUYnLCdob3Jpem9udGFsIGVkZ2VzJ10sWycjNkM1Q0U3JywndmVydGljYWwgZWRnZXMnXSxbJyMwRTlDOEYnLCdjdXJ2ZXMgLyBjb3JuZXJzJ10sWycjYzA0NjhmJywndGV4dHVyZXMnXV07CiAgcGFsZXR0ZXMuZm9yRWFjaCgocCxmaSk9PnsKICAgIGNvbnN0IHJvdz1kb2N1bWVudC5jcmVhdGVFbGVtZW50KCdkaXYnKTsgcm93LmNsYXNzTmFtZT0nZmlsdCc7CiAgICBjb25zdCBrPWRvY3VtZW50LmNyZWF0ZUVsZW1lbnQoJ2RpdicpOyBrLmNsYXNzTmFtZT0nZmsnOwogICAgZm9yKGxldCBpPTA7aTw5O2krKyl7Y29uc3QgYz1kb2N1bWVudC5jcmVhdGVFbGVtZW50KCdpJyk7CiAgICAgIGNvbnN0IHc9TWF0aC5zaW4oZmkqMS43K2kqMC45KTsgYy5zdHlsZS5iYWNrZ3JvdW5kPXc+MD9wWzBdOicjZGZlNmYzJzsgYy5zdHlsZS5vcGFjaXR5PU1hdGguYWJzKHcpKjAuNyswLjM7IGsuYXBwZW5kQ2hpbGQoYyk7fQogICAgY29uc3QgYXJyPWRvY3VtZW50LmNyZWF0ZUVsZW1lbnQoJ3NwYW4nKTsgYXJyLnRleHRDb250ZW50PSfihpInOyBhcnIuc3R5bGUuY29sb3I9J3ZhcigtLW11dGVkKSc7CiAgICBjb25zdCBtYXA9ZG9jdW1lbnQuY3JlYXRlRWxlbWVudCgnY2FudmFzJyk7IG1hcC5jbGFzc05hbWU9J2ZtYXAnOyBtYXAud2lkdGg9bWFwLmhlaWdodD00NjsKICAgIGNvbnN0IG1nPW1hcC5nZXRDb250ZXh0KCcyZCcpOwogICAgZm9yKGxldCB5PTA7eTwyMzt5KyspZm9yKGxldCB4PTA7eDwyMzt4KyspewogICAgICBjb25zdCB2PU1hdGguYWJzKE1hdGguc2luKGZpKjEuMyt4KjAuNCkqTWF0aC5jb3MoZmkreSowLjQpKSppbnRlbih4KjEuMix5KjEuMik7CiAgICAgIGNvbnN0IHJnYj1oZXgycmdiKHBbMF0pOyBjb25zdCB0PXY7CiAgICAgIG1nLmZpbGxTdHlsZT1gcmdiKCR7TWF0aC5yb3VuZCgyNTUtKDI1NS1yZ2JbMF0pKnQpfSwke01hdGgucm91bmQoMjU1LSgyNTUtcmdiWzFdKSp0KX0sJHtNYXRoLnJvdW5kKDI1NS0oMjU1LXJnYlsyXSkqdCl9KWA7CiAgICAgIG1nLmZpbGxSZWN0KHgqMix5KjIsMiwyKTsKICAgIH0KICAgIGNvbnN0IGxhYj1kb2N1bWVudC5jcmVhdGVFbGVtZW50KCdzcGFuJyk7IGxhYi5jbGFzc05hbWU9J2ZsYWJlbCc7IGxhYi50ZXh0Q29udGVudD0nY2gnK2ZpKyc6ICcrcFsxXTsKICAgIHJvdy5hcHBlbmRDaGlsZChrKTtyb3cuYXBwZW5kQ2hpbGQoYXJyKTtyb3cuYXBwZW5kQ2hpbGQobWFwKTtyb3cuYXBwZW5kQ2hpbGQobGFiKTsKICAgIGZzLmFwcGVuZENoaWxkKHJvdyk7CiAgfSk7Cn0KZnVuY3Rpb24gaGV4MnJnYihoKXtoPWgucmVwbGFjZSgnIycsJycpO3JldHVybiBbcGFyc2VJbnQoaC5zbGljZSgwLDIpLDE2KSxwYXJzZUludChoLnNsaWNlKDIsNCksMTYpLHBhcnNlSW50KGguc2xpY2UoNCw2KSwxNildO30KCmNvbnN0IGdyaWQ9JCgnZ3JpZCcpLCB3cmFwPWRvY3VtZW50LnF1ZXJ5U2VsZWN0b3IoJy53cmFwJyk7CmZ1bmN0aW9uIHJlbGF5b3V0KCl7IGdyaWQuY2xhc3NMaXN0LnRvZ2dsZSgnaXMtbmFycm93Jywgd2luZG93LmlubmVyV2lkdGg8ODIwKTsgfQpmdW5jdGlvbiBwb3N0SCgpeyBjb25zdCBoPXdyYXA/TWF0aC5jZWlsKHdyYXAuZ2V0Qm91bmRpbmdDbGllbnRSZWN0KCkuaGVpZ2h0KSszNDpkb2N1bWVudC5ib2R5Lm9mZnNldEhlaWdodDsKICBpZih3aW5kb3cucGFyZW50IT09d2luZG93KSB3aW5kb3cucGFyZW50LnBvc3RNZXNzYWdlKHt0eXBlOidhZS1mcmFtZS1oZWlnaHQnLGhlaWdodDpofSwnKicpOyB9CmZ1bmN0aW9uIHVwZGF0ZSgpeyByZWxheW91dCgpOyBwb3N0SCgpOyB9CndpbmRvdy5hZGRFdmVudExpc3RlbmVyKCdsb2FkJyx1cGRhdGUpOyB3aW5kb3cuYWRkRXZlbnRMaXN0ZW5lcigncmVzaXplJyx1cGRhdGUpOwppZih3aW5kb3cuUmVzaXplT2JzZXJ2ZXIpIG5ldyBSZXNpemVPYnNlcnZlcihwb3N0SCkub2JzZXJ2ZSh3cmFwfHxkb2N1bWVudC5ib2R5KTsKYnVpbGRGaWx0ZXJzKCk7IHJlc2V0T3V0KCk7IHJlbGF5b3V0KCk7IHJlbmRlcigpOwpzZXRUaW1lb3V0KHVwZGF0ZSwyMDApOyBzZXRUaW1lb3V0KHVwZGF0ZSw3MDApOwp9KSgpOwo8L3NjcmlwdD4KPC9ib2R5Pgo8L2h0bWw+Cg=="

HTML('''
<iframe id="conv-stride-frame"
        src="data:text/html;base64,''' + _html_b64 + '''"
        style="width:100%; height:820px; border:1px solid #DDE5F2;
               border-radius:16px; box-shadow:0 8px 24px rgba(108,92,231,.12);
               display:block;"
        loading="lazy" title="Convolution stride and channel lab"></iframe>
<script>
(function(){
  function onMsg(e){
    if (e.data && e.data.type === "ae-frame-height") {
      var f = document.getElementById("conv-stride-frame");
      if (f) f.style.height = (e.data.height) + "px";
    }
  }
  window.addEventListener("message", onMsg);
})();
</script>
''')


In [ ]:
# =====================================================
# TRAINING FUNCTION FOR CLASSIFICATION
# =====================================================

def accuracy(preds, targets):
    """Calculate accuracy: fraction of correct predictions."""
    return (preds.argmax(dim=1) == targets).float().mean()

def fit(epochs, model, loss_func, opt, train_dl, valid_dl):
    """
    Train a classification model.
    
    Arguments:
        epochs   - Number of complete passes through the data
        model    - The neural network to train
        loss_func - Function to compute loss (e.g., cross_entropy)
        opt      - Optimizer (e.g., SGD)
        train_dl - Training data loader
        valid_dl - Validation data loader
    
    Returns:
        Final (loss, accuracy) on validation set
    """
    for epoch in range(epochs):
        # Training phase
        model.train()
        for xb, yb in train_dl:
            # Forward pass
            preds = model(xb)
            loss = loss_func(preds, yb)
            
            # Backward pass
            loss.backward()
            opt.step()
            opt.zero_grad()
        
        # Validation phase
        model.eval()
        with torch.no_grad():
            tot_loss, tot_acc, count = 0., 0., 0
            for xb, yb in valid_dl:
                preds = model(xb)
                n = len(xb)
                count += n
                tot_loss += loss_func(preds, yb).item() * n
                tot_acc += accuracy(preds, yb).item() * n
        
        print(f"Epoch {epoch}: loss={tot_loss/count:.4f}, accuracy={tot_acc/count:.4f}")
    
    return tot_loss/count, tot_acc/count

# Understanding the Training Function for Classification

## Overview

This code defines two functions:
1. **`accuracy`** — Measures how many predictions are correct
2. **`fit`** — The main training loop that teaches the neural network

---

## The `accuracy` Function

```python
def accuracy(preds, targets):
    """Calculate accuracy: fraction of correct predictions."""
    return (preds.argmax(dim=1) == targets).float().mean()
```

### What It Does

Calculates the fraction of correct predictions (e.g., 0.85 means 85% correct).

### Step-by-Step Breakdown

| Step | Code | Example | Result |
|------|------|---------|--------|
| 1. Get predicted class | `preds.argmax(dim=1)` | `[[0.1, 0.9, 0.0], [0.8, 0.1, 0.1]]` | `[1, 0]` |
| 2. Compare to targets | `== targets` | `[1, 0] == [1, 2]` | `[True, False]` |
| 3. Convert to numbers | `.float()` | `[True, False]` | `[1.0, 0.0]` |
| 4. Calculate average | `.mean()` | `[1.0, 0.0]` | `0.5` (50%) |

### Visual Example

```
Predictions (raw scores for 3 classes):
  Image 0: [0.1, 0.9, 0.0] → argmax → Class 1 ✓ (target was 1)
  Image 1: [0.8, 0.1, 0.1] → argmax → Class 0 ✗ (target was 2)

Accuracy = 1 correct / 2 total = 0.5 (50%)
```

---

## The `fit` Function

```python
def fit(epochs, model, loss_func, opt, train_dl, valid_dl):
```

### Arguments Explained

| Argument | What It Is | Example |
|----------|-----------|---------|
| `epochs` | Number of complete passes through all data | `5` |
| `model` | The neural network to train | `cnn` |
| `loss_func` | Function measuring prediction error | `F.cross_entropy` |
| `opt` | Optimizer that updates weights | `optim.SGD(...)` |
| `train_dl` | DataLoader with training data | Batches of images + labels |
| `valid_dl` | DataLoader with validation data | Batches for evaluation |

---

## The Training Loop Explained

### High-Level Structure

```
For each epoch:
    1. TRAINING PHASE   → Learn from training data
    2. VALIDATION PHASE → Check performance on unseen data
    3. Print results
```

---

### Phase 1: Training

```python
model.train()
for xb, yb in train_dl:
    # Forward pass
    preds = model(xb)
    loss = loss_func(preds, yb)
    
    # Backward pass
    loss.backward()
    opt.step()
    opt.zero_grad()
```

#### What Each Line Does

| Line | Purpose | Analogy |
|------|---------|---------|
| `model.train()` | Enable training mode (activates dropout, etc.) | "Get ready to learn" |
| `for xb, yb in train_dl` | Loop through batches of images (`xb`) and labels (`yb`) | "Go through homework problems" |
| `preds = model(xb)` | **Forward pass**: Make predictions | "Attempt the problem" |
| `loss = loss_func(preds, yb)` | Calculate how wrong we were | "Check the answer key" |
| `loss.backward()` | **Backward pass**: Compute gradients | "Figure out what went wrong" |
| `opt.step()` | Update weights using gradients | "Adjust your understanding" |
| `opt.zero_grad()` | Reset gradients for next batch | "Clear your scratch paper" |

#### Visual Flow

```
    ┌─────────────┐
    │ Input Batch │ (xb: 256 images)
    │   28×28×1   │
    └──────┬──────┘
           │
           ▼ Forward Pass
    ┌─────────────┐
    │    Model    │ (CNN)
    └──────┬──────┘
           │
           ▼
    ┌─────────────┐
    │ Predictions │ (preds: 256 × 10 scores)
    └──────┬──────┘
           │
           ▼ Compare with labels (yb)
    ┌─────────────┐
    │    Loss     │ (single number: how wrong?)
    └──────┬──────┘
           │
           ▼ Backward Pass
    ┌─────────────┐
    │  Gradients  │ (which direction to adjust?)
    └──────┬──────┘
           │
           ▼ Optimizer Step
    ┌─────────────┐
    │Update Weights│ (make model better)
    └─────────────┘
```

---

### Phase 2: Validation

```python
model.eval()
with torch.no_grad():
    tot_loss, tot_acc, count = 0., 0., 0
    for xb, yb in valid_dl:
        preds = model(xb)
        n = len(xb)
        count += n
        tot_loss += loss_func(preds, yb).item() * n
        tot_acc += accuracy(preds, yb).item() * n
```

#### Key Differences from Training

| Aspect | Training | Validation |
|--------|----------|------------|
| Mode | `model.train()` | `model.eval()` |
| Gradients | Computed | `torch.no_grad()` — disabled |
| Weight updates | Yes | No |
| Purpose | Learn | Evaluate |

#### Why These Differences?

1. **`model.eval()`**: Disables dropout and uses running statistics for batch normalization — gives consistent predictions

2. **`torch.no_grad()`**: Saves memory and computation since we don't need gradients for evaluation

3. **No `backward()` or `opt.step()`**: We're just measuring, not learning

#### Accumulating Statistics

```python
tot_loss += loss_func(preds, yb).item() * n
tot_acc += accuracy(preds, yb).item() * n
count += n
```

We multiply by `n` (batch size) because we want weighted averages:

```
Batch 1: 256 images, 85% accuracy → contributes 256 × 0.85 = 217.6
Batch 2: 256 images, 90% accuracy → contributes 256 × 0.90 = 230.4
...
Final: (217.6 + 230.4 + ...) / total_images = weighted average
```

---

## Complete Flow Diagram

```
                    ┌─────────────────────────────────────┐
                    │           EPOCH 0                   │
                    └─────────────────────────────────────┘
                                    │
          ┌─────────────────────────┴─────────────────────────┐
          ▼                                                   ▼
┌─────────────────────┐                         ┌─────────────────────┐
│   TRAINING PHASE    │                         │  VALIDATION PHASE   │
│                     │                         │                     │
│ For each batch:     │                         │ For each batch:     │
│  • Forward pass     │                         │  • Forward pass     │
│  • Compute loss     │                         │  • Compute loss     │
│  • Backward pass    │                         │  • Compute accuracy │
│  • Update weights   │                         │  • Accumulate stats │
│                     │                         │                     │
│ (Learning happens!) │                         │ (No learning!)      │
└─────────────────────┘                         └──────────┬──────────┘
                                                           │
                                                           ▼
                                                ┌─────────────────────┐
                                                │ Print: loss=0.52,   │
                                                │ accuracy=0.82       │
                                                └─────────────────────┘
                                                           │
                                                           ▼
                                                      Epoch 1...
```

---

## Why Separate Training and Validation?

| Training Data | Validation Data |
|--------------|-----------------|
| Model learns from this | Model never learns from this |
| Can memorize patterns | Tests generalization |
| Loss goes down (expected) | Loss should also go down |

**If training loss ↓ but validation loss ↑** → Overfitting! Model memorized training data but can't generalize.

---

## Example Output

```
Epoch 0: loss=0.8234, accuracy=0.7156
Epoch 1: loss=0.5621, accuracy=0.8012
Epoch 2: loss=0.4892, accuracy=0.8298
Epoch 3: loss=0.4501, accuracy=0.8445
Epoch 4: loss=0.4289, accuracy=0.8523
```

Each epoch:
- Loss decreases → Model is making smaller errors
- Accuracy increases → Model is getting more predictions right

---

## Key Takeaways

1. **Training loop = Forward → Loss → Backward → Update** (repeat for all batches)

2. **Validation = Forward → Measure** (no learning, just evaluation)

3. **`model.train()` vs `model.eval()`** — Different behaviors for training vs inference

4. **`torch.no_grad()`** — Saves memory during validation

5. **Epochs** — One epoch = one complete pass through all training data

# Understanding Validation Accumulation: `.item()` and `* n`

## The Two Questions

```python
tot_loss += loss_func(preds, yb).item() * n
tot_acc  += accuracy(preds, yb).item() * n
```

### 1. Why `.item()`?

`loss_func(preds, yb)` returns a **PyTorch tensor** (a single-element tensor on the GPU), not a plain Python number.

```python
loss = loss_func(preds, yb)
print(type(loss))   # <class 'torch.Tensor'>
print(loss)         # tensor(0.4523, device='cuda:0')
```

**Problem:** If you accumulate tensors in a loop, PyTorch **keeps the entire computation graph** in memory because it thinks you might want to call `.backward()` later. Over hundreds of batches, this **leaks GPU memory** massively.

`.item()` extracts the value as a **plain Python float**, breaking it free from the computation graph:

```python
loss.item()         # 0.4523 (plain float, no graph, no GPU memory held)
```

> **Rule of thumb:** During validation (where you don't need gradients), always use `.item()` when accumulating scalar values to avoid memory leaks.

---

### 2. Why multiply by `n`?

This is about computing a **correct weighted average** when the last batch might be smaller.

#### The Problem

`loss_func` and `accuracy` both return **mean values for that batch**. But not all batches are the same size — the **last batch** is often smaller:

```
Batch 1: 64 samples → mean_loss = 0.45
Batch 2: 64 samples → mean_loss = 0.42
Batch 3: 64 samples → mean_loss = 0.38
Batch 4: 24 samples → mean_loss = 0.50  ← last batch, only 24 samples!
```

#### Wrong approach — simple average of means

```python
# WRONG: treats all batches equally
avg_loss = (0.45 + 0.42 + 0.38 + 0.50) / 4 = 0.4375
```

This gives the 24-sample batch **equal weight** to the 64-sample batches, which is mathematically incorrect.

#### Correct approach — weighted average

```python
# CORRECT: weight each batch by its size
# Step 1: Convert means back to totals (multiply by n)
tot_loss = (0.45 * 64) + (0.42 * 64) + (0.38 * 64) + (0.50 * 24) = 92.0

# Step 2: Divide by total count
count = 64 + 64 + 64 + 24 = 216
avg_loss = 92.0 / 216 = 0.4259
```

This is exactly what the code does:

```python
tot_loss += loss_func(preds, yb).item() * n   # mean × n = total for this batch
# ...later...
final_loss = tot_loss / count                  # grand total ÷ total samples = correct mean
```

---

## The Full Pattern Visualized

```
For each batch of size n:
    mean_loss = loss_func(preds, yb).item()    # e.g., 0.45
    batch_total = mean_loss * n                 # e.g., 0.45 × 64 = 28.8
    tot_loss += batch_total                     # accumulate totals
    count += n                                  # accumulate sample count

After all batches:
    correct_avg = tot_loss / count              # weighted average across ALL samples
```

This **multiply-by-n then divide-by-total** pattern is the standard way to compute correct dataset-level metrics when batch sizes vary.

In [ ]:
# =====================================================
# TRAIN THE CLASSIFIER
# =====================================================

# Create optimizer
opt = optim.SGD(cnn.parameters(), lr=lr)

# Train for 5 epochs
print("Training CNN classifier on Fashion MNIST...")
print("=" * 50)
loss, acc = fit(5, cnn, F.cross_entropy, opt, dt, dv)

print("=" * 50)
print(f"Final accuracy: {acc:.2%}")

**Fashion MNIST is harder than regular MNIST**, so ~85% accuracy is reasonable for this simple architecture. The important thing is that our data pipeline and training loop work correctly.

---

### Use more CPUs

In [ ]:
def collate_(b):
    return to_device(cf(b))

def data_loaders(dsd, bs, **kwargs):
    return {k: DataLoader(v, bs, num_workers=8, **kwargs) for k, v in dsd.items()}

# But putting things into device as done by collate_ is incompatible with num_workers

The answer to this problem is that we have to rewrite out `fit` function.

In [ ]:
dls = data_loaders(tds, bs, collate_fn=collate_)

In [ ]:
dt = dls['train']
dv = dls['valid']

xb, yb = next(iter(dt))

In [ ]:
labels = ds.features[y].names

In [ ]:
lbl_getter = itemgetter(*yb[:16])
titles = lbl_getter(labels)

---

# Part 4: Building the Autoencoder

---

## Autoencoder Architecture Overview

An autoencoder has two parts:

### 1. Encoder (Compression)
- Takes the input image
- Uses convolutional layers to **reduce spatial size**
- Produces a small "latent" representation

### 2. Decoder (Reconstruction)
- Takes the latent representation
- Uses "deconvolution" layers to **increase spatial size**
- Produces a reconstruction of the original image

```
Input (28×28) → [Encoder] → Latent (small) → [Decoder] → Output (28×28)
                   ↓                              ↓
           Conv layers with              Upsampling layers that
           stride=2 (shrink)             double size (expand)
```

---

## The "Deconvolution" Layer

The decoder needs to **increase** the spatial size. We call this "deconvolution" or "upconvolution", but technically it's:
1. **Upsampling** - Double the size by repeating pixels
2. **Convolution** - Apply a regular conv to smooth/refine

This is more stable than using `nn.ConvTranspose2d` (true deconvolution), which can cause "checkerboard artifacts".

In [ ]:
def deconv(ni, nf, ks=3, act=True):
    """
    Refactored upsampling layer (Deconvolution via Interpolation + Conv).
    
    Explanation of Spatial Dimensions:
    1. nn.UpsamplingNearest2d(scale_factor=2): 
       Doubles the height and width (e.g., 8x8 -> 16x16) by repeating values.
       
    2. nn.Conv2d(..., stride=1, padding=ks//2):
       With stride=1 and 'same' padding (ks//2), the spatial dimensions 
       are PRESERVED. It does NOT reduce the size. 
       If input is 16x16, output remains 16x16.
    """
    layers = [
        # Step 1: Double the size (e.g., 8x8 -> 16x16)
        nn.UpsamplingNearest2d(scale_factor=2),
        
        # Step 2: Convolve to learn features on the larger grid.
        # stride=1: ensures we don't downsample (size stays 16x16).
        # padding=ks//2: offsets the kernel width to keep dimensions identical.
        nn.Conv2d(ni, nf, stride=1, kernel_size=ks, padding=ks//2)
    ]
    
    if act: layers.append(nn.ReLU())
    return nn.Sequential(*layers)

# Understanding the `deconv` Function

## Overview

The `deconv` function performs **upsampling** — the opposite of what `conv` does. While `conv` shrinks spatial dimensions, `deconv` expands them.

```python
def deconv(ni, nf, ks=3, act=True):
    layers = [
        nn.UpsamplingNearest2d(scale_factor=2),  # Step 1: Double size
        nn.Conv2d(ni, nf, stride=1, kernel_size=ks, padding=ks//2)  # Step 2: Refine
    ]
    if act: layers.append(nn.ReLU())
    return nn.Sequential(*layers)
```

---

## The Two-Step Process

### Step 1: Upsample (Double the Size)

```python
nn.UpsamplingNearest2d(scale_factor=2)
```

This doubles the spatial dimensions by **repeating each pixel**:

```
Input (2×2):          Output (4×4):
┌───┬───┐            ┌───┬───┬───┬───┐
│ A │ B │            │ A │ A │ B │ B │
├───┼───┤     →      ├───┼───┼───┼───┤
│ C │ D │            │ A │ A │ B │ B │
└───┴───┘            ├───┼───┼───┼───┤
                     │ C │ C │ D │ D │
                     ├───┼───┼───┼───┤
                     │ C │ C │ D │ D │
                     └───┴───┴───┴───┘
```

**Problem**: The output looks "blocky" — we just copied pixels. This is where convolution helps.

---

### Step 2: Convolve (Refine the Features)

```python
nn.Conv2d(ni, nf, stride=1, kernel_size=ks, padding=ks//2)
```

With default `ks=3`, this becomes:

```python
nn.Conv2d(ni, nf, stride=1, kernel_size=3, padding=1)
```

This convolution:
1. **Smooths** the blocky upsampled image
2. **Learns** useful features on the larger grid
3. **Preserves** the spatial dimensions (key point!)

---

## Why Does Conv2d Preserve Spatial Dimensions?

### The Parameters

| Parameter | Value | Purpose |
|-----------|-------|---------|
| `stride=1` | 1 | Don't skip any positions |
| `kernel_size=3` | 3×3 | Size of the sliding window |
| `padding=1` | 1 | Add 1 pixel border of zeros |

### The Output Size Formula

$$\text{output} = \left\lfloor \frac{\text{input} + 2 \times \text{padding} - \text{kernel}}{\text{stride}} \right\rfloor + 1$$

For a 16×16 input with our parameters:

$$\text{output} = \left\lfloor \frac{16 + 2(1) - 3}{1} \right\rfloor + 1 = \left\lfloor \frac{15}{1} \right\rfloor + 1 = 16$$

**The output is the same size as the input!**

### Why `padding = ks // 2` Works

This is a general formula for "same" padding:

| Kernel Size | Padding (`ks//2`) | Output Size |
|-------------|-------------------|-------------|
| 3 | 1 | Same as input |
| 5 | 2 | Same as input |
| 7 | 3 | Same as input |

The padding compensates for the kernel "eating into" the edges.

---

## Visual Walkthrough

### Without Padding (padding=0)

```
Input: 16×16
Kernel: 3×3, stride=1, padding=0

The 3×3 kernel can only fit in positions where it doesn't hang off the edge:
  - First valid position: starts at pixel (0,0), covers (0-2, 0-2)
  - Last valid position: starts at pixel (13,13), covers (13-15, 13-15)
  - Total positions: 14 × 14

Output: 14×14 (we lost 2 pixels!)
```

### With Padding (padding=1)

```
Input: 16×16
Kernel: 3×3, stride=1, padding=1

Step 1: Add 1-pixel border of zeros
  - Padded size: 18×18

Step 2: Apply kernel
  - First valid position: starts at pixel (0,0) of padded image
  - Last valid position: starts at pixel (15,15) of padded image
  - Total positions: 16 × 16

Output: 16×16 (same as input!)
```

### Diagram

```
Original 4×4 input:           After padding=1 (now 6×6):
┌───┬───┬───┬───┐            ┌───┬───┬───┬───┬───┬───┐
│ a │ b │ c │ d │            │ 0 │ 0 │ 0 │ 0 │ 0 │ 0 │
├───┼───┼───┼───┤            ├───┼───┼───┼───┼───┼───┤
│ e │ f │ g │ h │            │ 0 │ a │ b │ c │ d │ 0 │
├───┼───┼───┼───┤     →      ├───┼───┼───┼───┼───┼───┤
│ i │ j │ k │ l │            │ 0 │ e │ f │ g │ h │ 0 │
├───┼───┼───┼───┤            ├───┼───┼───┼───┼───┼───┤
│ m │ n │ o │ p │            │ 0 │ i │ j │ k │ l │ 0 │
└───┴───┴───┴───┘            ├───┼───┼───┼───┼───┼───┤
                             │ 0 │ m │ n │ o │ p │ 0 │
                             ├───┼───┼───┼───┼───┼───┤
                             │ 0 │ 0 │ 0 │ 0 │ 0 │ 0 │
                             └───┴───┴───┴───┴───┴───┘

Now 3×3 kernel can start at position (0,0) and still produce 4×4 output!
```

---

## How Channels Change

While spatial dimensions are **preserved**, the number of channels **changes**:

```
Input:  (ni channels, H, W)     e.g., (4, 16, 16)
Output: (nf channels, H, W)     e.g., (2, 16, 16)
```

Each of the `nf` filters produces one output channel:

```
                     Filter 0 ──→ Output Channel 0
Input (4×16×16) ──→  Filter 1 ──→ Output Channel 1
                     
                     Output: (2, 16, 16)
```

---

## Complete `deconv` Transformation

Let's trace through `deconv(4, 2)` with an 8×8 input:

```
Step 1: UpsamplingNearest2d(scale_factor=2)
┌─────────────────┐         ┌─────────────────┐
│   (4, 8, 8)     │   ──→   │   (4, 16, 16)   │
│   4 channels    │         │   4 channels    │
│   8×8 spatial   │         │   16×16 spatial │
└─────────────────┘         └─────────────────┘
        Doubles spatial size, channels unchanged


Step 2: Conv2d(4, 2, stride=1, kernel_size=3, padding=1)
┌─────────────────┐         ┌─────────────────┐
│   (4, 16, 16)   │   ──→   │   (2, 16, 16)   │
│   4 channels    │         │   2 channels    │
│   16×16 spatial │         │   16×16 spatial │
└─────────────────┘         └─────────────────┘
        Spatial preserved, channels: 4 → 2


Step 3: ReLU (if act=True)
┌─────────────────┐         ┌─────────────────┐
│   (2, 16, 16)   │   ──→   │   (2, 16, 16)   │
│   may have      │         │   all values    │
│   negative vals │         │   ≥ 0           │
└─────────────────┘         └─────────────────┘
        Shape unchanged, negative values → 0
```

---

## Comparison: `conv` vs `deconv`

| Aspect | `conv` (Encoder) | `deconv` (Decoder) |
|--------|------------------|-------------------|
| **Spatial change** | Halves (stride=2) | Doubles (upsample) |
| **Channel change** | Usually increases | Usually decreases |
| **Purpose** | Compress | Expand |
| **In autoencoder** | Shrink to bottleneck | Reconstruct from bottleneck |

### In the Autoencoder

```
Encoder (conv):                    Decoder (deconv):
(1, 32, 32)                        (4, 8, 8)
    │ conv(1, 2)                       │ deconv(4, 2)
    ▼                                  ▼
(2, 16, 16)                        (2, 16, 16)
    │ conv(2, 4)                       │ deconv(2, 1)
    ▼                                  ▼
(4, 8, 8)   ─── Bottleneck ───→   (1, 32, 32)
```

---

## Why Not Just Use ConvTranspose2d?

PyTorch has `nn.ConvTranspose2d` which does "true" deconvolution. Why use Upsample + Conv instead?

### The Checkerboard Problem

`ConvTranspose2d` can create **checkerboard artifacts**:

```
Expected output:          ConvTranspose2d output:
┌─────────────────┐       ┌─────────────────┐
│ ░░░░░░░░░░░░░░░ │       │ ░▓░▓░▓░▓░▓░▓░▓░ │
│ ░░░░░░░░░░░░░░░ │       │ ▓░▓░▓░▓░▓░▓░▓░▓ │
│ ░░░░░░░░░░░░░░░ │       │ ░▓░▓░▓░▓░▓░▓░▓░ │
│ ░░░░░░░░░░░░░░░ │       │ ▓░▓░▓░▓░▓░▓░▓░▓ │
└─────────────────┘       └─────────────────┘
   Smooth                    Checkerboard!
```

This happens due to uneven overlap when the kernel "deconvolves."

### Upsample + Conv Avoids This

| Method | Pros | Cons |
|--------|------|------|
| `ConvTranspose2d` | Learnable upsampling | Checkerboard artifacts |
| `Upsample + Conv` | No artifacts, stable | Slightly more computation |

The `Upsample + Conv` approach is now standard in most modern architectures.

---

## Key Takeaways

1. **`stride=1`** — Kernel visits every position, no spatial reduction

2. **`padding=ks//2`** — Compensates for kernel size, preserves dimensions

3. **Two-step process** — Upsample first (double size), then convolve (refine features)

4. **Channel change is independent** — `nf` filters create `nf` output channels regardless of spatial operations

5. **Avoids artifacts** — This approach is more stable than `ConvTranspose2d`

### 🔍 Visualize It: "Same" Padding

Slide the **input size**, **kernel size**, and **padding** to see exactly why `padding = ks//2` makes a stride-1 convolution preserve the spatial size. The zero border and the corner kernel position show *why* it works.

In [ ]:
# ============================================================================
# INTERACTIVE VISUALIZATION (self-contained) -- run this cell.
# Why padding = ks//2 preserves spatial dimensions for a stride-1 conv.
# The widget lives in the separate file `padding_playground.html`; it is embedded here as a
# base64 data-URI iframe, so the notebook stays fully self-contained and works
# offline in Jupyter, Colab, VS Code, and the exported HTML. The iframe
# broadcasts its own content height, and the listener below resizes it to fit
# exactly -- full width, wrapped to content, with no empty space below.
# ============================================================================
from IPython.display import HTML
import base64

_html_b64 = "PCFET0NUWVBFIGh0bWw+CjxodG1sIGxhbmc9ImVuIj4KPGhlYWQ+CjxtZXRhIGNoYXJzZXQ9IlVURi04Ij4KPG1ldGEgbmFtZT0idmlld3BvcnQiIGNvbnRlbnQ9IndpZHRoPWRldmljZS13aWR0aCwgaW5pdGlhbC1zY2FsZT0xLjAiPgo8dGl0bGU+U2FtZSBQYWRkaW5nIFBsYXlncm91bmQ8L3RpdGxlPgo8c3R5bGU+CiAgOnJvb3R7CiAgICAtLWJnOiNFREYyRkI7IC0tY2FyZDojRkZGRkZGOyAtLWluazojMUYyQTQ0OyAtLW11dGVkOiM2NDc0OEI7CiAgICAtLXB1cnBsZTojNkM1Q0U3OyAtLXB1cnBsZS1zb2Z0OiNFRkVCRkY7CiAgICAtLXRlYWw6IzBFOUM4RjsgLS10ZWFsLXNvZnQ6I0UwRjVGMjsKICAgIC0tb3JhbmdlOiNFODgyMUY7IC0tb3JhbmdlLXNvZnQ6I0ZERUVEQzsKICAgIC0tbGluZTojRERFNUYyOyAtLXplcm86I2NmZTBmNzsKICAgIC0tc2hhZG93OjAgMTBweCAyOHB4IHJnYmEoMTA4LDkyLDIzMSwuMTQpOwogICAgLS1zaGFkb3ctc206MCA0cHggMTRweCByZ2JhKDEwOCw5MiwyMzEsLjEwKTsKICAgIC0tcmFkaXVzOjE2cHg7CiAgICAtLW1vbm86IlNGIE1vbm8iLHVpLW1vbm9zcGFjZSxNZW5sbyxDb25zb2xhcyxtb25vc3BhY2U7CiAgfQogICp7Ym94LXNpemluZzpib3JkZXItYm94O21hcmdpbjowO3BhZGRpbmc6MH0KICBib2R5e2JhY2tncm91bmQ6dmFyKC0tYmcpO2NvbG9yOnZhcigtLWluayk7CiAgICBmb250OjE1cHgvMS41NSAtYXBwbGUtc3lzdGVtLCJTZWdvZSBVSSIsSW50ZXIsUm9ib3RvLHNhbnMtc2VyaWY7CiAgICBwYWRkaW5nOjIycHggMTZweCAxOHB4O292ZXJmbG93LXg6aGlkZGVuO30KICAud3JhcHttYXgtd2lkdGg6MTAwMHB4O21hcmdpbjowIGF1dG99CiAgaGVhZGVyIGgxe2ZvbnQtc2l6ZToyM3B4O2ZvbnQtd2VpZ2h0OjgwMDtsZXR0ZXItc3BhY2luZzotLjAyZW19CiAgaGVhZGVyIGgxIC5obHtjb2xvcjp2YXIoLS1wdXJwbGUpfQogIGhlYWRlciBwLnN1Yntjb2xvcjp2YXIoLS1tdXRlZCk7bWFyZ2luLXRvcDo2cHg7bWF4LXdpZHRoOjc4MHB4fQogIGhlYWRlciBwLnN1YiBjb2Rle2ZvbnQtZmFtaWx5OnZhcigtLW1vbm8pO2ZvbnQtc2l6ZTouOTJlbTtiYWNrZ3JvdW5kOiNmZmY7Ym9yZGVyLXJhZGl1czo2cHg7cGFkZGluZzoxcHggNnB4O2JveC1zaGFkb3c6dmFyKC0tc2hhZG93LXNtKX0KCiAgLmNvbnRyb2xze2Rpc3BsYXk6ZmxleDthbGlnbi1pdGVtczpjZW50ZXI7Z2FwOjE2cHg7ZmxleC13cmFwOndyYXA7CiAgICBiYWNrZ3JvdW5kOnZhcigtLWNhcmQpO2JvcmRlci1yYWRpdXM6dmFyKC0tcmFkaXVzKTtib3gtc2hhZG93OnZhcigtLXNoYWRvdyk7cGFkZGluZzoxNHB4IDE4cHg7bWFyZ2luOjE4cHggMCAxNnB4fQogIC5jdGx7ZGlzcGxheTpmbGV4O2FsaWduLWl0ZW1zOmNlbnRlcjtnYXA6OXB4O2ZvbnQtc2l6ZToxM3B4O2ZvbnQtd2VpZ2h0OjcwMDtjb2xvcjp2YXIoLS1tdXRlZCl9CiAgLmN0bCBpbnB1dHthY2NlbnQtY29sb3I6dmFyKC0tcHVycGxlKTt3aWR0aDoxMjBweH0KICAuY3RsIC52e2ZvbnQtZmFtaWx5OnZhcigtLW1vbm8pO2ZvbnQtd2VpZ2h0OjgwMDtjb2xvcjp2YXIoLS1wdXJwbGUpO21pbi13aWR0aDoyOHB4O3RleHQtYWxpZ246Y2VudGVyfQogIC5zZWd7ZGlzcGxheTppbmxpbmUtZmxleDtiYWNrZ3JvdW5kOiNFRkYzRkI7Ym9yZGVyLXJhZGl1czoxMXB4O3BhZGRpbmc6M3B4O2dhcDozcHh9CiAgLnNlZyBidXR0b257Ym9yZGVyOm5vbmU7YmFja2dyb3VuZDp0cmFuc3BhcmVudDtmb250OmluaGVyaXQ7Zm9udC13ZWlnaHQ6NzAwO2JvcmRlci1yYWRpdXM6OXB4O3BhZGRpbmc6NnB4IDEzcHg7Y3Vyc29yOnBvaW50ZXI7Y29sb3I6dmFyKC0tbXV0ZWQpfQogIC5zZWcgYnV0dG9uLm9ue2JhY2tncm91bmQ6dmFyKC0tcHVycGxlKTtjb2xvcjojZmZmO2JveC1zaGFkb3c6dmFyKC0tc2hhZG93LXNtKX0KCiAgLmdyaWR7ZGlzcGxheTpncmlkO2dyaWQtdGVtcGxhdGUtY29sdW1uczoxLjFmciAuOWZyO2dhcDoxOHB4O2FsaWduLWl0ZW1zOnN0YXJ0fQogIC5ncmlkLmlzLW5hcnJvd3tncmlkLXRlbXBsYXRlLWNvbHVtbnM6MWZyfQogIC5jYXJke2JhY2tncm91bmQ6dmFyKC0tY2FyZCk7Ym9yZGVyLXJhZGl1czp2YXIoLS1yYWRpdXMpO2JveC1zaGFkb3c6dmFyKC0tc2hhZG93KTtwYWRkaW5nOjE2cHh9CiAgLmNhcmQgaDJ7Zm9udC1zaXplOjEycHg7Zm9udC13ZWlnaHQ6ODAwO2xldHRlci1zcGFjaW5nOi4wOGVtO3RleHQtdHJhbnNmb3JtOnVwcGVyY2FzZTtjb2xvcjp2YXIoLS1tdXRlZCk7bWFyZ2luLWJvdHRvbToxMnB4fQogIGNhbnZhc3t3aWR0aDoxMDAlO21heC13aWR0aDozNDBweDtoZWlnaHQ6YXV0bztkaXNwbGF5OmJsb2NrO21hcmdpbjowIGF1dG87YmFja2dyb3VuZDojZmJmZGZmO2JvcmRlci1yYWRpdXM6MTBweH0KICAubGVnZW5ke2Rpc3BsYXk6ZmxleDtnYXA6MTRweDtqdXN0aWZ5LWNvbnRlbnQ6Y2VudGVyO2ZsZXgtd3JhcDp3cmFwO21hcmdpbi10b3A6MTBweDtmb250LXNpemU6MTJweDtjb2xvcjp2YXIoLS1tdXRlZCl9CiAgLmxlZ2VuZCBzcGFue2Rpc3BsYXk6aW5saW5lLWZsZXg7YWxpZ24taXRlbXM6Y2VudGVyO2dhcDo2cHh9CiAgLnN3e3dpZHRoOjEzcHg7aGVpZ2h0OjEzcHg7Ym9yZGVyLXJhZGl1czozcHg7ZGlzcGxheTppbmxpbmUtYmxvY2t9CiAgLnN3LnJlYWx7YmFja2dyb3VuZDojYWFjNGVhfS5zdy56ZXJve2JhY2tncm91bmQ6dmFyKC0temVybyk7Ym9yZGVyOjFweCBkYXNoZWQgIzdmYThlMH0uc3cua3tiYWNrZ3JvdW5kOnJnYmEoMTA4LDkyLDIzMSwuMjUpO2JvcmRlcjoycHggc29saWQgdmFyKC0tcHVycGxlKX0KCiAgLmZvcm11bGF7YmFja2dyb3VuZDojRjZGOEZFO2JvcmRlci1sZWZ0OjRweCBzb2xpZCB2YXIoLS1wdXJwbGUpO2JvcmRlci1yYWRpdXM6MTBweDtwYWRkaW5nOjEycHggMTVweDtmb250LWZhbWlseTp2YXIoLS1tb25vKTtmb250LXNpemU6MTMuNXB4O2xpbmUtaGVpZ2h0OjEuODtvdmVyZmxvdy14OmF1dG99CiAgLmZvcm11bGEgYntjb2xvcjp2YXIoLS1wdXJwbGUpfSAuZm9ybXVsYSAucmVze2ZvbnQtd2VpZ2h0OjgwMH0KICAucmVzLnNhbWV7Y29sb3I6dmFyKC0tdGVhbCl9IC5yZXMuc2hydW5re2NvbG9yOiNkNjQ1NWN9CiAgLnZlcmRpY3R7bWFyZ2luLXRvcDoxMnB4O2JvcmRlci1yYWRpdXM6MTJweDtwYWRkaW5nOjExcHggMTRweDtmb250LXdlaWdodDo3MDA7Zm9udC1zaXplOjE0cHh9CiAgLnZlcmRpY3Quc2FtZXtiYWNrZ3JvdW5kOnZhcigtLXRlYWwtc29mdCk7Y29sb3I6dmFyKC0tdGVhbCl9CiAgLnZlcmRpY3Quc2hydW5re2JhY2tncm91bmQ6I2ZkZThlYztjb2xvcjojZDY0NTVjfQogIC5ub3Rle2NvbG9yOnZhcigtLW11dGVkKTtmb250LXNpemU6MTIuNXB4O21hcmdpbi10b3A6MTJweDtsaW5lLWhlaWdodDoxLjU1fQogIC5ub3RlIGJ7Y29sb3I6dmFyKC0tcHVycGxlKX0KICB0YWJsZXt3aWR0aDoxMDAlO2JvcmRlci1jb2xsYXBzZTpjb2xsYXBzZTtmb250LWZhbWlseTp2YXIoLS1tb25vKTtmb250LXNpemU6MTIuNXB4O21hcmdpbi10b3A6MTJweH0KICB0aCx0ZHtwYWRkaW5nOjVweCA4cHg7dGV4dC1hbGlnbjpjZW50ZXI7Ym9yZGVyLWJvdHRvbToxcHggc29saWQgI2VlZjJmOH0KICB0aHtjb2xvcjp2YXIoLS1tdXRlZCk7Zm9udC13ZWlnaHQ6NzAwfQogIHRyLmhse2JhY2tncm91bmQ6dmFyKC0tcHVycGxlLXNvZnQpfSB0ci5obCB0ZHtjb2xvcjp2YXIoLS1wdXJwbGUpO2ZvbnQtd2VpZ2h0OjgwMH0KICBmb290ZXJ7bWFyZ2luLXRvcDoxNnB4O2NvbG9yOnZhcigtLW11dGVkKTtmb250LXNpemU6MTIuNXB4O3RleHQtYWxpZ246Y2VudGVyfQogIEBtZWRpYSAocHJlZmVycy1yZWR1Y2VkLW1vdGlvbjogcmVkdWNlKXsqe3RyYW5zaXRpb246bm9uZSFpbXBvcnRhbnR9fQo8L3N0eWxlPgo8L2hlYWQ+Cjxib2R5Pgo8ZGl2IGNsYXNzPSJ3cmFwIj4KPGhlYWRlcj4KICA8aDE+IlNhbWUiIHBhZGRpbmc6IDxzcGFuIGNsYXNzPSJobCI+d2h5IHBhZGRpbmcgPSBrcyAvLyAyIGtlZXBzIHRoZSBzaXplPC9zcGFuPjwvaDE+CiAgPHAgY2xhc3M9InN1YiI+QSBzdHJpZGUtMSBjb252b2x1dGlvbiBub3JtYWxseSA8aT5zaHJpbmtzPC9pPiBhbiBpbWFnZSwgYmVjYXVzZSB0aGUga2VybmVsIGNhbid0IGhhbmcgb2ZmIHRoZSBlZGdlLgogIEFkZGluZyBhIGJvcmRlciBvZiB6ZXJvcyDigJQgPGNvZGU+cGFkZGluZyA9IGtzLy8yPC9jb2RlPiDigJQgZ2l2ZXMgdGhlIGtlcm5lbCByb29tIHRvIHN0YXJ0IGluIHRoZSBjb3JuZXIsIHNvIHRoZSBvdXRwdXQKICBjb21lcyBvdXQgdGhlIDxiIHN0eWxlPSJjb2xvcjp2YXIoLS10ZWFsKSI+c2FtZSBzaXplPC9iPiBhcyB0aGUgaW5wdXQuIFNsaWRlIHRoZSBjb250cm9scyBhbmQgd2F0Y2ggdGhlIG1hdGggcmVzb2x2ZS48L3A+CjwvaGVhZGVyPgoKPGRpdiBjbGFzcz0iY29udHJvbHMiPgogIDxkaXYgY2xhc3M9ImN0bCI+aW5wdXQKICAgIDxpbnB1dCB0eXBlPSJyYW5nZSIgaWQ9ImlucCIgbWluPSI0IiBtYXg9IjEwIiBzdGVwPSIxIiB2YWx1ZT0iNiI+PHNwYW4gY2xhc3M9InYiIGlkPSJpbnB2Ij42PC9zcGFuPjwvZGl2PgogIDxkaXYgY2xhc3M9ImN0bCI+a2VybmVsCiAgICA8ZGl2IGNsYXNzPSJzZWciIGlkPSJrc1NlZyI+CiAgICAgIDxidXR0b24gZGF0YS12PSIzIiBjbGFzcz0ib24iPjM8L2J1dHRvbj4KICAgICAgPGJ1dHRvbiBkYXRhLXY9IjUiPjU8L2J1dHRvbj4KICAgIDwvZGl2PgogIDwvZGl2PgogIDxkaXYgY2xhc3M9ImN0bCI+cGFkZGluZwogICAgPGlucHV0IHR5cGU9InJhbmdlIiBpZD0icGFkIiBtaW49IjAiIG1heD0iMyIgc3RlcD0iMSIgdmFsdWU9IjEiPjxzcGFuIGNsYXNzPSJ2IiBpZD0icGFkdiI+MTwvc3Bhbj48L2Rpdj4KPC9kaXY+Cgo8ZGl2IGNsYXNzPSJncmlkIiBpZD0iZ3JpZCI+CiAgPGRpdiBjbGFzcz0iY2FyZCI+CiAgICA8aDI+UGFkZGVkIGlucHV0ICsga2VybmVsIGNvdmVyYWdlPC9oMj4KICAgIDxjYW52YXMgaWQ9ImN2IiB3aWR0aD0iMzgwIiBoZWlnaHQ9IjM4MCI+PC9jYW52YXM+CiAgICA8ZGl2IGNsYXNzPSJsZWdlbmQiPgogICAgICA8c3Bhbj48aSBjbGFzcz0ic3cgcmVhbCI+PC9pPiByZWFsIHBpeGVsczwvc3Bhbj4KICAgICAgPHNwYW4+PGkgY2xhc3M9InN3IHplcm8iPjwvaT4gemVybyBwYWRkaW5nPC9zcGFuPgogICAgICA8c3Bhbj48aSBjbGFzcz0ic3cgayI+PC9pPiBrZXJuZWwgKGNvcm5lciBzdGFydCk8L3NwYW4+CiAgICA8L2Rpdj4KICA8L2Rpdj4KCiAgPGRpdiBjbGFzcz0iY2FyZCI+CiAgICA8aDI+T3V0cHV0LXNpemUgZm9ybXVsYTwvaDI+CiAgICA8ZGl2IGNsYXNzPSJmb3JtdWxhIiBpZD0iZm9ybXVsYSI+PC9kaXY+CiAgICA8ZGl2IGNsYXNzPSJ2ZXJkaWN0IiBpZD0idmVyZGljdCI+PC9kaXY+CiAgICA8ZGl2IGNsYXNzPSJub3RlIiBpZD0ibm90ZSI+PC9kaXY+CiAgICA8dGFibGU+CiAgICAgIDx0cj48dGg+a2VybmVsPC90aD48dGg+a3MgLy8gMjwvdGg+PHRoPnJlc3VsdCB2cyBpbnB1dDwvdGg+PC90cj4KICAgICAgPHRyIGlkPSJyMyI+PHRkPjM8L3RkPjx0ZD4xPC90ZD48dGQ+c2FtZTwvdGQ+PC90cj4KICAgICAgPHRyIGlkPSJyNSI+PHRkPjU8L3RkPjx0ZD4yPC90ZD48dGQ+c2FtZTwvdGQ+PC90cj4KICAgICAgPHRyIGlkPSJyNyI+PHRkPjc8L3RkPjx0ZD4zPC90ZD48dGQ+c2FtZTwvdGQ+PC90cj4KICAgIDwvdGFibGU+CiAgPC9kaXY+CjwvZGl2PgoKPGZvb3Rlcj5TZXQgcGFkZGluZyB0byAwIHRvIHNlZSB0aGUgb3V0cHV0IHNocmluazsgc2V0IGl0IHRvIDxjb2RlPmtzIC8vIDI8L2NvZGU+ICgxIGZvciBhIDPDlzMga2VybmVsKSB0byBnZXQgdGhlIHNpemUgYmFjayBleGFjdGx5LjwvZm9vdGVyPgo8L2Rpdj4KCjxzY3JpcHQ+CihmdW5jdGlvbigpewoidXNlIHN0cmljdCI7CmNvbnN0ICQ9aWQ9PmRvY3VtZW50LmdldEVsZW1lbnRCeUlkKGlkKTsKbGV0IE49NiwgS1M9MywgUEFEPTE7CgpmdW5jdGlvbiBvdXQoKXsgcmV0dXJuIE1hdGguZmxvb3IoKE4rMipQQUQtS1MpLzEpKzE7IH0KZnVuY3Rpb24gZHJhdygpewogIGNvbnN0IGN2PSQoJ2N2JyksZz1jdi5nZXRDb250ZXh0KCcyZCcpOwogIGNvbnN0IHRvdD1OKzIqUEFEOyBjb25zdCBTPWN2LndpZHRoL01hdGgubWF4KHRvdCwxKTsKICBnLmNsZWFyUmVjdCgwLDAsY3Yud2lkdGgsY3YuaGVpZ2h0KTsKICBmb3IobGV0IHk9MDt5PHRvdDt5KyspZm9yKGxldCB4PTA7eDx0b3Q7eCsrKXsKICAgIGNvbnN0IGlzWmVybyA9IHg8UEFEfHx5PFBBRHx8eD49UEFEK058fHk+PVBBRCtOOwogICAgZy5maWxsU3R5bGUgPSBpc1plcm8gPyAnI2RjZWJmZicgOiAnI2FhYzRlYSc7CiAgICBnLmZpbGxTdHlsZSA9IGlzWmVybyA/ICd2YXIoLS16ZXJvKScgOiAnI2FhYzRlYSc7CiAgICBnLmZpbGxTdHlsZSA9IGlzWmVybyA/IGdldENvbXB1dGVkU3R5bGUoZG9jdW1lbnQuZG9jdW1lbnRFbGVtZW50KS5nZXRQcm9wZXJ0eVZhbHVlKCctLXplcm8nKSA6ICcjYWFjNGVhJzsKICAgIGcuZmlsbFJlY3QoeCpTLHkqUyxTLFMpOwogICAgaWYoaXNaZXJvKXsgZy5maWxsU3R5bGU9J3JnYmEoNzAsMTEwLDE3MCwuNTUpJztnLmZvbnQ9YGJvbGQgJHtNYXRoLm1pbigxNixTKjAuNSl9cHggbW9ub3NwYWNlYDtnLnRleHRBbGlnbj0nY2VudGVyJztnLnRleHRCYXNlbGluZT0nbWlkZGxlJztnLmZpbGxUZXh0KCcwJyx4KlMrUy8yLHkqUytTLzIpO30KICB9CiAgLy8gZ3JpZAogIGcuc3Ryb2tlU3R5bGU9J3JnYmEoMTIwLDE0MCwxODAsLjI1KSc7Zy5saW5lV2lkdGg9MTsKICBmb3IobGV0IGk9MDtpPD10b3Q7aSsrKXtnLmJlZ2luUGF0aCgpO2cubW92ZVRvKGkqUywwKTtnLmxpbmVUbyhpKlMsY3YuaGVpZ2h0KTtnLnN0cm9rZSgpO2cuYmVnaW5QYXRoKCk7Zy5tb3ZlVG8oMCxpKlMpO2cubGluZVRvKGN2LndpZHRoLGkqUyk7Zy5zdHJva2UoKTt9CiAgLy8gcmVhbC1yZWdpb24gYm9yZGVyCiAgZy5zdHJva2VTdHlsZT0nIzdmYThlMCc7Zy5saW5lV2lkdGg9MjtnLnN0cm9rZVJlY3QoUEFEKlMsUEFEKlMsTipTLE4qUyk7CiAgLy8ga2VybmVsIHN0YXJ0aW5nIGluIHRoZSBjb3JuZXIgKGNvdmVycyBwYWRkZWQgdG9wLWxlZnQpCiAgZy5maWxsU3R5bGU9J3JnYmEoMTA4LDkyLDIzMSwuMjApJztnLmZpbGxSZWN0KDAsMCxLUypTLEtTKlMpOwogIGcuc3Ryb2tlU3R5bGU9JyM2QzVDRTcnO2cubGluZVdpZHRoPTM7Zy5zdHJva2VSZWN0KDAsMCxLUypTLEtTKlMpOwp9CgpmdW5jdGlvbiByZW5kZXIoKXsKICBjb25zdCBPPW91dCgpOyBjb25zdCBzYW1lPShPPT09Tik7CiAgJCgnZm9ybXVsYScpLmlubmVySFRNTD0KICAgIGBvdXRwdXQgPSDijIooaW5wdXQgKyAywrc8Yj5wYWQ8L2I+IOKIkiA8Yj5rZXJuZWw8L2I+KSAvIHN0cmlkZeKMiyArIDE8YnI+YCsKICAgIGAmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDs9IOKMiigke059ICsgMsK3PGI+JHtQQUR9PC9iPiDiiJIgPGI+JHtLU308L2I+KSAvIDHijIsgKyAxPGJyPmArCiAgICBgJm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7PSAke04rMipQQUQtS1N9ICsgMSA9IDxzcGFuIGNsYXNzPSJyZXMgJHtzYW1lPydzYW1lJzonc2hydW5rJ30iPiR7T308L3NwYW4+YDsKICBjb25zdCB2PSQoJ3ZlcmRpY3QnKTsKICBpZihzYW1lKXsgdi5jbGFzc05hbWU9J3ZlcmRpY3Qgc2FtZSc7IHYudGV4dENvbnRlbnQ9YOKckyBPdXRwdXQgJHtPfcOXJHtPfSA9IGlucHV0ICR7Tn3DlyR7Tn0uIFNpemUgcHJlc2VydmVkIWA7IH0KICBlbHNlIGlmKE88Til7IHYuY2xhc3NOYW1lPSd2ZXJkaWN0IHNocnVuayc7IHYudGV4dENvbnRlbnQ9YOKclyBPdXRwdXQgJHtPfcOXJHtPfSA8IGlucHV0ICR7Tn3DlyR7Tn0uIFRoZSBpbWFnZSBzaHJhbmsgYnkgJHtOLU99LmA7IH0KICBlbHNlIHsgdi5jbGFzc05hbWU9J3ZlcmRpY3Qgc2hydW5rJzsgdi50ZXh0Q29udGVudD1gT3V0cHV0ICR7T33DlyR7T30gPiBpbnB1dCDigJQgdG9vIG11Y2ggcGFkZGluZy5gOyB9CiAgJCgnbm90ZScpLmlubmVySFRNTCA9IChQQUQ9PT1LUz4+MSkKICAgID8gYFlvdSBwaWNrZWQgPGI+cGFkZGluZyA9IGtzLy8yID0gJHtLUz4+MX08L2I+LCB0aGUgbWFnaWMgdmFsdWUuIFRoZSB6ZXJvcyBleGFjdGx5IGNvbXBlbnNhdGUgZm9yIHRoZSBrZXJuZWwgZWF0aW5nIGludG8gdGhlIGVkZ2VzLCBzbyBhIHN0cmlkZS0xIGNvbnYga2VlcHMgdGhlIHNwYXRpYWwgc2l6ZSBpZGVudGljYWwg4oCUIHdoaWNoIGlzIGV4YWN0bHkgd2hhdCB0aGUgcmVmaW5lLXN0ZXAgaW5zaWRlIDxjb2RlPmRlY29udjwvY29kZT4gcmVsaWVzIG9uLmAKICAgIDogYFRyeSA8Yj5wYWRkaW5nID0ga3MvLzIgPSAke0tTPj4xfTwvYj4gZm9yIHRoaXMga2VybmVsIHRvIGxhbmQgYmFjayBvbiB0aGUgc2FtZSBzaXplLmA7CiAgLy8gaGlnaGxpZ2h0IHRoZSBtYXRjaGluZyByb3cgaW4gdGhlIHRhYmxlCiAgWydyMycsJ3I1JywncjcnXS5mb3JFYWNoKHI9PiQocikuY2xhc3NMaXN0LnJlbW92ZSgnaGwnKSk7CiAgaWYoS1M9PT0zKSQoJ3IzJykuY2xhc3NMaXN0LmFkZCgnaGwnKTsgaWYoS1M9PT01KSQoJ3I1JykuY2xhc3NMaXN0LmFkZCgnaGwnKTsKICBkcmF3KCk7IHBvc3RIKCk7Cn0KCiQoJ2lucCcpLmFkZEV2ZW50TGlzdGVuZXIoJ2lucHV0JywoKT0+eyBOPSskKCdpbnAnKS52YWx1ZTsgJCgnaW5wdicpLnRleHRDb250ZW50PU47IHJlbmRlcigpOyB9KTsKJCgncGFkJykuYWRkRXZlbnRMaXN0ZW5lcignaW5wdXQnLCgpPT57IFBBRD0rJCgncGFkJykudmFsdWU7ICQoJ3BhZHYnKS50ZXh0Q29udGVudD1QQUQ7IHJlbmRlcigpOyB9KTsKJCgna3NTZWcnKS5hZGRFdmVudExpc3RlbmVyKCdjbGljaycsZT0+e2NvbnN0IGI9ZS50YXJnZXQuY2xvc2VzdCgnYnV0dG9uJyk7aWYoIWIpcmV0dXJuO0tTPStiLmRhdGFzZXQudjsKICBkb2N1bWVudC5xdWVyeVNlbGVjdG9yQWxsKCcja3NTZWcgYnV0dG9uJykuZm9yRWFjaCh4PT54LmNsYXNzTGlzdC50b2dnbGUoJ29uJyx4PT09YikpO3JlbmRlcigpO30pOwoKY29uc3QgZ3JpZD0kKCdncmlkJyksIHdyYXA9ZG9jdW1lbnQucXVlcnlTZWxlY3RvcignLndyYXAnKTsKZnVuY3Rpb24gcmVsYXlvdXQoKXsgZ3JpZC5jbGFzc0xpc3QudG9nZ2xlKCdpcy1uYXJyb3cnLCB3aW5kb3cuaW5uZXJXaWR0aDw3ODApOyB9CmZ1bmN0aW9uIHBvc3RIKCl7IGNvbnN0IGg9d3JhcD9NYXRoLmNlaWwod3JhcC5nZXRCb3VuZGluZ0NsaWVudFJlY3QoKS5oZWlnaHQpKzM0OmRvY3VtZW50LmJvZHkub2Zmc2V0SGVpZ2h0OwogIGlmKHdpbmRvdy5wYXJlbnQhPT13aW5kb3cpIHdpbmRvdy5wYXJlbnQucG9zdE1lc3NhZ2Uoe3R5cGU6J2FlLWZyYW1lLWhlaWdodCcsaGVpZ2h0Omh9LCcqJyk7IH0KZnVuY3Rpb24gdXBkYXRlKCl7IHJlbGF5b3V0KCk7IHBvc3RIKCk7IH0Kd2luZG93LmFkZEV2ZW50TGlzdGVuZXIoJ2xvYWQnLHVwZGF0ZSk7IHdpbmRvdy5hZGRFdmVudExpc3RlbmVyKCdyZXNpemUnLHVwZGF0ZSk7CmlmKHdpbmRvdy5SZXNpemVPYnNlcnZlcikgbmV3IFJlc2l6ZU9ic2VydmVyKHBvc3RIKS5vYnNlcnZlKHdyYXB8fGRvY3VtZW50LmJvZHkpOwpyZWxheW91dCgpOyByZW5kZXIoKTsKc2V0VGltZW91dCh1cGRhdGUsMjAwKTsgc2V0VGltZW91dCh1cGRhdGUsNzAwKTsKfSkoKTsKPC9zY3JpcHQ+CjwvYm9keT4KPC9odG1sPgo="

HTML('''
<iframe id="padding-frame"
        src="data:text/html;base64,''' + _html_b64 + '''"
        style="width:100%; height:760px; border:1px solid #DDE5F2;
               border-radius:16px; box-shadow:0 8px 24px rgba(108,92,231,.12);
               display:block;"
        loading="lazy" title="Same padding playground"></iframe>
<script>
(function(){
  function onMsg(e){
    if (e.data && e.data.type === "ae-frame-height") {
      var f = document.getElementById("padding-frame");
      if (f) f.style.height = (e.data.height) + "px";
    }
  }
  window.addEventListener("message", onMsg);
})();
</script>
''')


### 🔍 Visualize It: Deconv = Upsample + Refine

Step through the two-step deconvolution on a tiny 2×2 patch: first nearest-neighbour **upsampling** (blocky copy), then a stride-1 **conv** that smooths it. The bottom panels compare `ConvTranspose2d` (checkerboard) with the `Upsample + Conv` approach this notebook uses.

In [ ]:
# ============================================================================
# INTERACTIVE VISUALIZATION (self-contained) -- run this cell.
# Two-step upsample-then-refine, and the checkerboard artifact comparison.
# The widget lives in the separate file `deconv_upsample_lab.html`; it is embedded here as a
# base64 data-URI iframe, so the notebook stays fully self-contained and works
# offline in Jupyter, Colab, VS Code, and the exported HTML. The iframe
# broadcasts its own content height, and the listener below resizes it to fit
# exactly -- full width, wrapped to content, with no empty space below.
# ============================================================================
from IPython.display import HTML
import base64

_html_b64 = "PCFET0NUWVBFIGh0bWw+CjxodG1sIGxhbmc9ImVuIj4KPGhlYWQ+CjxtZXRhIGNoYXJzZXQ9IlVURi04Ij4KPG1ldGEgbmFtZT0idmlld3BvcnQiIGNvbnRlbnQ9IndpZHRoPWRldmljZS13aWR0aCwgaW5pdGlhbC1zY2FsZT0xLjAiPgo8dGl0bGU+RGVjb252IFVwc2FtcGxlIExhYjwvdGl0bGU+CjxzdHlsZT4KICA6cm9vdHsKICAgIC0tYmc6I0VERjJGQjsgLS1jYXJkOiNGRkZGRkY7IC0taW5rOiMxRjJBNDQ7IC0tbXV0ZWQ6IzY0NzQ4QjsKICAgIC0tcHVycGxlOiM2QzVDRTc7IC0tcHVycGxlLXNvZnQ6I0VGRUJGRjsKICAgIC0tdGVhbDojMEU5QzhGOyAtLXRlYWwtc29mdDojRTBGNUYyOwogICAgLS1vcmFuZ2U6I0U4ODIxRjsgLS1vcmFuZ2Utc29mdDojRkRFRURDOwogICAgLS1saW5lOiNEREU1RjI7CiAgICAtLXNoYWRvdzowIDEwcHggMjhweCByZ2JhKDEwOCw5MiwyMzEsLjE0KTsKICAgIC0tc2hhZG93LXNtOjAgNHB4IDE0cHggcmdiYSgxMDgsOTIsMjMxLC4xMCk7CiAgICAtLXJhZGl1czoxNnB4OwogICAgLS1tb25vOiJTRiBNb25vIix1aS1tb25vc3BhY2UsTWVubG8sQ29uc29sYXMsbW9ub3NwYWNlOwogIH0KICAqe2JveC1zaXppbmc6Ym9yZGVyLWJveDttYXJnaW46MDtwYWRkaW5nOjB9CiAgYm9keXtiYWNrZ3JvdW5kOnZhcigtLWJnKTtjb2xvcjp2YXIoLS1pbmspOwogICAgZm9udDoxNXB4LzEuNTUgLWFwcGxlLXN5c3RlbSwiU2Vnb2UgVUkiLEludGVyLFJvYm90byxzYW5zLXNlcmlmOwogICAgcGFkZGluZzoyMnB4IDE2cHggMThweDtvdmVyZmxvdy14OmhpZGRlbjt9CiAgLndyYXB7bWF4LXdpZHRoOjExMjBweDttYXJnaW46MCBhdXRvfQogIGhlYWRlciBoMXtmb250LXNpemU6MjNweDtmb250LXdlaWdodDo4MDA7bGV0dGVyLXNwYWNpbmc6LS4wMmVtfQogIGhlYWRlciBoMSAuaGx7Y29sb3I6dmFyKC0tdGVhbCl9CiAgaGVhZGVyIHAuc3Vie2NvbG9yOnZhcigtLW11dGVkKTttYXJnaW4tdG9wOjZweDttYXgtd2lkdGg6ODQwcHh9CiAgaGVhZGVyIHAuc3ViIGNvZGV7Zm9udC1mYW1pbHk6dmFyKC0tbW9ubyk7Zm9udC1zaXplOi45MmVtO2JhY2tncm91bmQ6I2ZmZjtib3JkZXItcmFkaXVzOjZweDtwYWRkaW5nOjFweCA2cHg7Ym94LXNoYWRvdzp2YXIoLS1zaGFkb3ctc20pfQoKICAuY29udHJvbHN7ZGlzcGxheTpmbGV4O2FsaWduLWl0ZW1zOmNlbnRlcjtnYXA6MTJweDtmbGV4LXdyYXA6d3JhcDsKICAgIGJhY2tncm91bmQ6dmFyKC0tY2FyZCk7Ym9yZGVyLXJhZGl1czp2YXIoLS1yYWRpdXMpO2JveC1zaGFkb3c6dmFyKC0tc2hhZG93KTsKICAgIHBhZGRpbmc6MTNweCAxNnB4O21hcmdpbjoxOHB4IDAgMTZweDt9CiAgYnV0dG9ue2ZvbnQ6aW5oZXJpdDtmb250LXdlaWdodDo3MDA7Ym9yZGVyOm5vbmU7Ym9yZGVyLXJhZGl1czoxMnB4O2N1cnNvcjpwb2ludGVyO3BhZGRpbmc6OXB4IDE4cHg7dHJhbnNpdGlvbjp0cmFuc2Zvcm0gLjEyc30KICAjcHJldiwjbmV4dHtiYWNrZ3JvdW5kOnZhcigtLXB1cnBsZS1zb2Z0KTtjb2xvcjp2YXIoLS1wdXJwbGUpfQogICNwbGF5e2JhY2tncm91bmQ6dmFyKC0tdGVhbCk7Y29sb3I6I2ZmZjtib3gtc2hhZG93OjAgNnB4IDE2cHggcmdiYSgxNCwxNTYsMTQzLC4zMil9CiAgI3BsYXk6aG92ZXJ7dHJhbnNmb3JtOnRyYW5zbGF0ZVkoLTFweCl9CiAgI3Jlc2V0e2JhY2tncm91bmQ6dHJhbnNwYXJlbnQ7Y29sb3I6dmFyKC0tbXV0ZWQpO3RleHQtZGVjb3JhdGlvbjp1bmRlcmxpbmU7cGFkZGluZzo5cHggOHB4fQogIC5zdGVwbnVte2NvbG9yOnZhcigtLW11dGVkKTtmb250LXNpemU6MTNweDtmb250LXdlaWdodDo3MDA7bWluLXdpZHRoOjg4cHh9CiAgLmRvdHN7ZGlzcGxheTpmbGV4O2dhcDo2cHh9CiAgLmRvdHMgaXt3aWR0aDo5cHg7aGVpZ2h0OjlweDtib3JkZXItcmFkaXVzOjUwJTtiYWNrZ3JvdW5kOiNEN0RFRjA7dHJhbnNpdGlvbjouMnN9CiAgLmRvdHMgaS5vbntiYWNrZ3JvdW5kOnZhcigtLXRlYWwpO3RyYW5zZm9ybTpzY2FsZSgxLjI1KX0KICAuc3Bke2Rpc3BsYXk6ZmxleDthbGlnbi1pdGVtczpjZW50ZXI7Z2FwOjhweDttYXJnaW4tbGVmdDphdXRvO2ZvbnQtc2l6ZToxM3B4O2NvbG9yOnZhcigtLW11dGVkKTtmb250LXdlaWdodDo2MDB9CiAgLnNwZCBpbnB1dHthY2NlbnQtY29sb3I6dmFyKC0tcHVycGxlKTt3aWR0aDoxMTBweH0KCiAgLmdyaWR7ZGlzcGxheTpncmlkO2dyaWQtdGVtcGxhdGUtY29sdW1uczoxLjFmciAuOWZyO2dhcDoxOHB4O2FsaWduLWl0ZW1zOnN0YXJ0fQogIC5ncmlkLmlzLW5hcnJvd3tncmlkLXRlbXBsYXRlLWNvbHVtbnM6MWZyfQogIC5jYXJke2JhY2tncm91bmQ6dmFyKC0tY2FyZCk7Ym9yZGVyLXJhZGl1czp2YXIoLS1yYWRpdXMpO2JveC1zaGFkb3c6dmFyKC0tc2hhZG93KTtwYWRkaW5nOjE1cHggMTZweH0KICAuY2FyZCBoMntmb250LXNpemU6MTJweDtmb250LXdlaWdodDo4MDA7bGV0dGVyLXNwYWNpbmc6LjA4ZW07dGV4dC10cmFuc2Zvcm06dXBwZXJjYXNlO2NvbG9yOnZhcigtLW11dGVkKTttYXJnaW4tYm90dG9tOjEycHh9CgogIC5zdGFnZXdyYXB7ZGlzcGxheTpncmlkO2dyaWQtdGVtcGxhdGUtY29sdW1uczoxZnIgYXV0byAxZnI7Z2FwOjEycHg7YWxpZ24taXRlbXM6Y2VudGVyfQogIC5ncmlkYm94e3RleHQtYWxpZ246Y2VudGVyfQogIC5ncmlkYm94IGNhbnZhc3tiYWNrZ3JvdW5kOiNmZmY7Ym9yZGVyOjFweCBzb2xpZCB2YXIoLS1saW5lKTtib3JkZXItcmFkaXVzOjEwcHg7aW1hZ2UtcmVuZGVyaW5nOnBpeGVsYXRlZDt3aWR0aDoxMDAlO21heC13aWR0aDoyMDBweDtoZWlnaHQ6YXV0b30KICAuZ3JpZGJveCAudHtmb250LXNpemU6MTFweDtmb250LXdlaWdodDo3MDA7bWFyZ2luLXRvcDo1cHg7Y29sb3I6dmFyKC0tbXV0ZWQpfQogIC5hcnJvd21pZHtmb250LXNpemU6MjJweDtjb2xvcjp2YXIoLS10ZWFsKTt0ZXh0LWFsaWduOmNlbnRlcn0KICAuYXJyb3dtaWQgc21hbGx7ZGlzcGxheTpibG9jaztmb250LXNpemU6MTBweDtjb2xvcjp2YXIoLS1tdXRlZCk7Zm9udC13ZWlnaHQ6NzAwO21hcmdpbi10b3A6M3B4fQoKICAuc3RlcHRpdGxle2ZvbnQtc2l6ZToxOHB4O2ZvbnQtd2VpZ2h0OjgwMDttYXJnaW4tYm90dG9tOjdweH0KICAuZXhwbGFpbntjb2xvcjojMzM0MTVDfS5leHBsYWluIHB7bWFyZ2luOjAgMCA4cHh9CiAgLmV4cGxhaW4gY29kZXtmb250LWZhbWlseTp2YXIoLS1tb25vKTtmb250LXNpemU6LjllbTtiYWNrZ3JvdW5kOiNGMUY0RkM7Ym9yZGVyLXJhZGl1czo2cHg7cGFkZGluZzoxcHggNnB4fQogIC5leHBsYWluIGIudHtjb2xvcjp2YXIoLS10ZWFsKX0gLmV4cGxhaW4gYi5ve2NvbG9yOnZhcigtLW9yYW5nZSl9IC5leHBsYWluIGIucHtjb2xvcjp2YXIoLS1wdXJwbGUpfQogIHByZS5jb2Rle2ZvbnQtZmFtaWx5OnZhcigtLW1vbm8pO2ZvbnQtc2l6ZToxMi41cHg7bGluZS1oZWlnaHQ6MS42O2JhY2tncm91bmQ6I0Y2RjhGRTtib3JkZXItcmFkaXVzOjEycHg7cGFkZGluZzoxMXB4IDA7b3ZlcmZsb3cteDphdXRvO21hcmdpbi10b3A6MTJweH0KICAuY2x7cGFkZGluZzowIDE1cHg7d2hpdGUtc3BhY2U6cHJlfQogIC5jbC5obHtiYWNrZ3JvdW5kOnZhcigtLXRlYWwtc29mdCk7Ym9yZGVyLWxlZnQ6NHB4IHNvbGlkIHZhcigtLXRlYWwpO3BhZGRpbmctbGVmdDoxMXB4O2ZvbnQtd2VpZ2h0OjcwMH0KICAuY2wuY217Y29sb3I6IzhBOTRBQ30KCiAgLmNvbXBhcmV7ZGlzcGxheTpncmlkO2dyaWQtdGVtcGxhdGUtY29sdW1uczoxZnIgMWZyO2dhcDoxNHB4O21hcmdpbi10b3A6OHB4fQogIC5jb21wYXJlIC5jb2x7dGV4dC1hbGlnbjpjZW50ZXJ9CiAgLmNvbXBhcmUgY2FudmFze3dpZHRoOjEwMCU7bWF4LXdpZHRoOjE4MHB4O2hlaWdodDphdXRvO2JvcmRlci1yYWRpdXM6MTBweDtib3JkZXI6MXB4IHNvbGlkIHZhcigtLWxpbmUpO2ltYWdlLXJlbmRlcmluZzpwaXhlbGF0ZWR9CiAgLmNvbXBhcmUgLmh7Zm9udC1zaXplOjEycHg7Zm9udC13ZWlnaHQ6ODAwO21hcmdpbi1ib3R0b206NnB4fQogIC5jb21wYXJlIC5iYWR7Y29sb3I6I2Q2NDU1Y30uY29tcGFyZSAuZ29vZHtjb2xvcjp2YXIoLS10ZWFsKX0KICAuY29tcGFyZSAuZHtmb250LXNpemU6MTFweDtjb2xvcjp2YXIoLS1tdXRlZCk7bWFyZ2luLXRvcDo1cHh9CiAgZm9vdGVye21hcmdpbi10b3A6MTZweDtjb2xvcjp2YXIoLS1tdXRlZCk7Zm9udC1zaXplOjEyLjVweDt0ZXh0LWFsaWduOmNlbnRlcn0KICBAbWVkaWEgKHByZWZlcnMtcmVkdWNlZC1tb3Rpb246IHJlZHVjZSl7Knt0cmFuc2l0aW9uOm5vbmUhaW1wb3J0YW50fX0KPC9zdHlsZT4KPC9oZWFkPgo8Ym9keT4KPGRpdiBjbGFzcz0id3JhcCI+CjxoZWFkZXI+CiAgPGgxPkRlY29udiA9IDxzcGFuIGNsYXNzPSJobCI+dXBzYW1wbGUsIHRoZW4gcmVmaW5lPC9zcGFuPjwvaDE+CiAgPHAgY2xhc3M9InN1YiI+VGhlIGRlY29kZXIgZ3Jvd3MgaW1hZ2VzIGJhY2sgdXAuIEluc3RlYWQgb2Ygb25lIHRyaWNreSAidHJhbnNwb3NlZCBjb252b2x1dGlvbiIsIHdlIHVzZSBhIGNsZWFuCiAgdHdvLXN0ZXAgcmVjaXBlOiA8YiBzdHlsZT0iY29sb3I6dmFyKC0tdGVhbCkiPm5lYXJlc3QtbmVpZ2hib3VyIHVwc2FtcGxpbmc8L2I+IChqdXN0IGNvcHkgZWFjaCBwaXhlbCBpbnRvIGEgMsOXMiBibG9jaykKICBmb2xsb3dlZCBieSBhIHN0cmlkZS0xIDxiIHN0eWxlPSJjb2xvcjp2YXIoLS1wdXJwbGUpIj5jb252b2x1dGlvbjwvYj4gdGhhdCBzbW9vdGhzIGFuZCBsZWFybnMuIFN0ZXAgdGhyb3VnaCBpdCBiZWxvdy48L3A+CjwvaGVhZGVyPgoKPGRpdiBjbGFzcz0iY29udHJvbHMiPgogIDxidXR0b24gaWQ9InByZXYiPuKGkCBCYWNrPC9idXR0b24+CiAgPGJ1dHRvbiBpZD0icGxheSI+UGxheSDilrY8L2J1dHRvbj4KICA8YnV0dG9uIGlkPSJuZXh0Ij5OZXh0IOKGkjwvYnV0dG9uPgogIDxzcGFuIGNsYXNzPSJzdGVwbnVtIiBpZD0ic3RlcG51bSI+U3RlcCAwIC8gMzwvc3Bhbj4KICA8ZGl2IGNsYXNzPSJkb3RzIiBpZD0iZG90cyI+PC9kaXY+CiAgPGxhYmVsIGNsYXNzPSJzcGQiPnNwZWVkIDxpbnB1dCB0eXBlPSJyYW5nZSIgaWQ9InNwZCIgbWluPSIxIiBtYXg9IjYiIHN0ZXA9IjEiIHZhbHVlPSIzIj48c3BhbiBpZD0ic3BkdiI+MS4ww5c8L3NwYW4+PC9sYWJlbD4KICA8YnV0dG9uIGlkPSJyZXNldCI+UmVzZXQ8L2J1dHRvbj4KPC9kaXY+Cgo8ZGl2IGNsYXNzPSJncmlkIiBpZD0iZ3JpZCI+CiAgPGRpdiBjbGFzcz0iY2FyZCI+CiAgICA8aDI+VGhlIHR3by1zdGVwIGRlY29udiBvbiBhIDLDlzIgcGF0Y2g8L2gyPgogICAgPGRpdiBjbGFzcz0ic3RhZ2V3cmFwIj4KICAgICAgPGRpdiBjbGFzcz0iZ3JpZGJveCI+CiAgICAgICAgPGNhbnZhcyBpZD0ic3JjIiB3aWR0aD0iMTIwIiBoZWlnaHQ9IjEyMCI+PC9jYW52YXM+CiAgICAgICAgPGRpdiBjbGFzcz0idCIgaWQ9InNyY3QiPklucHV0IDLDlzI8L2Rpdj4KICAgICAgPC9kaXY+CiAgICAgIDxkaXYgY2xhc3M9ImFycm93bWlkIj7ilrY8c21hbGwgaWQ9Im1sIj5zdGVwPC9zbWFsbD48L2Rpdj4KICAgICAgPGRpdiBjbGFzcz0iZ3JpZGJveCI+CiAgICAgICAgPGNhbnZhcyBpZD0iZHN0IiB3aWR0aD0iMjQwIiBoZWlnaHQ9IjI0MCI+PC9jYW52YXM+CiAgICAgICAgPGRpdiBjbGFzcz0idCIgaWQ9ImRzdHQiPk91dHB1dCA0w5c0PC9kaXY+CiAgICAgIDwvZGl2PgogICAgPC9kaXY+CgogICAgPGgyIHN0eWxlPSJtYXJnaW4tdG9wOjE4cHgiPldoeSBub3QganVzdCBDb252VHJhbnNwb3NlMmQ/PC9oMj4KICAgIDxkaXYgY2xhc3M9ImNvbXBhcmUiPgogICAgICA8ZGl2IGNsYXNzPSJjb2wiPgogICAgICAgIDxkaXYgY2xhc3M9ImggYmFkIj5Db252VHJhbnNwb3NlMmQ8L2Rpdj4KICAgICAgICA8Y2FudmFzIGlkPSJjYiIgd2lkdGg9IjEyMCIgaGVpZ2h0PSIxMjAiPjwvY2FudmFzPgogICAgICAgIDxkaXYgY2xhc3M9ImQiPmNoZWNrZXJib2FyZCBhcnRpZmFjdHMgZnJvbSB1bmV2ZW4ga2VybmVsIG92ZXJsYXA8L2Rpdj4KICAgICAgPC9kaXY+CiAgICAgIDxkaXYgY2xhc3M9ImNvbCI+CiAgICAgICAgPGRpdiBjbGFzcz0iaCBnb29kIj5VcHNhbXBsZSArIENvbnY8L2Rpdj4KICAgICAgICA8Y2FudmFzIGlkPSJzbSIgd2lkdGg9IjEyMCIgaGVpZ2h0PSIxMjAiPjwvY2FudmFzPgogICAgICAgIDxkaXYgY2xhc3M9ImQiPnNtb290aCwgZXZlbiBjb3ZlcmFnZSDigJQgd2hhdCB0aGlzIG5vdGVib29rIHVzZXM8L2Rpdj4KICAgICAgPC9kaXY+CiAgICA8L2Rpdj4KICA8L2Rpdj4KCiAgPGRpdj4KICAgIDxkaXYgY2xhc3M9ImNhcmQiPgogICAgICA8ZGl2IGNsYXNzPSJzdGVwdGl0bGUiIGlkPSJ0aXRsZSI+PC9kaXY+CiAgICAgIDxkaXYgY2xhc3M9ImV4cGxhaW4iIGlkPSJleHBsYWluIj48L2Rpdj4KICAgICAgPHByZSBjbGFzcz0iY29kZSIgaWQ9ImNvZGUiPjwvcHJlPgogICAgPC9kaXY+CiAgPC9kaXY+CjwvZGl2PgoKPGZvb3Rlcj5QcmVzcyBQbGF5IHRvIHdhdGNoIHRoZSAyw5cyIHBhdGNoIGRvdWJsZSBpbiBzaXplIGFuZCB0aGVuIGdldCByZWZpbmVkLiBUaGUgY2hlY2tlcmJvYXJkIHBhbmVscyBvbiB0aGUgbGVmdCB1cGRhdGUgdG9vLjwvZm9vdGVyPgo8L2Rpdj4KCjxzY3JpcHQ+CihmdW5jdGlvbigpewoidXNlIHN0cmljdCI7CmNvbnN0ICQ9aWQ9PmRvY3VtZW50LmdldEVsZW1lbnRCeUlkKGlkKTsKY29uc3Qgc3BlZWRzPVswLjUsMC43NSwxLjAsMS41LDIuMCwzLjBdOwoKLy8gdGhlIDJ4MiBzb3VyY2UgdmFsdWVzCmNvbnN0IFNSQz1bWzAuODUsMC4zMF0sWzAuMjAsMC43MF1dOwpjb25zdCBsYWJlbHMyPVtbJ0EnLCdCJ10sWydDJywnRCddXTsKCmZ1bmN0aW9uIHZhbDJyZ2Iodil7Y29uc3QgYz1NYXRoLnJvdW5kKDI1NS12KjE4MCk7cmV0dXJuIGByZ2IoJHtjfSwke01hdGgubWluKDI1NSxjKzEyKX0sJHtNYXRoLm1pbigyNTUsYysyMil9KWA7fQoKZnVuY3Rpb24gZHJhd1NyYygpewogIGNvbnN0IGN2PSQoJ3NyYycpLGc9Y3YuZ2V0Q29udGV4dCgnMmQnKSxzPWN2LndpZHRoLzI7CiAgZm9yKGxldCB5PTA7eTwyO3krKylmb3IobGV0IHg9MDt4PDI7eCsrKXsKICAgIGcuZmlsbFN0eWxlPXZhbDJyZ2IoU1JDW3ldW3hdKTsgZy5maWxsUmVjdCh4KnMseSpzLHMscyk7CiAgICBnLmZpbGxTdHlsZT0ncmdiYSgzMSw0Miw2OCwuNiknOyBnLmZvbnQ9J2JvbGQgMjJweCBtb25vc3BhY2UnOyBnLnRleHRBbGlnbj0nY2VudGVyJzsgZy50ZXh0QmFzZWxpbmU9J21pZGRsZSc7CiAgICBnLmZpbGxUZXh0KGxhYmVsczJbeV1beF0sIHgqcytzLzIsIHkqcytzLzIpOwogIH0KICBnLnN0cm9rZVN0eWxlPScjRTg4MjFGJztnLmxpbmVXaWR0aD0yOwogIGZvcihsZXQgaT0wO2k8PTI7aSsrKXtnLmJlZ2luUGF0aCgpO2cubW92ZVRvKGkqcywwKTtnLmxpbmVUbyhpKnMsY3YuaGVpZ2h0KTtnLnN0cm9rZSgpO2cuYmVnaW5QYXRoKCk7Zy5tb3ZlVG8oMCxpKnMpO2cubGluZVRvKGN2LndpZHRoLGkqcyk7Zy5zdHJva2UoKTt9Cn0KCi8vIHN0ZXAgMDogbm90aGluZzsgMTogdXBzYW1wbGUgY29weTsgMjogY29udiByZWZpbmU7IHNob3cgcHJvZ3Jlc3NpdmUKbGV0IHJlZmluZWQ9bnVsbDsKZnVuY3Rpb24gY29tcHV0ZVJlZmluZWQoKXsKICAvLyB1cHNhbXBsZSB0byA0eDQKICBjb25zdCB1cD1BcnJheS5mcm9tKHtsZW5ndGg6NH0sKF8seSk9PkFycmF5LmZyb20oe2xlbmd0aDo0fSwoXyx4KT0+U1JDW3k+PjFdW3g+PjFdKSk7CiAgLy8gM3gzIGJveCBibHVyIChzYW1lIHBhZGRpbmcpIHRvIG1pbWljIHRoZSBjb252IHNtb290aGluZwogIGNvbnN0IG91dD1BcnJheS5mcm9tKHtsZW5ndGg6NH0sKCk9PkFycmF5KDQpLmZpbGwoMCkpOwogIGZvcihsZXQgeT0wO3k8NDt5KyspZm9yKGxldCB4PTA7eDw0O3grKyl7bGV0IHM9MCxjPTA7CiAgICBmb3IobGV0IGR5PS0xO2R5PD0xO2R5KyspZm9yKGxldCBkeD0tMTtkeDw9MTtkeCsrKXtjb25zdCBueD14K2R4LG55PXkrZHk7aWYobng8MHx8bnk8MHx8bng+PTR8fG55Pj00KWNvbnRpbnVlO3MrPXVwW255XVtueF07YysrO30KICAgIG91dFt5XVt4XT1zL2M7fQogIHJldHVybiB7dXAsb3V0fTsKfQpmdW5jdGlvbiBkcmF3RHN0KHN0ZXAscHJvZyl7CiAgY29uc3QgY3Y9JCgnZHN0JyksZz1jdi5nZXRDb250ZXh0KCcyZCcpLHM9Y3Yud2lkdGgvNDsKICBnLmNsZWFyUmVjdCgwLDAsY3Yud2lkdGgsY3YuaGVpZ2h0KTsKICBjb25zdCB7dXAsb3V0fT1jb21wdXRlUmVmaW5lZCgpOwogIGZvcihsZXQgeT0wO3k8NDt5KyspZm9yKGxldCB4PTA7eDw0O3grKyl7CiAgICBsZXQgdjsKICAgIGlmKHN0ZXA8MSl7IGcuZmlsbFN0eWxlPScjZjFmNWZjJzsgZy5maWxsUmVjdCh4KnMseSpzLHMscyk7IGNvbnRpbnVlOyB9CiAgICBpZihzdGVwPT09MSl7IHY9dXBbeV1beF07IH0KICAgIGVsc2UgeyB2PXVwW3ldW3hdKigxLXByb2cpK291dFt5XVt4XSpwcm9nOyB9CiAgICBnLmZpbGxTdHlsZT12YWwycmdiKHYpOyBnLmZpbGxSZWN0KHgqcyx5KnMscyxzKTsKICAgIGlmKHN0ZXA9PT0xKXsgZy5maWxsU3R5bGU9J3JnYmEoMzEsNDIsNjgsLjUpJztnLmZvbnQ9J2JvbGQgMTZweCBtb25vc3BhY2UnO2cudGV4dEFsaWduPSdjZW50ZXInO2cudGV4dEJhc2VsaW5lPSdtaWRkbGUnOwogICAgICBnLmZpbGxUZXh0KGxhYmVsczJbeT4+MV1beD4+MV0sIHgqcytzLzIsIHkqcytzLzIpOyB9CiAgfQogIC8vIGdyaWQgKyAyeDIgc291cmNlLWJsb2NrIG91dGxpbmVzCiAgZy5zdHJva2VTdHlsZT0ncmdiYSgxMjAsMTQwLDE4MCwuMjUpJztnLmxpbmVXaWR0aD0xOwogIGZvcihsZXQgaT0wO2k8PTQ7aSsrKXtnLmJlZ2luUGF0aCgpO2cubW92ZVRvKGkqcywwKTtnLmxpbmVUbyhpKnMsY3YuaGVpZ2h0KTtnLnN0cm9rZSgpO2cuYmVnaW5QYXRoKCk7Zy5tb3ZlVG8oMCxpKnMpO2cubGluZVRvKGN2LndpZHRoLGkqcyk7Zy5zdHJva2UoKTt9CiAgZy5zdHJva2VTdHlsZT0nIzBFOUM4Ric7Zy5saW5lV2lkdGg9Mi41OwogIGZvcihsZXQgaT0wO2k8PTQ7aSs9Mil7Zy5iZWdpblBhdGgoKTtnLm1vdmVUbyhpKnMsMCk7Zy5saW5lVG8oaSpzLGN2LmhlaWdodCk7Zy5zdHJva2UoKTtnLmJlZ2luUGF0aCgpO2cubW92ZVRvKDAsaSpzKTtnLmxpbmVUbyhjdi53aWR0aCxpKnMpO2cuc3Ryb2tlKCk7fQp9CgpmdW5jdGlvbiBkcmF3Q2hlY2tlcmJvYXJkcygpewogIC8vIGNoZWNrZXJib2FyZCAoYmFkKQogIGNvbnN0IGNiPSQoJ2NiJyksZz1jYi5nZXRDb250ZXh0KCcyZCcpLG49MTIscz1jYi53aWR0aC9uOwogIGZvcihsZXQgeT0wO3k8bjt5KyspZm9yKGxldCB4PTA7eDxuO3grKyl7CiAgICBjb25zdCBiYXNlPTAuNTsgY29uc3QgY2JrPSgoeCt5KSUyKT8wLjc4OjAuMjI7ICAgICAvLyBzdHJvbmcgYWx0ZXJuYXRpbmcgcGF0dGVybgogICAgZy5maWxsU3R5bGU9dmFsMnJnYihjYmsqMC43K2Jhc2UqMC4zKTsgZy5maWxsUmVjdCh4KnMseSpzLE1hdGguY2VpbChzKSxNYXRoLmNlaWwocykpOwogIH0KICAvLyBzbW9vdGggKGdvb2QpCiAgY29uc3Qgc209JCgnc20nKSxnMj1zbS5nZXRDb250ZXh0KCcyZCcpOwogIGZvcihsZXQgeT0wO3k8bjt5KyspZm9yKGxldCB4PTA7eDxuO3grKyl7CiAgICBjb25zdCBkeD0oeC1uLzIpL24sIGR5PSh5LW4vMikvbjsgY29uc3Qgdj0wLjctTWF0aC5zcXJ0KGR4KmR4K2R5KmR5KSowLjk7CiAgICBnMi5maWxsU3R5bGU9dmFsMnJnYihNYXRoLm1heCgwLE1hdGgubWluKDEsdikpKTsgZzIuZmlsbFJlY3QoeCpzLHkqcyxNYXRoLmNlaWwocyksTWF0aC5jZWlsKHMpKTsKICB9Cn0KCmNvbnN0IHN0ZXBzPVsKeyB0aXRsZTonQmVmb3JlIOKAlCBhIDLDlzIgcGF0Y2ggZGVlcCBpbiB0aGUgZGVjb2RlcicsCiAgbWw6J3N0YXJ0Jywgc3JjdDonSW5wdXQgMsOXMicsIGRzdHQ6J091dHB1dCAoZW1wdHkpJywgc3RlcDowLAogIGV4cGxhaW46YDxwPkRlZXAgaW5zaWRlIHRoZSBkZWNvZGVyIHRoZSBmZWF0dXJlIG1hcHMgYXJlIHRpbnkg4oCUIGhlcmUgYSA8YiBjbGFzcz0ibyI+MsOXMiBwYXRjaDwvYj4gd2l0aCB2YWx1ZXMgQSwgQiwgQywgRC4KICBXZSBuZWVkIHRvIGdyb3cgdGhpcyB0b3dhcmQgdGhlIGZ1bGwgMjjDlzI4IGltYWdlLiA8Y29kZT5kZWNvbnY8L2NvZGU+IHdpbGwgZG8gaXQgaW4gdHdvIG1vdmVzLjwvcD5gLAogIGNvZGU6W1snZGVmIGRlY29udihuaSwgbmYsIGtzPTMsIGFjdD1UcnVlKTonXSwKICAgICAgICBbJyAgICBsYXllcnMgPSBbJ10sCiAgICAgICAgWycgICAgICAgIG5uLlVwc2FtcGxpbmdOZWFyZXN0MmQoc2NhbGVfZmFjdG9yPTIpLCAgIyBzdGVwIDEnLCdjbSddLAogICAgICAgIFsnICAgICAgICBubi5Db252MmQobmksIG5mLCBzdHJpZGU9MSwnLCdjbSddLAogICAgICAgIFsnICAgICAgICAgICAgICAgICAga2VybmVsX3NpemU9a3MsIHBhZGRpbmc9a3MvLzIpICMgc3RlcCAyJywnY20nXSwKICAgICAgICBbJyAgICBdJ11dCn0sCnsgdGl0bGU6J1N0ZXAgMSDigJQgVXBzYW1wbGluZ05lYXJlc3QyZDogY29weSBlYWNoIHBpeGVsIGludG8gYSAyw5cyIGJsb2NrJywKICBtbDondXBzYW1wbGUnLCBzcmN0OidJbnB1dCAyw5cyJywgZHN0dDonVXBzYW1wbGVkIDTDlzQgKGJsb2NreSknLCBzdGVwOjEsCiAgZXhwbGFpbjpgPHA+PGNvZGU+bm4uVXBzYW1wbGluZ05lYXJlc3QyZChzY2FsZV9mYWN0b3I9Mik8L2NvZGU+IHNpbXBseSA8YiBjbGFzcz0idCI+cmVwZWF0czwvYj4gZXZlcnkgdmFsdWUgaW50byBhIDLDlzIgYmxvY2suCiAgQSBiZWNvbWVzIGEgMsOXMiBvZiBBLCBCIGEgMsOXMiBvZiBCLCBhbmQgc28gb24uIFNpemUgZG91YmxlczogPGI+MsOXMiDihpIgNMOXNDwvYj4uPC9wPgogIDxwPkl0IGlzIGZhc3QgYW5kIGhhcyA8aT5ubzwvaT4gbGVhcm5hYmxlIHdlaWdodHMg4oCUIGJ1dCB0aGUgcmVzdWx0IGlzIDxiPmJsb2NreTwvYj4gYmVjYXVzZSB3ZSBqdXN0IGNvcGllZC48L3A+YCwKICBjb2RlOltbJ25uLlVwc2FtcGxpbmdOZWFyZXN0MmQoc2NhbGVfZmFjdG9yPTIpJywnaGwnXSwKICAgICAgICBbJyddLAogICAgICAgIFsnIyAyw5cyICDihpIgIDTDlzQgICAoZWFjaCB2YWx1ZSBjb3BpZWQgNMOXKScsJ2NtJ10sCiAgICAgICAgWycjICBBIEIgICAgICAgIEEgQSBCIEInLCdjbSddLAogICAgICAgIFsnIyAgQyBEICAg4oaSICAgIEEgQSBCIEInLCdjbSddLAogICAgICAgIFsnIyAgICAgICAgICAgICBDIEMgRCBEJywnY20nXSwKICAgICAgICBbJyMgICAgICAgICAgICAgQyBDIEQgRCcsJ2NtJ11dCn0sCnsgdGl0bGU6J1N0ZXAgMiDigJQgQ29udjJkIChzdHJpZGUgMSwgc2FtZSBwYWRkaW5nKTogc21vb3RoICYgbGVhcm4nLAogIG1sOidyZWZpbmUnLCBzcmN0OidJbnB1dCAyw5cyJywgZHN0dDonUmVmaW5lZCA0w5c0IChzbW9vdGgpJywgc3RlcDoyLAogIGV4cGxhaW46YDxwPk5vdyBhIDxiIGNsYXNzPSJwIj5zdHJpZGUtMSBjb252b2x1dGlvbjwvYj4gd2l0aCA8Y29kZT5wYWRkaW5nID0ga3MvLzI8L2NvZGU+IHNsaWRlcyBvdmVyIHRoZSBibG9ja3kgZ3JpZC4KICBCZWNhdXNlIHN0cmlkZSBpcyAxIGFuZCBwYWRkaW5nIG1hdGNoZXMgdGhlIGtlcm5lbCwgdGhlIDxiPnNpemUgc3RheXMgNMOXNDwvYj4g4oCUIGJ1dCB0aGUgaGFyc2ggYmxvY2sgZWRnZXMgZ2V0IGJsZW5kZWQsCiAgYW5kIHRoZSBmaWx0ZXIgbGVhcm5zIHJlYWwgZmVhdHVyZXMgb24gdGhlIGJpZ2dlciBncmlkLjwvcD4KICA8cD5XYXRjaCB0aGUgYmxvY2tzIG1lbHQgaW50byBzbW9vdGggZ3JhZGllbnRzIGFzIHRoZSBjb252IGFwcGxpZXMuPC9wPmAsCiAgY29kZTpbWydubi5Db252MmQobmksIG5mLCBzdHJpZGU9MSwnLCdobCddLAogICAgICAgIFsnICAgICAgICAgIGtlcm5lbF9zaXplPTMsIHBhZGRpbmc9MSknLCdobCddLAogICAgICAgIFsnJ10sCiAgICAgICAgWycjIHN0cmlkZT0xICsgcGFkZGluZz1rcy8vMiAg4oaSICBzaXplIHByZXNlcnZlZCcsJ2NtJ10sCiAgICAgICAgWycjIG91dHB1dCA9IOKMiig0ICsgMsK3MSDiiJIgMykvMeKMiyArIDEgPSA0JywnY20nXV0KfQpdOwoKbGV0IGN1cj0wLCBwbGF5aW5nPWZhbHNlLCB0aW1lcj1udWxsLCBhbmltPW51bGw7CmNvbnN0IGRvdHM9JCgnZG90cycpOyBzdGVwcy5mb3JFYWNoKCgpPT57Y29uc3QgaT1kb2N1bWVudC5jcmVhdGVFbGVtZW50KCdpJyk7ZG90cy5hcHBlbmRDaGlsZChpKTt9KTsKCmZ1bmN0aW9uIHJlbmRlcihhbmltYXRlQ29udil7CiAgY29uc3Qgcz1zdGVwc1tjdXJdOwogIGRyYXdTcmMoKTsgZHJhd0NoZWNrZXJib2FyZHMoKTsKICAkKCdzcmN0JykudGV4dENvbnRlbnQ9cy5zcmN0OyAkKCdkc3R0JykudGV4dENvbnRlbnQ9cy5kc3R0OyAkKCdtbCcpLnRleHRDb250ZW50PXMubWw7CiAgaWYocy5zdGVwPT09MiAmJiBhbmltYXRlQ29udil7CiAgICBsZXQgcD0wOyBpZihhbmltKWNhbmNlbEFuaW1hdGlvbkZyYW1lKGFuaW0pOwogICAgY29uc3QgcnVuPSgpPT57IHArPTAuMDQqc3BlZWRzWyskKCdzcGQnKS52YWx1ZS0xXTsgaWYocD4xKXA9MTsgZHJhd0RzdCgyLHApOyBpZihwPDEpYW5pbT1yZXF1ZXN0QW5pbWF0aW9uRnJhbWUocnVuKTsgfTsKICAgIHJ1bigpOwogIH0gZWxzZSBkcmF3RHN0KHMuc3RlcCwgMSk7CiAgJCgnc3RlcG51bScpLnRleHRDb250ZW50PWBTdGVwICR7Y3VyfSAvICR7c3RlcHMubGVuZ3RoLTF9YDsKICBbLi4uZG90cy5jaGlsZHJlbl0uZm9yRWFjaCgoZCxpKT0+ZC5jbGFzc0xpc3QudG9nZ2xlKCdvbicsaT09PWN1cikpOwogICQoJ3RpdGxlJykudGV4dENvbnRlbnQ9cy50aXRsZTsgJCgnZXhwbGFpbicpLmlubmVySFRNTD1zLmV4cGxhaW47CiAgY29uc3QgY29kZUVsPSQoJ2NvZGUnKTsgY29kZUVsLmlubmVySFRNTD0nJzsKICBmb3IoY29uc3QgbGluZSBvZiBzLmNvZGUpe2NvbnN0IGQ9ZG9jdW1lbnQuY3JlYXRlRWxlbWVudCgnZGl2Jyk7ZC5jbGFzc05hbWU9J2NsJysobGluZVsxXT8nICcrbGluZVsxXTonJyk7ZC50ZXh0Q29udGVudD1saW5lWzBdPT09Jyc/J1x1MDBBMCc6bGluZVswXTtjb2RlRWwuYXBwZW5kQ2hpbGQoZCk7fQogICQoJ3ByZXYnKS5kaXNhYmxlZD1jdXI9PT0wOyQoJ3ByZXYnKS5zdHlsZS5vcGFjaXR5PWN1cj09PTA/LjQ1OjE7CiAgJCgnbmV4dCcpLmRpc2FibGVkPWN1cj09PXN0ZXBzLmxlbmd0aC0xOyQoJ25leHQnKS5zdHlsZS5vcGFjaXR5PWN1cj09PXN0ZXBzLmxlbmd0aC0xPy41OjE7CiAgcG9zdEgoKTsKfQpmdW5jdGlvbiBnbyhpLGEpeyBjdXI9TWF0aC5tYXgoMCxNYXRoLm1pbihzdGVwcy5sZW5ndGgtMSxpKSk7IHJlbmRlcihhKTsgfQpmdW5jdGlvbiBzdG9wUGxheSgpeyBwbGF5aW5nPWZhbHNlOyBpZih0aW1lciljbGVhclRpbWVvdXQodGltZXIpOyAkKCdwbGF5JykudGV4dENvbnRlbnQ9J1BsYXkg4pa2JzsgfQpmdW5jdGlvbiB0aWNrKCl7IGlmKCFwbGF5aW5nKXJldHVybjsgaWYoY3VyPj1zdGVwcy5sZW5ndGgtMSl7c3RvcFBsYXkoKTtyZXR1cm47fSBnbyhjdXIrMSx0cnVlKTsgdGltZXI9c2V0VGltZW91dCh0aWNrLDE4MDAvc3BlZWRzWyskKCdzcGQnKS52YWx1ZS0xXSk7IH0KJCgncGxheScpLmFkZEV2ZW50TGlzdGVuZXIoJ2NsaWNrJywoKT0+eyBpZihwbGF5aW5nKXtzdG9wUGxheSgpO3JldHVybjt9IGlmKGN1cj49c3RlcHMubGVuZ3RoLTEpZ28oMCxmYWxzZSk7IHBsYXlpbmc9dHJ1ZTskKCdwbGF5JykudGV4dENvbnRlbnQ9J1BhdXNlIOKPuCc7IHRpbWVyPXNldFRpbWVvdXQodGljaywxNDAwL3NwZWVkc1srJCgnc3BkJykudmFsdWUtMV0pOyB9KTsKJCgnbmV4dCcpLmFkZEV2ZW50TGlzdGVuZXIoJ2NsaWNrJywoKT0+e3N0b3BQbGF5KCk7Z28oY3VyKzEsdHJ1ZSk7fSk7CiQoJ3ByZXYnKS5hZGRFdmVudExpc3RlbmVyKCdjbGljaycsKCk9PntzdG9wUGxheSgpO2dvKGN1ci0xLGZhbHNlKTt9KTsKJCgncmVzZXQnKS5hZGRFdmVudExpc3RlbmVyKCdjbGljaycsKCk9PntzdG9wUGxheSgpO2dvKDAsZmFsc2UpO30pOwokKCdzcGQnKS5hZGRFdmVudExpc3RlbmVyKCdpbnB1dCcsKCk9PnskKCdzcGR2JykudGV4dENvbnRlbnQ9c3BlZWRzWyskKCdzcGQnKS52YWx1ZS0xXS50b0ZpeGVkKDEpKyfDlyc7fSk7CmRvY3VtZW50LmFkZEV2ZW50TGlzdGVuZXIoJ2tleWRvd24nLGU9PntpZihlLmtleT09PSdBcnJvd1JpZ2h0Jyl7c3RvcFBsYXkoKTtnbyhjdXIrMSx0cnVlKTt9aWYoZS5rZXk9PT0nQXJyb3dMZWZ0Jyl7c3RvcFBsYXkoKTtnbyhjdXItMSxmYWxzZSk7fX0pOwoKY29uc3QgZ3JpZD0kKCdncmlkJyksIHdyYXA9ZG9jdW1lbnQucXVlcnlTZWxlY3RvcignLndyYXAnKTsKZnVuY3Rpb24gcmVsYXlvdXQoKXsgZ3JpZC5jbGFzc0xpc3QudG9nZ2xlKCdpcy1uYXJyb3cnLCB3aW5kb3cuaW5uZXJXaWR0aDw4MjApOyB9CmZ1bmN0aW9uIHBvc3RIKCl7IGNvbnN0IGg9d3JhcD9NYXRoLmNlaWwod3JhcC5nZXRCb3VuZGluZ0NsaWVudFJlY3QoKS5oZWlnaHQpKzM0OmRvY3VtZW50LmJvZHkub2Zmc2V0SGVpZ2h0OwogIGlmKHdpbmRvdy5wYXJlbnQhPT13aW5kb3cpIHdpbmRvdy5wYXJlbnQucG9zdE1lc3NhZ2Uoe3R5cGU6J2FlLWZyYW1lLWhlaWdodCcsaGVpZ2h0Omh9LCcqJyk7IH0KZnVuY3Rpb24gdXBkYXRlKCl7IHJlbGF5b3V0KCk7IHBvc3RIKCk7IH0Kd2luZG93LmFkZEV2ZW50TGlzdGVuZXIoJ2xvYWQnLHVwZGF0ZSk7IHdpbmRvdy5hZGRFdmVudExpc3RlbmVyKCdyZXNpemUnLHVwZGF0ZSk7CmlmKHdpbmRvdy5SZXNpemVPYnNlcnZlcikgbmV3IFJlc2l6ZU9ic2VydmVyKHBvc3RIKS5vYnNlcnZlKHdyYXB8fGRvY3VtZW50LmJvZHkpOwpyZWxheW91dCgpOyByZW5kZXIoZmFsc2UpOwpzZXRUaW1lb3V0KHVwZGF0ZSwyMDApOyBzZXRUaW1lb3V0KHVwZGF0ZSw3MDApOwp9KSgpOwo8L3NjcmlwdD4KPC9ib2R5Pgo8L2h0bWw+Cg=="

HTML('''
<iframe id="deconv-frame"
        src="data:text/html;base64,''' + _html_b64 + '''"
        style="width:100%; height:840px; border:1px solid #DDE5F2;
               border-radius:16px; box-shadow:0 8px 24px rgba(108,92,231,.12);
               display:block;"
        loading="lazy" title="Deconv upsample lab"></iframe>
<script>
(function(){
  function onMsg(e){
    if (e.data && e.data.type === "ae-frame-height") {
      var f = document.getElementById("deconv-frame");
      if (f) f.style.height = (e.data.height) + "px";
    }
  }
  window.addEventListener("message", onMsg);
})();
</script>
''')


**Why use Upsampling + Conv instead of ConvTranspose2d?**

| Method | Pros | Cons |
|--------|------|------|
| **ConvTranspose2d** | Learnable upsampling | Can cause checkerboard artifacts |
| **Upsample + Conv** | No artifacts, more stable | Slightly more computation |

The artifacts come from uneven overlap when ConvTranspose2d "deconvolves". Using explicit upsampling avoids this.

---

## Training Functions for Autoencoders

Autoencoders have a different training objective than classifiers:

| Classifier | Autoencoder |
|------------|-------------|
| Input: image | Input: image |
| Target: label (0-9) | Target: **same image** |
| Loss: cross-entropy | Loss: **MSE** (mean squared error) |
| Goal: predict class | Goal: **reconstruct input** |

In [ ]:
# =====================================================
# EVALUATION FUNCTION FOR AUTOENCODERS
# =====================================================

def eval_ae(model, loss_func, valid_dl, epoch=0):
    """
    Evaluate an autoencoder on the validation set.
    
    Key difference from classifier evaluation:
    - We compare the model's output to the INPUT (xb), not the label
    - The _ in 'for xb, _ in valid_dl' ignores the labels
    
    Arguments:
        model     - The autoencoder to evaluate
        loss_func - Reconstruction loss function (MSE)
        valid_dl  - Validation data loader
        epoch     - Current epoch number (for display)
    """
    model.eval()  # Set to evaluation mode (disables dropout, etc.)
    
    with torch.no_grad():  # Don't compute gradients (faster)
        tot_loss, count = 0., 0
        
        for xb, _ in valid_dl:  # _ ignores labels - we don't use them!
            # Forward pass: try to reconstruct the input
            pred = model(xb)
            
            n = len(xb)
            count += n
            
            # Loss: how different is pred from the ORIGINAL INPUT xb?
            # Note: we compare pred to xb, not to labels!
            tot_loss += loss_func(pred, xb).item() * n
    
    print(f"Epoch {epoch}: loss = {tot_loss/count:.4f}")

# Understanding the Autoencoder Evaluation Function

## Overview

```python
def eval_ae(model, loss_func, valid_dl, epoch=0):
```

This function measures **how well the autoencoder can reconstruct images**. Unlike a classifier that checks "did you guess the right label?", an autoencoder asks "does your output look like the input?"

---

## The Key Insight: Autoencoders vs Classifiers

| Aspect | Classifier | Autoencoder |
|--------|------------|-------------|
| **Input** | Image | Image |
| **Output** | Class label (0-9) | Reconstructed image |
| **Target** | The label `yb` | The input itself `xb` |
| **Question** | "What is this?" | "Can you copy this?" |

```
Classifier:
  Input: 👟 (image of sneaker)
  Output: 7
  Target: 7 (label)
  Loss: Was 7 correct? ✓

Autoencoder:
  Input: 👟 (image of sneaker)
  Output: 👟 (reconstructed sneaker)
  Target: 👟 (the original input!)
  Loss: How similar are they?
```

---

## Line-by-Line Explanation

### 1. Set Evaluation Mode

```python
model.eval()
```

| What It Does | Why It Matters |
|--------------|----------------|
| Switches model to evaluation mode | Disables dropout (if any) |
| | Uses fixed statistics for batch normalization |
| | Gives consistent, reproducible predictions |

**Analogy**: It's like telling a student "this is the real test, not practice anymore."

---

### 2. Disable Gradient Computation

```python
with torch.no_grad():
```

| What It Does | Why It Matters |
|--------------|----------------|
| Stops tracking operations for gradients | Saves memory |
| | Speeds up computation |
| | We don't need gradients — we're not learning |

**Analogy**: You don't need scratch paper when you're just checking answers.

---

### 3. Initialize Counters

```python
tot_loss, count = 0., 0
```

- `tot_loss`: Accumulates the total reconstruction error
- `count`: Counts total number of images evaluated

---

### 4. Loop Through Validation Data

```python
for xb, _ in valid_dl:
```

| Part | Meaning |
|------|---------|
| `xb` | Batch of input images |
| `_` | Labels (we ignore them!) |

**Why ignore labels?** Autoencoders don't care about labels. They just try to reconstruct whatever image you give them.

```
Classifier needs labels:     Autoencoder ignores labels:
  Image: 👟                    Image: 👟
  Label: 7  ← used!            Label: 7  ← ignored!
```

---

### 5. Forward Pass (Reconstruction)

```python
pred = model(xb)
```

The autoencoder tries to reconstruct the input:

```
Input (xb):                    Output (pred):
┌─────────────┐               ┌─────────────┐
│             │               │             │
│   Original  │  ──model──→   │ Reconstruct │
│    Image    │               │    -ion     │
│             │               │             │
└─────────────┘               └─────────────┘
     28×28                         28×28
```

---

### 6. Calculate Reconstruction Loss

```python
tot_loss += loss_func(pred, xb).item() * n
```

**This is the crucial difference from classifiers!**

| Classifier | Autoencoder |
|------------|-------------|
| `loss_func(pred, yb)` | `loss_func(pred, xb)` |
| Compare to **label** | Compare to **input** |

#### What MSE Loss Measures

```python
loss = MSE(pred, xb)  # Mean Squared Error
```

$$\text{MSE} = \frac{1}{n} \sum_{i=1}^{n} (\text{pred}_i - \text{input}_i)^2$$

For each pixel:
- Calculate difference between predicted and original
- Square it (makes all differences positive)
- Average over all pixels

```
Original pixel: 0.8     Predicted pixel: 0.7
Difference: 0.1
Squared: 0.01

Lower MSE = Better reconstruction
```

---

### 7. Accumulate Statistics

```python
n = len(xb)
count += n
tot_loss += loss_func(pred, xb).item() * n
```

| Variable | Purpose |
|----------|---------|
| `n` | Number of images in this batch |
| `count` | Running total of all images |
| `.item()` | Convert tensor to Python number |
| `* n` | Weight by batch size for proper averaging |

**Why multiply by `n`?**

```
Batch 1: 256 images, loss = 0.05  → contributes 256 × 0.05 = 12.8
Batch 2: 256 images, loss = 0.04  → contributes 256 × 0.04 = 10.24
Batch 3: 100 images, loss = 0.06  → contributes 100 × 0.06 = 6.0
                                     ─────────────────────────────
Total: 612 images                    Total loss: 29.04

Average loss = 29.04 / 612 = 0.0475
```

This handles the case where the last batch might be smaller.

---

### 8. Print Results

```python
print(f"Epoch {epoch}: loss = {tot_loss/count:.4f}")
```

Prints the average reconstruction loss:

```
Epoch 0: loss = 0.0892
Epoch 1: loss = 0.0654
Epoch 2: loss = 0.0521
...
```

**Lower loss = Better reconstruction!**

---

## Visual Summary

```
                    AUTOENCODER EVALUATION
                    
    ┌─────────────────────────────────────────────┐
    │              Validation Data                │
    │  ┌─────┐ ┌─────┐ ┌─────┐ ┌─────┐ ┌─────┐   │
    │  │ 👟  │ │ 👔  │ │ 👗  │ │ 👜  │ │ 🥾  │   │
    │  └─────┘ └─────┘ └─────┘ └─────┘ └─────┘   │
    └────────────────────┬────────────────────────┘
                         │
                         ▼
              ┌─────────────────────┐
              │    AUTOENCODER      │
              │  ┌───────────────┐  │
              │  │   Encoder     │  │
              │  │  (compress)   │  │
              │  └───────┬───────┘  │
              │          │          │
              │  ┌───────▼───────┐  │
              │  │   Bottleneck  │  │
              │  └───────┬───────┘  │
              │          │          │
              │  ┌───────▼───────┐  │
              │  │   Decoder     │  │
              │  │  (reconstruct)│  │
              │  └───────────────┘  │
              └──────────┬──────────┘
                         │
                         ▼
    ┌─────────────────────────────────────────────┐
    │            Reconstructions                  │
    │  ┌─────┐ ┌─────┐ ┌─────┐ ┌─────┐ ┌─────┐   │
    │  │ 👟  │ │ 👔  │ │ 👗  │ │ 👜  │ │ 🥾  │   │
    │  │~blurry~│    │      │      │      │      │
    │  └─────┘ └─────┘ └─────┘ └─────┘ └─────┘   │
    └────────────────────┬────────────────────────┘
                         │
                         ▼
              ┌─────────────────────┐
              │   COMPARE (MSE)     │
              │                     │
              │  Original vs Recon  │
              │  How different?     │
              └──────────┬──────────┘
                         │
                         ▼
              ┌─────────────────────┐
              │  loss = 0.0521      │
              │  (lower = better)   │
              └─────────────────────┘
```

---

## Comparison: Classifier vs Autoencoder Evaluation

```python
# CLASSIFIER evaluation
for xb, yb in valid_dl:
    pred = model(xb)           # Predict class scores
    loss = loss_func(pred, yb) # Compare to LABELS
    acc = accuracy(pred, yb)   # Did we guess right?

# AUTOENCODER evaluation  
for xb, _ in valid_dl:         # Ignore labels!
    pred = model(xb)           # Reconstruct image
    loss = loss_func(pred, xb) # Compare to INPUT
    # No accuracy - just reconstruction quality
```

---

## Key Takeaways

1. **Autoencoders compare output to input** — not to labels

2. **Labels are ignored** — the `_` in `for xb, _ in valid_dl` discards them

3. **MSE measures reconstruction quality** — lower = better

4. **No accuracy metric** — we only measure how close the reconstruction is

5. **Same eval principles apply** — `model.eval()`, `torch.no_grad()`, accumulate statistics

In [ ]:
# =====================================================
# TRAINING FUNCTION FOR AUTOENCODERS
# =====================================================

def fit_ae(epochs, model, loss_func, opt, train_dl, valid_dl):
    """
    Train an autoencoder.
    
    Key difference from classifier training:
    - Target is the input image itself, not a label
    - We use loss_func(model(xb), xb) instead of loss_func(model(xb), yb)
    
    Arguments:
        epochs    - Number of training epochs
        model     - The autoencoder to train
        loss_func - Reconstruction loss (MSE)
        opt       - Optimizer
        train_dl  - Training data loader
        valid_dl  - Validation data loader
    """
    for epoch in range(epochs):
        # Training phase
        model.train()
        
        for xb, _ in train_dl:  # Ignore labels with _
            # Forward pass: reconstruct the input
            pred = model(xb)
            
            # Loss: compare prediction to ORIGINAL INPUT
            # This is the key autoencoder insight:
            #   We want pred to match xb as closely as possible
            loss = loss_func(pred, xb) # We are trying to recreate the original image
            
            # Backward pass: compute gradients
            loss.backward()
            
            # Update weights
            opt.step()
            opt.zero_grad()
        
        # Evaluation phase
        eval_ae(model, loss_func, valid_dl, epoch)

# Why Sigmoid Output is Correct Here — No Range Mismatch

## Concern

> The model outputs Sigmoid values in `[0, 1]`, but the original images might have a larger range — wouldn't that cause a wrong comparison in `loss_func(model(xb), xb)`?

**Great instinct — but there's no mismatch.** The input images are **already in `[0, 1]` range**.

---

## Where the Scaling Happens

Earlier in the notebook, the data is transformed using `TF.to_tensor`:

```python
def transformi(b):
    b[x] = [TF.to_tensor(o) for o in b[x]]
```

`TF.to_tensor` does three things:

1. Converts the PIL image to a PyTorch tensor
2. Rearranges dimensions from `(H, W, C)` to `(C, H, W)`
3. **Scales pixel values from `[0, 255]` → `[0.0, 1.0]`** by dividing by 255

So by the time data reaches the model, `xb` already lives in `[0, 1]`.

---

## Why Sigmoid Is the Right Choice

The MSE loss computes the **pixel-wise squared difference** between the reconstruction and the original:

```python
loss = F.mse_loss(model(xb), xb)
#                 ↑              ↑
#          Sigmoid output    Original input
#           range [0, 1]     range [0, 1]  ← both match!
```

| Component | Value Range |
|---|---|
| Input `xb` (after `to_tensor`) | `[0.0, 1.0]` |
| Output `model(xb)` (after Sigmoid) | `[0.0, 1.0]` |
| MSE Loss | Compares same-range values ✓ |

If the model output were **unbounded** (no Sigmoid), it could predict values like `-0.3` or `2.5`, which are impossible pixel values. Sigmoid **constrains** the output to the valid range, making the reconstruction physically meaningful.

---

## What Would Go Wrong Without Sigmoid?

Without Sigmoid, the last layer would output raw values from the `deconv` layer, which could be **any real number**. Two problems:

1. **Invalid reconstructions** — pixel values outside `[0, 1]` don't correspond to real images
2. **Harder optimization** — the model would need to learn the correct output range on top of learning the reconstruction, making training slower and less stable

---

## Summary

```
Raw image:    [0, 255]  (integers)
     │
     ▼  TF.to_tensor (÷ 255)
Input xb:     [0.0, 1.0]  (floats)
     │
     ▼  Autoencoder
Output:       [0.0, 1.0]  (floats, thanks to Sigmoid)
     │
     ▼  F.mse_loss(output, input)
Loss:         Both in [0, 1] — fair comparison ✓
```

The Sigmoid isn't causing a mismatch — it's **ensuring** a match. It guarantees the model's output lives in exactly the same range as its input.

# Understanding the Autoencoder Training Function

## Overview

```python
def fit_ae(epochs, model, loss_func, opt, train_dl, valid_dl):
```

This function **teaches the autoencoder to reconstruct images**. The key insight: instead of learning to predict labels, the autoencoder learns to **copy its input through a bottleneck**.

---

## The Core Difference: Classifier vs Autoencoder Training

| Aspect | Classifier | Autoencoder |
|--------|------------|-------------|
| **Goal** | Predict the correct label | Reconstruct the input |
| **Target** | Label `yb` | Input `xb` |
| **Loss** | `loss_func(pred, yb)` | `loss_func(pred, xb)` |
| **Uses labels?** | Yes | No |

```
Classifier Training:
  Input: 👟 → Model → Output: [0.1, 0.0, ..., 0.9, 0.0]
  Target: 7 (sneaker)
  Question: "Did you predict class 7?"

Autoencoder Training:
  Input: 👟 → Model → Output: 👟 (reconstructed)
  Target: 👟 (the same input!)
  Question: "Does your output look like the input?"
```

---

## The Training Loop Structure

```
For each epoch:
    │
    ├── TRAINING PHASE
    │   For each batch:
    │     1. Forward pass (reconstruct)
    │     2. Calculate loss (compare to input)
    │     3. Backward pass (compute gradients)
    │     4. Update weights
    │
    └── EVALUATION PHASE
        Check reconstruction quality on validation data
```

---

## Line-by-Line Explanation

### 1. Loop Through Epochs

```python
for epoch in range(epochs):
```

One epoch = one complete pass through all training data.

```
Epoch 0: See all 60,000 images once
Epoch 1: See all 60,000 images again
Epoch 2: See all 60,000 images again
...
```

---

### 2. Enable Training Mode

```python
model.train()
```

| What It Does | Why It Matters |
|--------------|----------------|
| Activates training behavior | Enables dropout (if present) |
| | Uses batch statistics for normalization |
| | Model knows "I'm learning now" |

---

### 3. Loop Through Training Batches

```python
for xb, _ in train_dl:
```

| Part | Meaning |
|------|---------|
| `xb` | Batch of images (e.g., 256 images) |
| `_` | Labels — **we throw them away!** |

**Why ignore labels?**

The autoencoder doesn't need to know "this is a sneaker" or "this is a dress." It just needs to learn: "whatever comes in, make the same thing come out."

```
Classifier uses both:          Autoencoder uses only images:
  xb (images) ✓                  xb (images) ✓
  yb (labels) ✓                  yb (labels) ✗ (ignored)
```

---

### 4. Forward Pass (Reconstruction)

```python
pred = model(xb)
```

The image goes through the autoencoder:

```
Input Image          Encoder           Bottleneck          Decoder          Output
   (xb)                                                                      (pred)
┌─────────┐       ┌─────────┐       ┌─────────┐       ┌─────────┐       ┌─────────┐
│  28×28  │ ───→  │ Compress│ ───→  │  8×8×4  │ ───→  │ Expand  │ ───→  │  28×28  │
│ Original│       │         │       │ (small!)│       │         │       │  Recon  │
└─────────┘       └─────────┘       └─────────┘       └─────────┘       └─────────┘
  784 values                          256 values                          784 values
```

The bottleneck forces the network to learn **what's important** — it can't memorize everything!

---

### 5. Calculate Loss (The Key Line!)

```python
loss = loss_func(pred, xb)
```

**This is where autoencoders differ from classifiers:**

```python
# Classifier:
loss = loss_func(pred, yb)  # Compare to LABEL

# Autoencoder:
loss = loss_func(pred, xb)  # Compare to INPUT
```

#### What MSE Loss Calculates

```
Original (xb):        Prediction (pred):      Difference:
┌───┬───┬───┐        ┌───┬───┬───┐          ┌───┬───┬───┐
│0.9│0.8│0.7│        │0.8│0.7│0.6│          │0.1│0.1│0.1│
├───┼───┼───┤   vs   ├───┼───┼───┤    →     ├───┼───┼───┤
│0.6│0.5│0.4│        │0.5│0.5│0.4│          │0.1│0.0│0.0│
└───┴───┴───┘        └───┴───┴───┘          └───┴───┴───┘

MSE = average of (differences²) = average of [0.01, 0.01, 0.01, 0.01, 0, 0]
    = 0.0067
```

**Lower MSE = reconstruction looks more like original!**

---

### 6. Backward Pass

```python
loss.backward()
```

PyTorch calculates gradients: **"How should each weight change to reduce the loss?"**

```
loss = 0.0067
       │
       ▼ backward()
┌─────────────────────────────────────┐
│ Gradients computed for every weight │
│                                     │
│ Decoder weights: ∂loss/∂w_decoder   │
│ Encoder weights: ∂loss/∂w_encoder   │
└─────────────────────────────────────┘
```

---

### 7. Update Weights

```python
opt.step()
opt.zero_grad()
```

| Line | What It Does |
|------|--------------|
| `opt.step()` | Adjust all weights using gradients |
| `opt.zero_grad()` | Reset gradients to zero for next batch |

```
Before opt.step():              After opt.step():
weight = 0.5                    weight = 0.5 - lr × gradient
gradient = 0.02                        = 0.5 - 0.1 × 0.02
learning_rate = 0.1                    = 0.498
```

**Why zero gradients?**

Gradients accumulate by default. If we don't reset them, the next batch's gradients would add to the old ones — wrong!

---

### 8. Evaluation Phase

```python
eval_ae(model, loss_func, valid_dl, epoch)
```

After training on all batches, check how well we're doing on data the model hasn't trained on:

```
Training data: Model learns from this
Validation data: Model is tested on this (no learning!)
```

This tells us if we're actually getting better or just memorizing.

---

## Complete Visual Flow

```
                         EPOCH 0
                            │
         ┌──────────────────┴──────────────────┐
         ▼                                     │
┌─────────────────────────────────────┐        │
│         TRAINING PHASE              │        │
│                                     │        │
│  Batch 1: 256 images                │        │
│  ┌─────────────────────────────┐    │        │
│  │ 1. pred = model(xb)         │    │        │
│  │ 2. loss = MSE(pred, xb)     │◄───┼── Compare to INPUT, not label!
│  │ 3. loss.backward()          │    │        │
│  │ 4. opt.step()               │    │        │
│  │ 5. opt.zero_grad()          │    │        │
│  └─────────────────────────────┘    │        │
│                                     │        │
│  Batch 2: 256 images                │        │
│  └─────────────────────────────┘    │        │
│              ...                    │        │
│  Batch N: (remaining images)        │        │
│  └─────────────────────────────┘    │        │
│                                     │        │
└─────────────────┬───────────────────┘        │
                  │                            │
                  ▼                            │
┌─────────────────────────────────────┐        │
│        EVALUATION PHASE             │        │
│                                     │        │
│  eval_ae(model, loss_func, ...)     │        │
│                                     │        │
│  Output: "Epoch 0: loss = 0.0892"   │        │
└─────────────────┬───────────────────┘        │
                  │                            │
                  ▼                            │
              EPOCH 1 ─────────────────────────┘
                  │
                 ...
```

---

## What the Autoencoder Learns

Through this training process, the autoencoder learns:

### Encoder Learns:
- What features are **most important**
- How to **compress** 784 pixels into 256 values
- Which details can be **discarded**

### Decoder Learns:
- How to **reconstruct** from compressed data
- How to **fill in** missing details
- What a "typical" image looks like

```
Training Progress:

Epoch 0:  Input: 👟  →  Output: 🌫️ (blurry blob)     Loss: 0.089
Epoch 5:  Input: 👟  →  Output: 👟 (recognizable)    Loss: 0.045
Epoch 10: Input: 👟  →  Output: 👟 (pretty good!)    Loss: 0.032
```

---

## Side-by-Side: Classifier vs Autoencoder Training

```python
# ==================== CLASSIFIER ====================
def fit(epochs, model, loss_func, opt, train_dl, valid_dl):
    for epoch in range(epochs):
        model.train()
        for xb, yb in train_dl:        # ← Uses labels!
            pred = model(xb)
            loss = loss_func(pred, yb)  # ← Compare to LABEL
            loss.backward()
            opt.step()
            opt.zero_grad()
        # evaluate with accuracy...

# ==================== AUTOENCODER ====================
def fit_ae(epochs, model, loss_func, opt, train_dl, valid_dl):
    for epoch in range(epochs):
        model.train()
        for xb, _ in train_dl:         # ← Ignores labels!
            pred = model(xb)
            loss = loss_func(pred, xb)  # ← Compare to INPUT
            loss.backward()
            opt.step()
            opt.zero_grad()
        eval_ae(model, loss_func, valid_dl, epoch)
```

**The only differences:**
1. `yb` → `_` (ignore labels)
2. `loss_func(pred, yb)` → `loss_func(pred, xb)` (compare to input)

---

## Key Takeaways

1. **Labels are ignored** — Autoencoders are **unsupervised** (no labels needed)

2. **Target is the input** — `loss_func(pred, xb)` compares output to input

3. **Bottleneck forces learning** — The network must figure out what's important to compress

4. **Same training mechanics** — Forward, backward, update still work the same way

5. **Lower loss = better reconstruction** — The goal is to make output match input

---

## Building the Autoencoder Architecture

Now let's design the actual autoencoder. We need to handle the 28×28 input carefully:

**Problem:** 28 doesn't divide evenly by 2 multiple times.
- 28 → 14 → 7 → 3.5 (not integer!)

**Solution:** Pad to 32×32 first, then downsample/upsample, then crop back.
- 32 → 16 → 8 → 16 → 32 (all integers!)

In [ ]:
# =====================================================
# AUTOENCODER ARCHITECTURE
# =====================================================

ae = nn.Sequential(
    # ===== PREPROCESSING =====
    # Pad 28x28 to 32x32 by adding 2 pixels on each side
    # ZeroPad2d(2) adds 2 zeros to left, right, top, bottom
    nn.ZeroPad2d(2),        # 28x28 -> 32x32
    
    # ===== ENCODER (Compress) =====
    # conv(1, 2): 1 input channel -> 2 output channels, stride=2
    conv(1, 2),              # 32x32 -> 16x16, 2 channels
    conv(2, 4),              # 16x16 -> 8x8,   4 channels
    # At this point, we have an 8x8x4 "latent representation"
    # That's 256 values (compared to 784 original pixels)
    
    # ===== DECODER (Reconstruct) =====
    # deconv layers double the spatial size
    deconv(4, 2),            # 8x8   -> 16x16, 2 channels
    deconv(2, 1, act=False), # 16x16 -> 32x32, 1 channel (no ReLU!)
    
    # ===== POSTPROCESSING =====
    # Crop back from 32x32 to 28x28
    # ZeroPad2d(-2) removes 2 pixels from each side
    nn.ZeroPad2d(-2),        # 32x32 -> 28x28
    
    # Sigmoid squashes output to [0, 1] range (like our input)
    nn.Sigmoid()
).to(def_device)

print("Autoencoder Architecture:")
print("=" * 50)
print(ae)

### Understanding the Architecture

Let's trace through the shapes:

| Layer | Operation | Shape | Notes |
|-------|-----------|-------|-------|
| Input | - | (1, 28, 28) | Original image |
| ZeroPad2d(2) | Pad | (1, 32, 32) | Add border |
| conv(1, 2) | Encode | (2, 16, 16) | Halve size |
| conv(2, 4) | Encode | (4, 8, 8) | **Bottleneck!** |
| deconv(4, 2) | Decode | (2, 16, 16) | Double size |
| deconv(2, 1) | Decode | (1, 32, 32) | Double size |
| ZeroPad2d(-2) | Crop | (1, 28, 28) | Remove border |
| Sigmoid | Activate | (1, 28, 28) | Squash to [0,1] |

**The bottleneck** at 8×8×4 = 256 values is where the compression happens. The original image has 28×28 = 784 pixels, so we're compressing to about 1/3 the size!

### 🔍 Visualize It: The Architecture in 3D

Each layer is a 3-D block — width × height are the spatial size, depth is the channel count. **Drag to rotate, scroll to zoom.** Step through the layers (or click one on the rail) to watch the volumes shrink to the bottleneck and grow back out into the classic hourglass.

In [ ]:
# ============================================================================
# INTERACTIVE VISUALIZATION (self-contained) -- run this cell.
# 3D tensor volumes: encoder compresses, bottleneck, decoder expands.
# The widget lives in the separate file `ae_3d_bottleneck.html`; it is embedded here as a
# base64 data-URI iframe, so the notebook stays fully self-contained and works
# offline in Jupyter, Colab, VS Code, and the exported HTML. The iframe
# broadcasts its own content height, and the listener below resizes it to fit
# exactly -- full width, wrapped to content, with no empty space below.
# ============================================================================
from IPython.display import HTML
import base64

_html_b64 = "PCFET0NUWVBFIGh0bWw+CjxodG1sIGxhbmc9ImVuIj4KPGhlYWQ+CjxtZXRhIGNoYXJzZXQ9IlVURi04Ij4KPG1ldGEgbmFtZT0idmlld3BvcnQiIGNvbnRlbnQ9IndpZHRoPWRldmljZS13aWR0aCwgaW5pdGlhbC1zY2FsZT0xLjAiPgo8dGl0bGU+M0QgQXV0b2VuY29kZXIgQXJjaGl0ZWN0dXJlPC90aXRsZT4KPHN0eWxlPgogIDpyb290ewogICAgLS1iZzojRURGMkZCOyAtLWNhcmQ6I0ZGRkZGRjsgLS1pbms6IzFGMkE0NDsgLS1tdXRlZDojNjQ3NDhCOwogICAgLS1wdXJwbGU6IzZDNUNFNzsgLS1wdXJwbGUtc29mdDojRUZFQkZGOwogICAgLS10ZWFsOiMwRTlDOEY7IC0tdGVhbC1zb2Z0OiNFMEY1RjI7CiAgICAtLW9yYW5nZTojRTg4MjFGOyAtLW9yYW5nZS1zb2Z0OiNGREVFREM7CiAgICAtLWxpbmU6I0RERTVGMjsKICAgIC0tc2hhZG93OjAgMTBweCAyOHB4IHJnYmEoMTA4LDkyLDIzMSwuMTQpOwogICAgLS1zaGFkb3ctc206MCA0cHggMTRweCByZ2JhKDEwOCw5MiwyMzEsLjEwKTsKICAgIC0tcmFkaXVzOjE2cHg7CiAgICAtLW1vbm86IlNGIE1vbm8iLHVpLW1vbm9zcGFjZSxNZW5sbyxDb25zb2xhcyxtb25vc3BhY2U7CiAgfQogICp7Ym94LXNpemluZzpib3JkZXItYm94O21hcmdpbjowO3BhZGRpbmc6MH0KICBib2R5e2JhY2tncm91bmQ6dmFyKC0tYmcpO2NvbG9yOnZhcigtLWluayk7CiAgICBmb250OjE1cHgvMS41NSAtYXBwbGUtc3lzdGVtLCJTZWdvZSBVSSIsSW50ZXIsUm9ib3RvLHNhbnMtc2VyaWY7CiAgICBwYWRkaW5nOjIycHggMTZweCAxOHB4O292ZXJmbG93LXg6aGlkZGVuO30KICAud3JhcHttYXgtd2lkdGg6MTA4MHB4O21hcmdpbjowIGF1dG99CiAgaGVhZGVyIGgxe2ZvbnQtc2l6ZToyM3B4O2ZvbnQtd2VpZ2h0OjgwMDtsZXR0ZXItc3BhY2luZzotLjAyZW19CiAgaGVhZGVyIGgxIC5obHtjb2xvcjp2YXIoLS1wdXJwbGUpfQogIGhlYWRlciBwLnN1Yntjb2xvcjp2YXIoLS1tdXRlZCk7bWFyZ2luLXRvcDo2cHg7bWF4LXdpZHRoOjgyMHB4fQogIGhlYWRlciBwLnN1YiBjb2Rle2ZvbnQtZmFtaWx5OnZhcigtLW1vbm8pO2ZvbnQtc2l6ZTouOTJlbTtiYWNrZ3JvdW5kOiNmZmY7Ym9yZGVyLXJhZGl1czo2cHg7cGFkZGluZzoxcHggNnB4O2JveC1zaGFkb3c6dmFyKC0tc2hhZG93LXNtKX0KCiAgLmNvbnRyb2xze2Rpc3BsYXk6ZmxleDthbGlnbi1pdGVtczpjZW50ZXI7Z2FwOjE0cHg7ZmxleC13cmFwOndyYXA7CiAgICBiYWNrZ3JvdW5kOnZhcigtLWNhcmQpO2JvcmRlci1yYWRpdXM6dmFyKC0tcmFkaXVzKTtib3gtc2hhZG93OnZhcigtLXNoYWRvdyk7cGFkZGluZzoxM3B4IDE2cHg7bWFyZ2luOjE4cHggMCAxNnB4fQogIGJ1dHRvbntmb250OmluaGVyaXQ7Zm9udC13ZWlnaHQ6NzAwO2JvcmRlcjpub25lO2JvcmRlci1yYWRpdXM6MTJweDtjdXJzb3I6cG9pbnRlcjtwYWRkaW5nOjlweCAxNnB4O3RyYW5zaXRpb246dHJhbnNmb3JtIC4xMnN9CiAgI3ByZXYsI25leHR7YmFja2dyb3VuZDp2YXIoLS1wdXJwbGUtc29mdCk7Y29sb3I6dmFyKC0tcHVycGxlKX0KICAjcGxheXtiYWNrZ3JvdW5kOnZhcigtLXB1cnBsZSk7Y29sb3I6I2ZmZjtib3gtc2hhZG93OjAgNnB4IDE2cHggcmdiYSgxMDgsOTIsMjMxLC4zNSl9CiAgI3BsYXk6aG92ZXJ7dHJhbnNmb3JtOnRyYW5zbGF0ZVkoLTFweCl9CiAgI3Jlc2V0e2JhY2tncm91bmQ6dHJhbnNwYXJlbnQ7Y29sb3I6dmFyKC0tbXV0ZWQpO3RleHQtZGVjb3JhdGlvbjp1bmRlcmxpbmU7cGFkZGluZzo5cHggOHB4fQogIC5zdGVwbnVte2NvbG9yOnZhcigtLW11dGVkKTtmb250LXNpemU6MTNweDtmb250LXdlaWdodDo3MDA7bWluLXdpZHRoOjg4cHh9CiAgLnNwZHtkaXNwbGF5OmZsZXg7YWxpZ24taXRlbXM6Y2VudGVyO2dhcDo4cHg7bWFyZ2luLWxlZnQ6YXV0bztmb250LXNpemU6MTNweDtjb2xvcjp2YXIoLS1tdXRlZCk7Zm9udC13ZWlnaHQ6NjAwfQogIC5zcGQgaW5wdXR7YWNjZW50LWNvbG9yOnZhcigtLXB1cnBsZSk7d2lkdGg6MTEwcHh9CgogIC5sYXllcnJhaWx7ZGlzcGxheTpncmlkO2dyaWQtdGVtcGxhdGUtY29sdW1uczpyZXBlYXQoNywxZnIpO2dhcDo2cHg7bWFyZ2luOjAgMCAxNHB4fQogIC5sYXllcnJhaWwgLmxye2JhY2tncm91bmQ6I2ZmZjtib3JkZXI6MnB4IHNvbGlkIHZhcigtLWxpbmUpO2JvcmRlci1yYWRpdXM6MTBweDtwYWRkaW5nOjdweCA0cHg7dGV4dC1hbGlnbjpjZW50ZXI7Y3Vyc29yOnBvaW50ZXI7dHJhbnNpdGlvbjouMTVzO3VzZXItc2VsZWN0Om5vbmV9CiAgLmxheWVycmFpbCAubHI6aG92ZXJ7Ym9yZGVyLWNvbG9yOnZhcigtLXB1cnBsZSl9CiAgLmxheWVycmFpbCAubHIgLm5te2ZvbnQtc2l6ZToxMXB4O2ZvbnQtd2VpZ2h0OjgwMH0KICAubGF5ZXJyYWlsIC5sciAuc2h7Zm9udC1mYW1pbHk6dmFyKC0tbW9ubyk7Zm9udC1zaXplOjkuNXB4O2NvbG9yOnZhcigtLW11dGVkKTttYXJnaW4tdG9wOjJweH0KICAubGF5ZXJyYWlsIC5sci5hY3RpdmV7Ym94LXNoYWRvdzp2YXIoLS1zaGFkb3cpfQogIC5sYXllcnJhaWwgLmxyLmVuYy5hY3RpdmV7YmFja2dyb3VuZDp2YXIoLS1vcmFuZ2UpO2JvcmRlci1jb2xvcjp2YXIoLS1vcmFuZ2UpfQogIC5sYXllcnJhaWwgLmxyLmJuLmFjdGl2ZXtiYWNrZ3JvdW5kOnZhcigtLXB1cnBsZSk7Ym9yZGVyLWNvbG9yOnZhcigtLXB1cnBsZSl9CiAgLmxheWVycmFpbCAubHIuZGVjLmFjdGl2ZXtiYWNrZ3JvdW5kOnZhcigtLXRlYWwpO2JvcmRlci1jb2xvcjp2YXIoLS10ZWFsKX0KICAubGF5ZXJyYWlsIC5sci5hY3RpdmUgLm5tLC5sYXllcnJhaWwgLmxyLmFjdGl2ZSAuc2h7Y29sb3I6I2ZmZn0KCiAgLmdyaWR7ZGlzcGxheTpncmlkO2dyaWQtdGVtcGxhdGUtY29sdW1uczoxLjNmciAuODVmcjtnYXA6MThweDthbGlnbi1pdGVtczpzdGFydH0KICAuZ3JpZC5pcy1uYXJyb3d7Z3JpZC10ZW1wbGF0ZS1jb2x1bW5zOjFmcn0KICAuY2FyZHtiYWNrZ3JvdW5kOnZhcigtLWNhcmQpO2JvcmRlci1yYWRpdXM6dmFyKC0tcmFkaXVzKTtib3gtc2hhZG93OnZhcigtLXNoYWRvdyk7cGFkZGluZzoxNHB4fQogIC5jYXJkIGgye2ZvbnQtc2l6ZToxMnB4O2ZvbnQtd2VpZ2h0OjgwMDtsZXR0ZXItc3BhY2luZzouMDhlbTt0ZXh0LXRyYW5zZm9ybTp1cHBlcmNhc2U7Y29sb3I6dmFyKC0tbXV0ZWQpO21hcmdpbi1ib3R0b206MTBweH0KICAjc3RhZ2V7d2lkdGg6MTAwJTtoZWlnaHQ6MzgwcHg7Ym9yZGVyLXJhZGl1czoxMnB4O2JvcmRlcjoxcHggc29saWQgdmFyKC0tbGluZSk7YmFja2dyb3VuZDojZmJmZGZmO2N1cnNvcjpncmFiO3RvdWNoLWFjdGlvbjpub25lO2Rpc3BsYXk6YmxvY2t9CiAgLmhpbnR7dGV4dC1hbGlnbjpjZW50ZXI7Zm9udC1zaXplOjExLjVweDtjb2xvcjp2YXIoLS1tdXRlZCk7bWFyZ2luLXRvcDo4cHh9CgogIC5zdGVwdGl0bGV7Zm9udC1zaXplOjE3cHg7Zm9udC13ZWlnaHQ6ODAwO21hcmdpbi1ib3R0b206N3B4fQogIC5leHBsYWlue2NvbG9yOiMzMzQxNUN9LmV4cGxhaW4gcHttYXJnaW46MCAwIDhweH0KICAuZXhwbGFpbiBjb2Rle2ZvbnQtZmFtaWx5OnZhcigtLW1vbm8pO2ZvbnQtc2l6ZTouOWVtO2JhY2tncm91bmQ6I0YxRjRGQztib3JkZXItcmFkaXVzOjZweDtwYWRkaW5nOjFweCA2cHh9CiAgLmV4cGxhaW4gYi5ve2NvbG9yOnZhcigtLW9yYW5nZSl9LmV4cGxhaW4gYi50e2NvbG9yOnZhcigtLXRlYWwpfS5leHBsYWluIGIucHtjb2xvcjp2YXIoLS1wdXJwbGUpfQogIC5zaGFwZWJveHtkaXNwbGF5OmZsZXg7Z2FwOjhweDttYXJnaW4tdG9wOjEwcHh9CiAgLnNoYXBlYm94IC5waWxse2ZsZXg6MTtiYWNrZ3JvdW5kOiNGNkY4RkU7Ym9yZGVyLXJhZGl1czoxMHB4O3BhZGRpbmc6OHB4O3RleHQtYWxpZ246Y2VudGVyfQogIC5zaGFwZWJveCAucGlsbCAubntmb250LWZhbWlseTp2YXIoLS1tb25vKTtmb250LXdlaWdodDo4MDA7Zm9udC1zaXplOjE1cHh9CiAgLnNoYXBlYm94IC5waWxsIC50e2ZvbnQtc2l6ZToxMHB4O2NvbG9yOnZhcigtLW11dGVkKTtmb250LXdlaWdodDo3MDA7dGV4dC10cmFuc2Zvcm06dXBwZXJjYXNlfQogIHByZS5jb2Rle2ZvbnQtZmFtaWx5OnZhcigtLW1vbm8pO2ZvbnQtc2l6ZToxMnB4O2xpbmUtaGVpZ2h0OjEuNjtiYWNrZ3JvdW5kOiNGNkY4RkU7Ym9yZGVyLXJhZGl1czoxMnB4O3BhZGRpbmc6MTBweCAwO292ZXJmbG93LXg6YXV0bzttYXJnaW4tdG9wOjEwcHh9CiAgLmNse3BhZGRpbmc6MCAxNHB4O3doaXRlLXNwYWNlOnByZX0KICAuY2wuaGx7YmFja2dyb3VuZDp2YXIoLS1wdXJwbGUtc29mdCk7Ym9yZGVyLWxlZnQ6NHB4IHNvbGlkIHZhcigtLXB1cnBsZSk7cGFkZGluZy1sZWZ0OjEwcHg7Zm9udC13ZWlnaHQ6NzAwfQogIC5jbC5jbXtjb2xvcjojOEE5NEFDfQogIGZvb3RlcnttYXJnaW4tdG9wOjE2cHg7Y29sb3I6dmFyKC0tbXV0ZWQpO2ZvbnQtc2l6ZToxMi41cHg7dGV4dC1hbGlnbjpjZW50ZXJ9CiAgQG1lZGlhIChwcmVmZXJzLXJlZHVjZWQtbW90aW9uOiByZWR1Y2Upeyp7dHJhbnNpdGlvbjpub25lIWltcG9ydGFudH19Cjwvc3R5bGU+CjwvaGVhZD4KPGJvZHk+CjxkaXYgY2xhc3M9IndyYXAiPgo8aGVhZGVyPgogIDxoMT5UaGUgYXV0b2VuY29kZXIgYXMgYW4gPHNwYW4gY2xhc3M9ImhsIj5ob3VyZ2xhc3M8L3NwYW4+PC9oMT4KICA8cCBjbGFzcz0ic3ViIj5FYWNoIGxheWVyIGlzIGEgMy1EIGJsb2NrOiA8Yj53aWR0aCDDlyBoZWlnaHQ8L2I+IGFyZSB0aGUgc3BhdGlhbCBzaXplLCA8Yj5kZXB0aDwvYj4gaXMgdGhlIG51bWJlciBvZiBjaGFubmVscy4KICBUaGUgZW5jb2RlciBtYWtlcyBibG9ja3MgPGIgc3R5bGU9ImNvbG9yOnZhcigtLW9yYW5nZSkiPnRoaW5uZXIgYnV0IGRlZXBlcjwvYj4gdW50aWwgdGhlIDxiIHN0eWxlPSJjb2xvcjp2YXIoLS1wdXJwbGUpIj5ib3R0bGVuZWNrPC9iPiwKICB0aGVuIHRoZSBkZWNvZGVyIG1pcnJvcnMgaXQgYmFjayBvdXQuIDxiPkRyYWcgdG8gcm90YXRlLCBzY3JvbGwgdG8gem9vbS48L2I+IFN0ZXAgdGhyb3VnaCB0aGUgbGF5ZXJzIG9yIGNsaWNrIG9uZSBvbiB0aGUgcmFpbC48L3A+CjwvaGVhZGVyPgoKPGRpdiBjbGFzcz0iY29udHJvbHMiPgogIDxidXR0b24gaWQ9InByZXYiPuKGkCBCYWNrPC9idXR0b24+CiAgPGJ1dHRvbiBpZD0icGxheSI+UGxheSDilrY8L2J1dHRvbj4KICA8YnV0dG9uIGlkPSJuZXh0Ij5OZXh0IOKGkjwvYnV0dG9uPgogIDxzcGFuIGNsYXNzPSJzdGVwbnVtIiBpZD0ic3RlcG51bSI+TGF5ZXIgMCAvIDY8L3NwYW4+CiAgPGxhYmVsIGNsYXNzPSJzcGQiPnNwZWVkIDxpbnB1dCB0eXBlPSJyYW5nZSIgaWQ9InNwZCIgbWluPSIxIiBtYXg9IjYiIHN0ZXA9IjEiIHZhbHVlPSIzIj48c3BhbiBpZD0ic3BkdiI+MS4ww5c8L3NwYW4+PC9sYWJlbD4KICA8YnV0dG9uIGlkPSJyZXNldCI+UmVzZXQgdmlldzwvYnV0dG9uPgo8L2Rpdj4KCjxkaXYgY2xhc3M9ImxheWVycmFpbCIgaWQ9InJhaWwiPjwvZGl2PgoKPGRpdiBjbGFzcz0iZ3JpZCIgaWQ9ImdyaWQiPgogIDxkaXYgY2xhc3M9ImNhcmQiPgogICAgPGgyPlRlbnNvciB2b2x1bWVzIHRocm91Z2ggdGhlIG5ldHdvcms8L2gyPgogICAgPGRpdiBpZD0ic3RhZ2UiPjwvZGl2PgogICAgPGRpdiBjbGFzcz0iaGludCI+T3JhbmdlID0gZW5jb2RlciAoY29tcHJlc3MpIMK3IFB1cnBsZSA9IGJvdHRsZW5lY2sgwrcgVGVhbCA9IGRlY29kZXIgKGV4cGFuZCkuIERyYWcgdG8gb3JiaXQuPC9kaXY+CiAgPC9kaXY+CiAgPGRpdj4KICAgIDxkaXYgY2xhc3M9ImNhcmQiPgogICAgICA8ZGl2IGNsYXNzPSJzdGVwdGl0bGUiIGlkPSJ0aXRsZSI+PC9kaXY+CiAgICAgIDxkaXYgY2xhc3M9ImV4cGxhaW4iIGlkPSJleHBsYWluIj48L2Rpdj4KICAgICAgPGRpdiBjbGFzcz0ic2hhcGVib3giPgogICAgICAgIDxkaXYgY2xhc3M9InBpbGwiPjxkaXYgY2xhc3M9Im4iIGlkPSJzcC1zaGFwZSI+McOXMjjDlzI4PC9kaXY+PGRpdiBjbGFzcz0idCI+c2hhcGU8L2Rpdj48L2Rpdj4KICAgICAgICA8ZGl2IGNsYXNzPSJwaWxsIj48ZGl2IGNsYXNzPSJuIiBpZD0ic3AtdmFscyI+Nzg0PC9kaXY+PGRpdiBjbGFzcz0idCI+dmFsdWVzPC9kaXY+PC9kaXY+CiAgICAgIDwvZGl2PgogICAgICA8cHJlIGNsYXNzPSJjb2RlIiBpZD0iY29kZSI+PC9wcmU+CiAgICA8L2Rpdj4KICA8L2Rpdj4KPC9kaXY+Cgo8Zm9vdGVyPlRoZSBuYXJyb3cgd2Fpc3QgaW4gdGhlIG1pZGRsZSBpcyB0aGUgd2hvbGUgcG9pbnQ6IGV2ZXJ5IHJlY29uc3RydWN0aW9uIG11c3QgYmUgcmVidWlsdCBmcm9tIGp1c3QgdGhlIDI1NiBudW1iZXJzIHN0b3JlZCB0aGVyZS48L2Zvb3Rlcj4KPC9kaXY+Cgo8c2NyaXB0IHNyYz0iaHR0cHM6Ly9jZG5qcy5jbG91ZGZsYXJlLmNvbS9hamF4L2xpYnMvdGhyZWUuanMvcjEyOC90aHJlZS5taW4uanMiPjwvc2NyaXB0Pgo8c2NyaXB0PgooZnVuY3Rpb24oKXsKInVzZSBzdHJpY3QiOwpjb25zdCAkPWlkPT5kb2N1bWVudC5nZXRFbGVtZW50QnlJZChpZCk7CmNvbnN0IHNwZWVkcz1bMC41LDAuNzUsMS4wLDEuNSwyLjAsMy4wXTsKCi8vIGxheWVyIHNwZWNzOiBuYW1lLCBjaGFubmVscyhkZXB0aCksIHNwYXRpYWwody9oKSwgcm9sZSwgeC1wb3NpdGlvbgpjb25zdCBMPVsKICB7bm06J0lucHV0JywgIHNoOicxw5cyOMOXMjgnLCBjaDoxLCAgc3A6MjgsIHZhbHM6Nzg0LCByb2xlOidlbmMnLCB0aXRsZTonSW5wdXQgaW1hZ2UnLAogICBleDpgPHA+T25lIGdyYXlzY2FsZSBpbWFnZTogPGNvZGU+KDEsIDI4LCAyOCk8L2NvZGU+LiBBIHRoaW4gYmxvY2sgKDEgY2hhbm5lbCBkZWVwKSBidXQgd2lkZSBhbmQgdGFsbC48L3A+YCwKICAgY29kZTpbWyd4YiAjIChicywgMSwgMjgsIDI4KSddXX0sCiAge25tOidQYWQnLCAgICBzaDonMcOXMzLDlzMyJywgY2g6MSwgIHNwOjMyLCB2YWxzOjEwMjQsIHJvbGU6J2VuYycsIHRpdGxlOidaZXJvUGFkMmQoMiknLAogICBleDpgPHA+V2UgcGFkIHRvIDxjb2RlPigxLCAzMiwgMzIpPC9jb2RlPiBzbyB0aGUgaGFsdmluZyBkaXZpZGVzIGNsZWFubHkgZG93biB0byB0aGUgYm90dGxlbmVjay48L3A+YCwKICAgY29kZTpbWydubi5aZXJvUGFkMmQoMiksICAjIDI4IOKGkiAzMicsJ2hsJ11dfSwKICB7bm06J2NvbnYxJywgIHNoOicyw5cxNsOXMTYnLCBjaDoyLCAgc3A6MTYsIHZhbHM6NTEyLCByb2xlOidlbmMnLCB0aXRsZTonY29udigxLCAyKSDigJQgZW5jb2RlJywKICAgZXg6YDxwPkZpcnN0IHN0cmlkZWQgY29udjogc3BhdGlhbCA8YiBjbGFzcz0ibyI+aGFsdmVzPC9iPiAzMuKGkjE2LCBjaGFubmVscyBnbyAx4oaSMi4gVGhlIGJsb2NrIGdldHMKICAgPGI+dGhpbm5lciBhbmQgYSBiaXQgZGVlcGVyPC9iPi48L3A+YCwKICAgY29kZTpbWydjb252KDEsIDIpLCAgIyAzMsOXMzIg4oaSIDE2w5cxNiwgMmNoJywnaGwnXV19LAogIHtubTonY29udjInLCAgc2g6JzTDlzjDlzgnLCBjaDo0LCAgc3A6OCwgIHZhbHM6MjU2LCByb2xlOidibicsIHRpdGxlOidjb252KDIsIDQpIOKAlCB0aGUgYm90dGxlbmVjaycsCiAgIGV4OmA8cD5UaGUgbmFycm93IHdhaXN0OiA8Y29kZT4oNCwgOCwgOCk8L2NvZGU+ID0gPGIgY2xhc3M9InAiPjI1NiB2YWx1ZXM8L2I+LiBUaGlzIGlzIHRoZSBjb21wcmVzc2VkIGxhdGVudCBjb2RlIOKAlAogICBhIDxiPjMuMDbDlzwvYj4gc3F1ZWV6ZSBmcm9tIHRoZSBvcmlnaW5hbCA3ODQuIEV2ZXJ5dGhpbmcgZG93bnN0cmVhbSBpcyByZWJ1aWx0IGZyb20gaGVyZS48L3A+YCwKICAgY29kZTpbWydjb252KDIsIDQpLCAgIyAxNsOXMTYg4oaSIDjDlzgsIDRjaCcsJ2hsJ10sCiAgICAgICAgIFsnIyA0wrc4wrc4ID0gMjU2ICDihpAgYm90dGxlbmVjaycsJ2NtJ11dfSwKICB7bm06J2RlY29udjEnLHNoOicyw5cxNsOXMTYnLCBjaDoyLCAgc3A6MTYsIHZhbHM6NTEyLCByb2xlOidkZWMnLCB0aXRsZTonZGVjb252KDQsIDIpIOKAlCBkZWNvZGUnLAogICBleDpgPHA+Tm93IHdlIGV4cGFuZC4gPGNvZGU+ZGVjb252PC9jb2RlPiB1cHNhbXBsZXMgOOKGkjE2IGFuZCByZWZpbmVzLCBjaGFubmVscyA04oaSMi4gVGhlIGJsb2NrIGdyb3dzCiAgIDxiIGNsYXNzPSJ0Ij53aWRlciBhZ2FpbjwvYj4uPC9wPmAsCiAgIGNvZGU6W1snZGVjb252KDQsIDIpLCAgIyA4w5c4IOKGkiAxNsOXMTYsIDJjaCcsJ2hsJ11dfSwKICB7bm06J2RlY29udjInLHNoOicxw5czMsOXMzInLCBjaDoxLCAgc3A6MzIsIHZhbHM6MTAyNCwgcm9sZTonZGVjJywgdGl0bGU6J2RlY29udigyLCAxKSDigJQgZGVjb2RlJywKICAgZXg6YDxwPlNlY29uZCBkZWNvbnY6IDE24oaSMzIgc3BhdGlhbGx5LCBiYWNrIHRvIDEgY2hhbm5lbC4gQWxtb3N0IHRoZSBvcmlnaW5hbCBzaXplLCBubyBSZUxVIG9uIHRoaXMgbGFzdAogICBmZWF0dXJlIGxheWVyLjwvcD5gLAogICBjb2RlOltbJ2RlY29udigyLCAxLCBhY3Q9RmFsc2UpLCAjIOKGkiAzMsOXMzInLCdobCddXX0sCiAge25tOidPdXRwdXQnLCBzaDonMcOXMjjDlzI4JywgY2g6MSwgIHNwOjI4LCB2YWxzOjc4NCwgcm9sZTonZGVjJywgdGl0bGU6J0Nyb3AgKyBTaWdtb2lkIOKAlCBvdXRwdXQnLAogICBleDpgPHA+Q3JvcCBiYWNrIHRvIDxjb2RlPigxLCAyOCwgMjgpPC9jb2RlPiBhbmQgc3F1YXNoIHRocm91Z2ggPGNvZGU+U2lnbW9pZDwvY29kZT4gaW50byA8Y29kZT5bMCwxXTwvY29kZT4uCiAgIFNhbWUgc2hhcGUgYXMgdGhlIGlucHV0IOKAlCB0aGUgaG91cmdsYXNzIGlzIGNvbXBsZXRlLjwvcD5gLAogICBjb2RlOltbJ25uLlplcm9QYWQyZCgtMiksICAjIDMyIOKGkiAyOCcsJ2NtJ10sCiAgICAgICAgIFsnbm4uU2lnbW9pZCgpICAgICAgICMgWzAsMV0nLCdobCddXX0KXTsKCi8vIGJ1aWxkIHJhaWwKY29uc3QgcmFpbD0kKCdyYWlsJyk7CkwuZm9yRWFjaCgobCxpKT0+e2NvbnN0IGQ9ZG9jdW1lbnQuY3JlYXRlRWxlbWVudCgnZGl2Jyk7ZC5jbGFzc05hbWU9J2xyICcrbC5yb2xlO2QuZGF0YXNldC5pPWk7CiAgZC5pbm5lckhUTUw9YDxkaXYgY2xhc3M9Im5tIj4ke2wubm19PC9kaXY+PGRpdiBjbGFzcz0ic2giPiR7bC5zaH08L2Rpdj5gOwogIGQuYWRkRXZlbnRMaXN0ZW5lcignY2xpY2snLCgpPT57c3RvcFBsYXkoKTtnbyhpKTt9KTtyYWlsLmFwcGVuZENoaWxkKGQpO30pOwoKLy8gLS0tLSB0aHJlZS5qcyBzY2VuZSAtLS0tCmNvbnN0IHN0YWdlPSQoJ3N0YWdlJyk7CmxldCBXPXN0YWdlLmNsaWVudFdpZHRofHw3MDAsIEg9MzgwOwpjb25zdCByZW5kZXJlcj1uZXcgVEhSRUUuV2ViR0xSZW5kZXJlcih7YW50aWFsaWFzOnRydWV9KTsKcmVuZGVyZXIuc2V0U2l6ZShXLEgpO3JlbmRlcmVyLnNldFBpeGVsUmF0aW8od2luZG93LmRldmljZVBpeGVsUmF0aW98fDEpOwpyZW5kZXJlci5zZXRDbGVhckNvbG9yKDB4ZmJmZGZmLDEpO3N0YWdlLmFwcGVuZENoaWxkKHJlbmRlcmVyLmRvbUVsZW1lbnQpOwpjb25zdCBzY2VuZT1uZXcgVEhSRUUuU2NlbmUoKTsKY29uc3QgY2FtZXJhPW5ldyBUSFJFRS5QZXJzcGVjdGl2ZUNhbWVyYSg0MixXL0gsMC4xLDIwMCk7CnNjZW5lLmFkZChuZXcgVEhSRUUuQW1iaWVudExpZ2h0KDB4ZmZmZmZmLDAuODUpKTsKY29uc3QgZGw9bmV3IFRIUkVFLkRpcmVjdGlvbmFsTGlnaHQoMHhmZmZmZmYsMC41KTtkbC5wb3NpdGlvbi5zZXQoNiwxMCw4KTtzY2VuZS5hZGQoZGwpOwpjb25zdCBkbDI9bmV3IFRIUkVFLkRpcmVjdGlvbmFsTGlnaHQoMHhmZmZmZmYsMC4yNSk7ZGwyLnBvc2l0aW9uLnNldCgtNiw0LC04KTtzY2VuZS5hZGQoZGwyKTsKCmNvbnN0IENPTD17ZW5jOjB4RTg4MjFGLCBibjoweDZDNUNFNywgZGVjOjB4MEU5QzhGfTsKY29uc3QgU1A9MS43OyAgICAgICAgICAvLyB4IGdhcCBiZXR3ZWVuIGJsb2Nrcwpjb25zdCBibG9ja3M9W107CmNvbnN0IHRvdGFsVz0oTC5sZW5ndGgtMSkqU1A7CkwuZm9yRWFjaCgobCxpKT0+ewogIGNvbnN0IHc9bC5zcC8zMioyLjQ7ICAgICAgICAgIC8vIHNwYXRpYWwg4oaSIGN1YmUgdy9oCiAgY29uc3QgZGVwdGg9TWF0aC5tYXgoMC4xOCwgbC5jaC80KjEuNik7IC8vIGNoYW5uZWxzIOKGkiBkZXB0aAogIGNvbnN0IGdlbz1uZXcgVEhSRUUuQm94R2VvbWV0cnkoZGVwdGgsIHcsIHcpOwogIGNvbnN0IG1hdD1uZXcgVEhSRUUuTWVzaExhbWJlcnRNYXRlcmlhbCh7Y29sb3I6Q09MW2wucm9sZV0sdHJhbnNwYXJlbnQ6dHJ1ZSxvcGFjaXR5OjAuNTV9KTsKICBjb25zdCBtZXNoPW5ldyBUSFJFRS5NZXNoKGdlbyxtYXQpOwogIG1lc2gucG9zaXRpb24uc2V0KGkqU1AgLSB0b3RhbFcvMiwgMCwgMCk7CiAgc2NlbmUuYWRkKG1lc2gpOwogIGNvbnN0IGVkZ2VzPW5ldyBUSFJFRS5MaW5lU2VnbWVudHMobmV3IFRIUkVFLkVkZ2VzR2VvbWV0cnkoZ2VvKSwKICAgIG5ldyBUSFJFRS5MaW5lQmFzaWNNYXRlcmlhbCh7Y29sb3I6Q09MW2wucm9sZV19KSk7CiAgZWRnZXMucG9zaXRpb24uY29weShtZXNoLnBvc2l0aW9uKTsgc2NlbmUuYWRkKGVkZ2VzKTsKICBibG9ja3MucHVzaCh7bWVzaCxlZGdlcyxiYXNlOjAuNTV9KTsKfSk7Ci8vIGNvbm5lY3RpbmcgcmliYm9uIChhIHRoaW4gbGluZSB0aHJvdWdoIGNlbnRlcnMpCmNvbnN0IHB0cz1ibG9ja3MubWFwKGI9PmIubWVzaC5wb3NpdGlvbi5jbG9uZSgpKTsKY29uc3QgcmliYm9uPW5ldyBUSFJFRS5MaW5lKG5ldyBUSFJFRS5CdWZmZXJHZW9tZXRyeSgpLnNldEZyb21Qb2ludHMocHRzKSwKICBuZXcgVEhSRUUuTGluZUJhc2ljTWF0ZXJpYWwoe2NvbG9yOjB4YjljM2RjfSkpOwpzY2VuZS5hZGQocmliYm9uKTsKCmxldCB0aGV0YT0wLjksIHBoaT0xLjE1LCBSPTEzOwpmdW5jdGlvbiBwbGFjZUNhbSgpe2NhbWVyYS5wb3NpdGlvbi5zZXQoUipNYXRoLnNpbihwaGkpKk1hdGguY29zKHRoZXRhKSxSKk1hdGguY29zKHBoaSksUipNYXRoLnNpbihwaGkpKk1hdGguc2luKHRoZXRhKSk7Y2FtZXJhLmxvb2tBdCgwLDAsMCk7fQpsZXQgZHJhZ2dpbmc9ZmFsc2UsbHg9MCxseT0wOwpjb25zdCBlbD1yZW5kZXJlci5kb21FbGVtZW50OwplbC5hZGRFdmVudExpc3RlbmVyKCdwb2ludGVyZG93bicsZT0+e2RyYWdnaW5nPXRydWU7bHg9ZS5jbGllbnRYO2x5PWUuY2xpZW50WTtlbC5zZXRQb2ludGVyQ2FwdHVyZShlLnBvaW50ZXJJZCk7ZWwuc3R5bGUuY3Vyc29yPSdncmFiYmluZyc7fSk7CmVsLmFkZEV2ZW50TGlzdGVuZXIoJ3BvaW50ZXJtb3ZlJyxlPT57aWYoIWRyYWdnaW5nKXJldHVybjt0aGV0YSs9KGUuY2xpZW50WC1seCkqMC4wMDg7cGhpPU1hdGgubWluKDEuNSxNYXRoLm1heCgwLjI1LHBoaS0oZS5jbGllbnRZLWx5KSowLjAwNikpO2x4PWUuY2xpZW50WDtseT1lLmNsaWVudFk7cGxhY2VDYW0oKTt9KTsKZWwuYWRkRXZlbnRMaXN0ZW5lcigncG9pbnRlcnVwJywoKT0+e2RyYWdnaW5nPWZhbHNlO2VsLnN0eWxlLmN1cnNvcj0nZ3JhYic7fSk7CmVsLmFkZEV2ZW50TGlzdGVuZXIoJ3doZWVsJyxlPT57ZS5wcmV2ZW50RGVmYXVsdCgpO1I9TWF0aC5taW4oMjIsTWF0aC5tYXgoNS41LFIrZS5kZWx0YVkqMC4wMSkpO3BsYWNlQ2FtKCk7fSx7cGFzc2l2ZTpmYWxzZX0pOwoKbGV0IGN1cj0wLCBwbGF5aW5nPWZhbHNlLCB0aW1lcj1udWxsOwpmdW5jdGlvbiBoaWdobGlnaHQoKXsKICBibG9ja3MuZm9yRWFjaCgoYixpKT0+ewogICAgY29uc3Qgb249aT09PWN1cjsKICAgIGIubWVzaC5tYXRlcmlhbC5vcGFjaXR5ID0gb24/MC45MjowLjQ7CiAgICBiLmVkZ2VzLm1hdGVyaWFsLm9wYWNpdHkgPSBvbj8xOjAuNTsKICAgIGIubWVzaC5zY2FsZS5zZXRTY2FsYXIob24/MS4wODoxKTsKICAgIGIuZWRnZXMuc2NhbGUuc2V0U2NhbGFyKG9uPzEuMDg6MSk7CiAgfSk7Cn0KZnVuY3Rpb24gcmVuZGVyUGFuZWwoKXsKICBjb25zdCBsPUxbY3VyXTsKICAkKCdzdGVwbnVtJykudGV4dENvbnRlbnQ9YExheWVyICR7Y3VyfSAvICR7TC5sZW5ndGgtMX1gOwogICQoJ3RpdGxlJykudGV4dENvbnRlbnQ9bC50aXRsZTsKICAkKCdleHBsYWluJykuaW5uZXJIVE1MPWwuZXg7CiAgJCgnc3Atc2hhcGUnKS50ZXh0Q29udGVudD1sLnNoOyAkKCdzcC12YWxzJykudGV4dENvbnRlbnQ9bC52YWxzOwogIGNvbnN0IGNvZGVFbD0kKCdjb2RlJyk7Y29kZUVsLmlubmVySFRNTD0nJzsKICBmb3IoY29uc3QgbGluZSBvZiBsLmNvZGUpe2NvbnN0IGQ9ZG9jdW1lbnQuY3JlYXRlRWxlbWVudCgnZGl2Jyk7ZC5jbGFzc05hbWU9J2NsJysobGluZVsxXT8nICcrbGluZVsxXTonJyk7ZC50ZXh0Q29udGVudD1saW5lWzBdPT09Jyc/J1x1MDBBMCc6bGluZVswXTtjb2RlRWwuYXBwZW5kQ2hpbGQoZCk7fQogIGRvY3VtZW50LnF1ZXJ5U2VsZWN0b3JBbGwoJy5sYXllcnJhaWwgLmxyJykuZm9yRWFjaCgoZSxpKT0+ZS5jbGFzc0xpc3QudG9nZ2xlKCdhY3RpdmUnLGk9PT1jdXIpKTsKICAkKCdwcmV2JykuZGlzYWJsZWQ9Y3VyPT09MDskKCdwcmV2Jykuc3R5bGUub3BhY2l0eT1jdXI9PT0wPy40NToxOwogICQoJ25leHQnKS5kaXNhYmxlZD1jdXI9PT1MLmxlbmd0aC0xOyQoJ25leHQnKS5zdHlsZS5vcGFjaXR5PWN1cj09PUwubGVuZ3RoLTE/LjU6MTsKICBoaWdobGlnaHQoKTsgcG9zdEgoKTsKfQpmdW5jdGlvbiBnbyhpKXtjdXI9TWF0aC5tYXgoMCxNYXRoLm1pbihMLmxlbmd0aC0xLGkpKTtyZW5kZXJQYW5lbCgpO30KZnVuY3Rpb24gc3RvcFBsYXkoKXtwbGF5aW5nPWZhbHNlO2lmKHRpbWVyKWNsZWFyVGltZW91dCh0aW1lcik7JCgncGxheScpLnRleHRDb250ZW50PSdQbGF5IOKWtic7fQpmdW5jdGlvbiB0aWNrKCl7aWYoIXBsYXlpbmcpcmV0dXJuO2lmKGN1cj49TC5sZW5ndGgtMSl7c3RvcFBsYXkoKTtyZXR1cm47fWdvKGN1cisxKTt0aW1lcj1zZXRUaW1lb3V0KHRpY2ssMTUwMC9zcGVlZHNbKyQoJ3NwZCcpLnZhbHVlLTFdKTt9CiQoJ3BsYXknKS5hZGRFdmVudExpc3RlbmVyKCdjbGljaycsKCk9PntpZihwbGF5aW5nKXtzdG9wUGxheSgpO3JldHVybjt9aWYoY3VyPj1MLmxlbmd0aC0xKWdvKDApO3BsYXlpbmc9dHJ1ZTskKCdwbGF5JykudGV4dENvbnRlbnQ9J1BhdXNlIOKPuCc7dGltZXI9c2V0VGltZW91dCh0aWNrLDExMDAvc3BlZWRzWyskKCdzcGQnKS52YWx1ZS0xXSk7fSk7CiQoJ25leHQnKS5hZGRFdmVudExpc3RlbmVyKCdjbGljaycsKCk9PntzdG9wUGxheSgpO2dvKGN1cisxKTt9KTsKJCgncHJldicpLmFkZEV2ZW50TGlzdGVuZXIoJ2NsaWNrJywoKT0+e3N0b3BQbGF5KCk7Z28oY3VyLTEpO30pOwokKCdyZXNldCcpLmFkZEV2ZW50TGlzdGVuZXIoJ2NsaWNrJywoKT0+e3RoZXRhPTAuOTtwaGk9MS4xNTtSPTEzO3BsYWNlQ2FtKCk7fSk7CiQoJ3NwZCcpLmFkZEV2ZW50TGlzdGVuZXIoJ2lucHV0JywoKT0+eyQoJ3NwZHYnKS50ZXh0Q29udGVudD1zcGVlZHNbKyQoJ3NwZCcpLnZhbHVlLTFdLnRvRml4ZWQoMSkrJ8OXJzt9KTsKZG9jdW1lbnQuYWRkRXZlbnRMaXN0ZW5lcigna2V5ZG93bicsZT0+e2lmKGUua2V5PT09J0Fycm93UmlnaHQnKXtzdG9wUGxheSgpO2dvKGN1cisxKTt9aWYoZS5rZXk9PT0nQXJyb3dMZWZ0Jyl7c3RvcFBsYXkoKTtnbyhjdXItMSk7fX0pOwoKLy8gcmVzaXplIHRocmVlIGNhbnZhcyB3aXRoIGNvbnRhaW5lcgpmdW5jdGlvbiByZXNpemUzZCgpe1c9c3RhZ2UuY2xpZW50V2lkdGh8fDcwMDtyZW5kZXJlci5zZXRTaXplKFcsSCk7Y2FtZXJhLmFzcGVjdD1XL0g7Y2FtZXJhLnVwZGF0ZVByb2plY3Rpb25NYXRyaXgoKTt9Cgpjb25zdCBncmlkPSQoJ2dyaWQnKSwgd3JhcD1kb2N1bWVudC5xdWVyeVNlbGVjdG9yKCcud3JhcCcpOwpmdW5jdGlvbiByZWxheW91dCgpeyBncmlkLmNsYXNzTGlzdC50b2dnbGUoJ2lzLW5hcnJvdycsIHdpbmRvdy5pbm5lcldpZHRoPDg0MCk7IHJlc2l6ZTNkKCk7IH0KZnVuY3Rpb24gcG9zdEgoKXsgY29uc3QgaD13cmFwP01hdGguY2VpbCh3cmFwLmdldEJvdW5kaW5nQ2xpZW50UmVjdCgpLmhlaWdodCkrMzQ6ZG9jdW1lbnQuYm9keS5vZmZzZXRIZWlnaHQ7CiAgaWYod2luZG93LnBhcmVudCE9PXdpbmRvdykgd2luZG93LnBhcmVudC5wb3N0TWVzc2FnZSh7dHlwZTonYWUtZnJhbWUtaGVpZ2h0JyxoZWlnaHQ6aH0sJyonKTsgfQpmdW5jdGlvbiB1cGRhdGUoKXsgcmVsYXlvdXQoKTsgcG9zdEgoKTsgfQoKZnVuY3Rpb24gYW5pbWF0ZSgpe3JlcXVlc3RBbmltYXRpb25GcmFtZShhbmltYXRlKTtpZighZHJhZ2dpbmcpe3RoZXRhKz0wLjAwMTY7cGxhY2VDYW0oKTt9cmVuZGVyZXIucmVuZGVyKHNjZW5lLGNhbWVyYSk7fQpwbGFjZUNhbSgpO2FuaW1hdGUoKTtyZW5kZXJQYW5lbCgpOwoKd2luZG93LmFkZEV2ZW50TGlzdGVuZXIoJ2xvYWQnLHVwZGF0ZSk7IHdpbmRvdy5hZGRFdmVudExpc3RlbmVyKCdyZXNpemUnLHVwZGF0ZSk7CmlmKHdpbmRvdy5SZXNpemVPYnNlcnZlcikgbmV3IFJlc2l6ZU9ic2VydmVyKHBvc3RIKS5vYnNlcnZlKHdyYXB8fGRvY3VtZW50LmJvZHkpOwpyZWxheW91dCgpOyB1cGRhdGUoKTsKc2V0VGltZW91dCh1cGRhdGUsMjAwKTsgc2V0VGltZW91dCh1cGRhdGUsNzAwKTsKfSkoKTsKPC9zY3JpcHQ+CjwvYm9keT4KPC9odG1sPgo="

HTML('''
<iframe id="ae3d-frame"
        src="data:text/html;base64,''' + _html_b64 + '''"
        style="width:100%; height:800px; border:1px solid #DDE5F2;
               border-radius:16px; box-shadow:0 8px 24px rgba(108,92,231,.12);
               display:block;"
        loading="lazy" title="3D autoencoder architecture hourglass"></iframe>
<script>
(function(){
  function onMsg(e){
    if (e.data && e.data.type === "ae-frame-height") {
      var f = document.getElementById("ae3d-frame");
      if (f) f.style.height = (e.data.height) + "px";
    }
  }
  window.addEventListener("message", onMsg);
})();
</script>
''')


In [ ]:
# =====================================================
# VERIFY THE ARCHITECTURE
# =====================================================

# Test with a batch of images
test_output = ae(xb)

print(f"Input shape:  {xb.shape}")
print(f"Output shape: {test_output.shape}")
print(f"\nShapes match: {xb.shape == test_output.shape}")
print(f"\nOutput range: [{test_output.min():.3f}, {test_output.max():.3f}]")
print("(Should be in [0, 1] due to Sigmoid)")

---

# Part 5: Training the Autoencoder

---

## The Loss Function: MSE

We use **Mean Squared Error (MSE)** as our loss function:

$$\text{MSE} = \frac{1}{n} \sum_{i=1}^{n} (\text{pred}_i - \text{target}_i)^2$$

This measures the average squared difference between each predicted pixel and the corresponding original pixel. Lower MSE = better reconstruction.

In [ ]:
# =====================================================
# EVALUATE BEFORE TRAINING (RANDOM WEIGHTS)
# =====================================================

print("Loss before training (random weights):")
eval_ae(ae, F.mse_loss, dv)

The initial loss is high because the network outputs random noise. Let's train it!

In [ ]:
# =====================================================
# TRAINING PHASE 1: LOW LEARNING RATE
# =====================================================

# Start with a low learning rate to be safe
opt = optim.SGD(ae.parameters(), lr=0.01)

print("Phase 1: Training with lr=0.01")
print("=" * 40)
fit_ae(5, ae, F.mse_loss, opt, dt, dv)

In [ ]:
# =====================================================
# TRAINING PHASE 2: HIGHER LEARNING RATE
# =====================================================

# Now that the network is partially trained, we can use a higher LR
opt = optim.SGD(ae.parameters(), lr=0.1) # opt = optim.AdamW(ae.parameters(), lr=0.1)

print("\nPhase 2: Training with lr=0.1")
print("=" * 40)
fit_ae(5, ae, F.mse_loss, opt, dt, dv)

**The loss decreased significantly!** Let's see what the reconstructions look like.

---

# Part 6: Visualizing the Results

---

In [ ]:
# =====================================================
# GENERATE RECONSTRUCTIONS
# =====================================================

# Run a batch through the autoencoder
ae.eval()  # Set to evaluation mode
with torch.no_grad():
    reconstructions = ae(xb)

print(f"Original images shape: {xb.shape}")
print(f"Reconstructions shape: {reconstructions.shape}")

In [ ]:
# =====================================================
# DISPLAY RECONSTRUCTIONS
# =====================================================

print("Reconstructed Images:")
show_images(reconstructions[:16].cpu(), nrows=2, ncols=8, imsize=1.5)

In [ ]:
# =====================================================
# DISPLAY ORIGINAL IMAGES FOR COMPARISON
# =====================================================

print("Original Images:")
show_images(xb[:16].cpu(), nrows=2, ncols=8, imsize=1.5)

In [ ]:
# =====================================================
# SIDE-BY-SIDE COMPARISON
# =====================================================

# Let's show original and reconstruction side by side
fig, axes = plt.subplots(4, 8, figsize=(16, 8))

for i in range(8):
    # Top row: original
    show_image(xb[i].cpu(), ax=axes[0, i])
    if i == 0:
        axes[0, i].set_ylabel("Original", fontsize=12)
    
    # Second row: reconstruction
    show_image(reconstructions[i].cpu(), ax=axes[1, i])
    if i == 0:
        axes[1, i].set_ylabel("Reconstructed", fontsize=12)
    
    # Third row: another set of originals
    show_image(xb[i+8].cpu(), ax=axes[2, i])
    if i == 0:
        axes[2, i].set_ylabel("Original", fontsize=12)
    
    # Fourth row: their reconstructions
    show_image(reconstructions[i+8].cpu(), ax=axes[3, i])
    if i == 0:
        axes[3, i].set_ylabel("Reconstructed", fontsize=12)

plt.suptitle("Autoencoder: Original vs Reconstructed", fontsize=14)
plt.tight_layout()

**Observations:**

1. The reconstructions capture the **overall shape** of each clothing item
2. **Details are smoothed out** - this is expected with compression
3. The network learned to preserve the most important features
4. Some items reconstruct better than others (simpler shapes = easier)

---

In [ ]:
# =====================================================
# COMPUTE PER-IMAGE RECONSTRUCTION ERROR
# =====================================================

# Calculate MSE for each image individually
with torch.no_grad():
    # Compute squared differences
    sq_diff = (reconstructions - xb) ** 2
    # Average over each image (dims 1, 2, 3 are channel, height, width)
    per_image_mse = sq_diff.mean(dim=(1, 2, 3))

print("Reconstruction error (MSE) for first 16 images:")
for i in range(16):
    label = labels[yb[i].item()]
    print(f"  Image {i:2d} ({label:12s}): MSE = {per_image_mse[i]:.4f}")

---

# Part 7: Understanding What the Autoencoder Learned

---

## The Latent Space

The middle of the autoencoder (after the encoder, before the decoder) contains a compressed representation. Let's examine it.

In [ ]:
# =====================================================
# EXAMINING THE LATENT REPRESENTATION
# =====================================================

# Create just the encoder part
encoder = nn.Sequential(
    ae[0],  # ZeroPad2d
    ae[1],  # conv(1, 2)
    ae[2],  # conv(2, 4)
).to(def_device)

# Get the latent representation of our batch
with torch.no_grad():
    latent = encoder(xb)

print(f"Input shape:  {xb.shape}")
print(f"Latent shape: {latent.shape}")
print(f"\nCompression ratio: {xb.numel() / latent.numel():.1f}x")
print(f"  Original:   {28*28} = 784 values per image")
print(f"  Compressed: {8*8*4} = 256 values per image")

In [ ]:
# =====================================================
# VISUALIZE THE LATENT CHANNELS
# =====================================================

# Look at the 4 latent channels for one image
img_idx = 0

fig, axes = plt.subplots(1, 5, figsize=(15, 3))

# Original image
show_image(xb[img_idx].cpu(), ax=axes[0])
axes[0].set_title(f"Original\n{labels[yb[img_idx].item()]}")

# 4 latent channels
for c in range(4):
    show_image(latent[img_idx, c].cpu(), ax=axes[c+1], noframe=False)
    axes[c+1].set_title(f"Latent Channel {c}")

plt.suptitle("Latent Representation (8x8 per channel)", fontsize=14)
plt.tight_layout()

**Each latent channel captures different aspects of the image!** The decoder learns to combine these channels to reconstruct the original.

---

### 🔍 Visualize It: Walking Through the Latent Space

This is the payoff. Drag the marker around the compressed latent space (or click a cluster) and watch the decoder rebuild an image from that exact point. Moving *between* clusters morphs one garment into another — the sign of a well-organized latent space.

In [ ]:
# ============================================================================
# INTERACTIVE VISUALIZATION (self-contained) -- run this cell.
# Drag through the compressed code; watch reconstructions morph between clusters.
# The widget lives in the separate file `latent_space_explorer.html`; it is embedded here as a
# base64 data-URI iframe, so the notebook stays fully self-contained and works
# offline in Jupyter, Colab, VS Code, and the exported HTML. The iframe
# broadcasts its own content height, and the listener below resizes it to fit
# exactly -- full width, wrapped to content, with no empty space below.
# ============================================================================
from IPython.display import HTML
import base64

_html_b64 = "PCFET0NUWVBFIGh0bWw+CjxodG1sIGxhbmc9ImVuIj4KPGhlYWQ+CjxtZXRhIGNoYXJzZXQ9IlVURi04Ij4KPG1ldGEgbmFtZT0idmlld3BvcnQiIGNvbnRlbnQ9IndpZHRoPWRldmljZS13aWR0aCwgaW5pdGlhbC1zY2FsZT0xLjAiPgo8dGl0bGU+TGF0ZW50IFNwYWNlIEV4cGxvcmVyPC90aXRsZT4KPHN0eWxlPgogIDpyb290ewogICAgLS1iZzojRURGMkZCOyAtLWNhcmQ6I0ZGRkZGRjsgLS1pbms6IzFGMkE0NDsgLS1tdXRlZDojNjQ3NDhCOwogICAgLS1wdXJwbGU6IzZDNUNFNzsgLS1wdXJwbGUtc29mdDojRUZFQkZGOwogICAgLS10ZWFsOiMwRTlDOEY7IC0tdGVhbC1zb2Z0OiNFMEY1RjI7CiAgICAtLW9yYW5nZTojRTg4MjFGOyAtLW9yYW5nZS1zb2Z0OiNGREVFREM7CiAgICAtLWxpbmU6I0RERTVGMjsKICAgIC0tc2hhZG93OjAgMTBweCAyOHB4IHJnYmEoMTA4LDkyLDIzMSwuMTQpOwogICAgLS1zaGFkb3ctc206MCA0cHggMTRweCByZ2JhKDEwOCw5MiwyMzEsLjEwKTsKICAgIC0tcmFkaXVzOjE2cHg7CiAgICAtLW1vbm86IlNGIE1vbm8iLHVpLW1vbm9zcGFjZSxNZW5sbyxDb25zb2xhcyxtb25vc3BhY2U7CiAgfQogICp7Ym94LXNpemluZzpib3JkZXItYm94O21hcmdpbjowO3BhZGRpbmc6MH0KICBib2R5e2JhY2tncm91bmQ6dmFyKC0tYmcpO2NvbG9yOnZhcigtLWluayk7CiAgICBmb250OjE1cHgvMS41NSAtYXBwbGUtc3lzdGVtLCJTZWdvZSBVSSIsSW50ZXIsUm9ib3RvLHNhbnMtc2VyaWY7CiAgICBwYWRkaW5nOjIycHggMTZweCAxOHB4O292ZXJmbG93LXg6aGlkZGVuO30KICAud3JhcHttYXgtd2lkdGg6MTA4MHB4O21hcmdpbjowIGF1dG99CiAgaGVhZGVyIGgxe2ZvbnQtc2l6ZToyM3B4O2ZvbnQtd2VpZ2h0OjgwMDtsZXR0ZXItc3BhY2luZzotLjAyZW19CiAgaGVhZGVyIGgxIC5obHtjb2xvcjp2YXIoLS1wdXJwbGUpfQogIGhlYWRlciBwLnN1Yntjb2xvcjp2YXIoLS1tdXRlZCk7bWFyZ2luLXRvcDo2cHg7bWF4LXdpZHRoOjg0MHB4fQogIGhlYWRlciBwLnN1YiBjb2Rle2ZvbnQtZmFtaWx5OnZhcigtLW1vbm8pO2ZvbnQtc2l6ZTouOTJlbTtiYWNrZ3JvdW5kOiNmZmY7Ym9yZGVyLXJhZGl1czo2cHg7cGFkZGluZzoxcHggNnB4O2JveC1zaGFkb3c6dmFyKC0tc2hhZG93LXNtKX0KCiAgLmdyaWR7ZGlzcGxheTpncmlkO2dyaWQtdGVtcGxhdGUtY29sdW1uczoxLjE1ZnIgLjg1ZnI7Z2FwOjE4cHg7YWxpZ24taXRlbXM6c3RhcnQ7bWFyZ2luLXRvcDoxOHB4fQogIC5ncmlkLmlzLW5hcnJvd3tncmlkLXRlbXBsYXRlLWNvbHVtbnM6MWZyfQogIC5jYXJke2JhY2tncm91bmQ6dmFyKC0tY2FyZCk7Ym9yZGVyLXJhZGl1czp2YXIoLS1yYWRpdXMpO2JveC1zaGFkb3c6dmFyKC0tc2hhZG93KTtwYWRkaW5nOjE1cHggMTZweH0KICAuY2FyZCBoMntmb250LXNpemU6MTJweDtmb250LXdlaWdodDo4MDA7bGV0dGVyLXNwYWNpbmc6LjA4ZW07dGV4dC10cmFuc2Zvcm06dXBwZXJjYXNlO2NvbG9yOnZhcigtLW11dGVkKTttYXJnaW4tYm90dG9tOjEwcHg7ZGlzcGxheTpmbGV4O2p1c3RpZnktY29udGVudDpzcGFjZS1iZXR3ZWVufQogICNsYXRlbnR7d2lkdGg6MTAwJTtoZWlnaHQ6YXV0bztkaXNwbGF5OmJsb2NrO2JhY2tncm91bmQ6I2ZiZmRmZjtib3JkZXI6MXB4IHNvbGlkIHZhcigtLWxpbmUpO2JvcmRlci1yYWRpdXM6MTJweDtjdXJzb3I6Y3Jvc3NoYWlyO3RvdWNoLWFjdGlvbjpub25lfQogIC5sZWdlbmR7ZGlzcGxheTpmbGV4O2dhcDo5cHg7ZmxleC13cmFwOndyYXA7bWFyZ2luLXRvcDoxMHB4O2ZvbnQtc2l6ZToxMS41cHg7Y29sb3I6dmFyKC0tbXV0ZWQpfQogIC5sZWdlbmQgc3BhbntkaXNwbGF5OmlubGluZS1mbGV4O2FsaWduLWl0ZW1zOmNlbnRlcjtnYXA6NXB4fQogIC5kb3R7d2lkdGg6MTBweDtoZWlnaHQ6MTBweDtib3JkZXItcmFkaXVzOjUwJTtkaXNwbGF5OmlubGluZS1ibG9ja30KCiAgLnJlY297ZGlzcGxheTpmbGV4O2ZsZXgtZGlyZWN0aW9uOmNvbHVtbjthbGlnbi1pdGVtczpjZW50ZXI7Z2FwOjhweH0KICBjYW52YXMucmVjb257d2lkdGg6MTcwcHg7aGVpZ2h0OjE3MHB4O2JvcmRlci1yYWRpdXM6MTJweDtib3JkZXI6MXB4IHNvbGlkIHZhcigtLWxpbmUpO2ltYWdlLXJlbmRlcmluZzpwaXhlbGF0ZWQ7YmFja2dyb3VuZDojZmZmfQogIC5ybGFiZWx7Zm9udC13ZWlnaHQ6ODAwO2ZvbnQtc2l6ZToxNXB4O2NvbG9yOnZhcigtLXB1cnBsZSl9CiAgLmNvb3Jkc3tmb250LWZhbWlseTp2YXIoLS1tb25vKTtmb250LXNpemU6MTJweDtjb2xvcjp2YXIoLS1tdXRlZCk7YmFja2dyb3VuZDojRjZGOEZFO2JvcmRlci1yYWRpdXM6OHB4O3BhZGRpbmc6NnB4IDExcHg7dGV4dC1hbGlnbjpjZW50ZXJ9CiAgLmNoaXBze2Rpc3BsYXk6ZmxleDtnYXA6NnB4O2ZsZXgtd3JhcDp3cmFwO2p1c3RpZnktY29udGVudDpjZW50ZXI7bWFyZ2luLXRvcDo2cHh9CiAgLmNoaXBzIGJ1dHRvbntmb250OmluaGVyaXQ7Zm9udC1zaXplOjExcHg7Zm9udC13ZWlnaHQ6NzAwO2JvcmRlcjpub25lO2JvcmRlci1yYWRpdXM6OTk5cHg7cGFkZGluZzo1cHggMTFweDtjdXJzb3I6cG9pbnRlcjtiYWNrZ3JvdW5kOnZhcigtLXB1cnBsZS1zb2Z0KTtjb2xvcjp2YXIoLS1wdXJwbGUpfQogIC5jaGlwcyBidXR0b246aG92ZXJ7YmFja2dyb3VuZDp2YXIoLS1wdXJwbGUpO2NvbG9yOiNmZmZ9CiAgLmV4cGxhaW57Y29sb3I6IzMzNDE1Qztmb250LXNpemU6MTMuNXB4O21hcmdpbi10b3A6MTJweH0KICAuZXhwbGFpbiBwe21hcmdpbjowIDAgOHB4fQogIC5leHBsYWluIGIucHtjb2xvcjp2YXIoLS1wdXJwbGUpfS5leHBsYWluIGIudHtjb2xvcjp2YXIoLS10ZWFsKX0uZXhwbGFpbiBiLm97Y29sb3I6dmFyKC0tb3JhbmdlKX0KICAua2V5Ym94e21hcmdpbi10b3A6OHB4O2JhY2tncm91bmQ6dmFyKC0tcHVycGxlLXNvZnQpO2JvcmRlci1yYWRpdXM6MTJweDtwYWRkaW5nOjlweCAxM3B4O2NvbG9yOiM0NjM2Qzk7Zm9udC13ZWlnaHQ6NjAwO2ZvbnQtc2l6ZToxM3B4fQogIGZvb3RlcnttYXJnaW4tdG9wOjE2cHg7Y29sb3I6dmFyKC0tbXV0ZWQpO2ZvbnQtc2l6ZToxMi41cHg7dGV4dC1hbGlnbjpjZW50ZXJ9CiAgQG1lZGlhIChwcmVmZXJzLXJlZHVjZWQtbW90aW9uOiByZWR1Y2Upeyp7dHJhbnNpdGlvbjpub25lIWltcG9ydGFudH19Cjwvc3R5bGU+CjwvaGVhZD4KPGJvZHk+CjxkaXYgY2xhc3M9IndyYXAiPgo8aGVhZGVyPgogIDxoMT5XYWxrIHRocm91Z2ggdGhlIDxzcGFuIGNsYXNzPSJobCI+bGF0ZW50IHNwYWNlPC9zcGFuPjwvaDE+CiAgPHAgY2xhc3M9InN1YiI+QWZ0ZXIgdHJhaW5pbmcsIHRoZSBib3R0bGVuZWNrIGlzIG5vdCByYW5kb20g4oCUIHNpbWlsYXIgY2xvdGhlcyBsYW5kIG5lYXIgZWFjaCBvdGhlci4gVGhpcyBpcyBhIDItRCBza2V0Y2ggb2YKICB0aGF0IGNvbXByZXNzZWQgc3BhY2UuIDxiPkRyYWcgdGhlIHB1cnBsZSBtYXJrZXI8L2I+IChvciBjbGljayBhIGNsdXN0ZXIpIGFuZCB3YXRjaCB0aGUgZGVjb2RlciByZWJ1aWxkIGFuIGltYWdlIGZyb20KICB0aGF0IHBvaW50LiBNb3ZlIGJldHdlZW4gY2x1c3RlcnMgYW5kIHRoZSByZWNvbnN0cnVjdGlvbiA8YiBzdHlsZT0iY29sb3I6dmFyKC0tcHVycGxlKSI+bW9ycGhzIHNtb290aGx5PC9iPiDigJQgcHJvb2YgdGhlCiAgY29kZSBpcyBtZWFuaW5nZnVsLCBub3QganVzdCBtZW1vcml6ZWQuPC9wPgo8L2hlYWRlcj4KCjxkaXYgY2xhc3M9ImdyaWQiIGlkPSJncmlkIj4KICA8ZGl2IGNsYXNzPSJjYXJkIj4KICAgIDxoMj5MYXRlbnQgc3BhY2UgKGVuY29kZXIgb3V0cHV0LCAyLUQgdmlldykgPHNwYW4gc3R5bGU9ImZvbnQtZmFtaWx5OnZhcigtLW1vbm8pO2ZvbnQtd2VpZ2h0OjYwMDtjb2xvcjp2YXIoLS1tdXRlZCkiPmRyYWcgbWU8L3NwYW4+PC9oMj4KICAgIDxjYW52YXMgaWQ9ImxhdGVudCIgd2lkdGg9IjU0MCIgaGVpZ2h0PSI0MjAiPjwvY2FudmFzPgogICAgPGRpdiBjbGFzcz0ibGVnZW5kIiBpZD0ibGVnZW5kIj48L2Rpdj4KICA8L2Rpdj4KCiAgPGRpdiBjbGFzcz0iY2FyZCI+CiAgICA8aDI+RGVjb2RlciByZWJ1aWxkcyBmcm9tIHRoaXMgcG9pbnQ8L2gyPgogICAgPGRpdiBjbGFzcz0icmVjbyI+CiAgICAgIDxjYW52YXMgY2xhc3M9InJlY29uIiBpZD0icmVjb24iIHdpZHRoPSIyOCIgaGVpZ2h0PSIyOCI+PC9jYW52YXM+CiAgICAgIDxkaXYgY2xhc3M9InJsYWJlbCIgaWQ9InJsYWJlbCI+4oCUPC9kaXY+CiAgICAgIDxkaXYgY2xhc3M9ImNvb3JkcyIgaWQ9ImNvb3JkcyI+eiA9ICgwLjAwLCAwLjAwKTwvZGl2PgogICAgICA8ZGl2IGNsYXNzPSJjaGlwcyIgaWQ9ImNoaXBzIj48L2Rpdj4KICAgIDwvZGl2PgogICAgPGRpdiBjbGFzcz0iZXhwbGFpbiI+CiAgICAgIDxwPlRoZSBtYXJrZXIncyBwb3NpdGlvbiBpcyBhIDItRCBzdGFuZC1pbiBmb3IgdGhlIDxiIGNsYXNzPSJwIj4yNTYtbnVtYmVyIGxhdGVudCBjb2RlPC9iPi4gVGhlIGRlY29kZXIgdHVybnMKICAgICAgd2hhdGV2ZXIgcG9pbnQgeW91IHBpY2sgaW50byBhIDI4w5cyOCBpbWFnZS48L3A+CiAgICAgIDxwPkxhbmQgPGI+aW5zaWRlIGEgY2x1c3RlcjwvYj4g4oaSIGEgY2xlYW4sIGNvbmZpZGVudCByZWNvbnN0cnVjdGlvbi4gTGFuZCA8Yj5iZXR3ZWVuIGNsdXN0ZXJzPC9iPiDihpIgYSBibHVycnkgYmxlbmQsCiAgICAgIGJlY2F1c2UgdGhhdCByZWdpb24gd2Fzbid0IHNlZW4gbXVjaCBpbiB0cmFpbmluZy48L3A+CiAgICAgIDxkaXYgY2xhc3M9ImtleWJveCI+U21vb3RoIG1vcnBoaW5nIGJldHdlZW4gY2x1c3RlcnMgaXMgdGhlIHNpZ25hdHVyZSBvZiBhIHdlbGwtb3JnYW5pemVkIGxhdGVudCBzcGFjZSDigJQgdGhlIHdob2xlIHJlYXNvbiBhdXRvZW5jb2RlcnMgYXJlIHVzZWZ1bCBmb3IgY29tcHJlc3Npb24gYW5kIGZlYXR1cmUgbGVhcm5pbmcuPC9kaXY+CiAgICA8L2Rpdj4KICA8L2Rpdj4KPC9kaXY+Cgo8Zm9vdGVyPkRyYWcgc2xvd2x5IGFjcm9zcyB0aGUgZ2FwIGJldHdlZW4gdHdvIGNsdXN0ZXJzIHRvIHNlZSBvbmUgZ2FybWVudCBkaXNzb2x2ZSBpbnRvIGFub3RoZXIuPC9mb290ZXI+CjwvZGl2PgoKPHNjcmlwdD4KKGZ1bmN0aW9uKCl7CiJ1c2Ugc3RyaWN0IjsKY29uc3QgJD1pZD0+ZG9jdW1lbnQuZ2V0RWxlbWVudEJ5SWQoaWQpOwpjb25zdCBjdj0kKCdsYXRlbnQnKSwgZz1jdi5nZXRDb250ZXh0KCcyZCcpOwoKLy8gY2x1c3RlciBwcm90b3R5cGVzOiBuYW1lLCBjb2xvciwgY2VudGVyKG5vcm1hbGl6ZWQgMC4uMSksIHNoYXBlLWZuIGtleQpjb25zdCBDTFVTVD1bCiAge25tOidULXNoaXJ0JywgYzonI0U4ODIxRicsIHg6MC4yMix5OjAuMzAsIGtpbmQ6J3RzaGlydCd9LAogIHtubTonVHJvdXNlcicsIGM6JyM2QzVDRTcnLCB4OjAuNzgseTowLjI0LCBraW5kOid0cm91c2VyJ30sCiAge25tOidQdWxsb3ZlcicsYzonIzBFOUM4RicsIHg6MC4zMCx5OjAuNzIsIGtpbmQ6J3B1bGxvdmVyJ30sCiAge25tOidTbmVha2VyJywgYzonI2MwNDY4ZicsIHg6MC43NCx5OjAuNzQsIGtpbmQ6J3NuZWFrZXInfSwKICB7bm06J0JhZycsICAgICBjOicjM2I3ZGQ4JywgeDowLjUyLHk6MC41MCwga2luZDonYmFnJ30sCl07CgovLyBzY2F0dGVyIHBvaW50cyBhcm91bmQgZWFjaCBjbHVzdGVyCmNvbnN0IFBUUz1bXTsKQ0xVU1QuZm9yRWFjaCgoY2wsY2kpPT57IGZvcihsZXQgaT0wO2k8NDA7aSsrKXsKICBjb25zdCBhPU1hdGgucmFuZG9tKCkqTWF0aC5QSSoyLCByPU1hdGgucmFuZG9tKCkqMC4wNzsKICBQVFMucHVzaCh7eDpjbC54K01hdGguY29zKGEpKnIsIHk6Y2wueStNYXRoLnNpbihhKSpyKjEuMSwgYzpjbC5jfSk7Cn19KTsKCmxldCBtej17eDowLjIyLHk6MC4zMH07IC8vIG1hcmtlciAoc3RhcnQgb24gVC1zaGlydCkKCmZ1bmN0aW9uIHRvUHgocCl7cmV0dXJuIFtwLngqY3Yud2lkdGgsIHAueSpjdi5oZWlnaHRdO30KZnVuY3Rpb24gZHJhd0xhdGVudCgpewogIGcuY2xlYXJSZWN0KDAsMCxjdi53aWR0aCxjdi5oZWlnaHQpOwogIC8vIHNvZnQgY2x1c3RlciBoYWxvcwogIENMVVNULmZvckVhY2goY2w9Pntjb25zdFtweCxweV09dG9QeChjbCk7CiAgICBjb25zdCBncmQ9Zy5jcmVhdGVSYWRpYWxHcmFkaWVudChweCxweSw0LHB4LHB5LDcwKTsKICAgIGdyZC5hZGRDb2xvclN0b3AoMCxjbC5jKyczMycpO2dyZC5hZGRDb2xvclN0b3AoMSxjbC5jKycwMCcpOwogICAgZy5maWxsU3R5bGU9Z3JkO2cuYmVnaW5QYXRoKCk7Zy5hcmMocHgscHksNzAsMCw3KTtnLmZpbGwoKTt9KTsKICAvLyBwb2ludHMKICBQVFMuZm9yRWFjaChwPT57Y29uc3RbcHgscHldPXRvUHgocCk7Zy5maWxsU3R5bGU9cC5jKydhYSc7Zy5iZWdpblBhdGgoKTtnLmFyYyhweCxweSwzLjIsMCw3KTtnLmZpbGwoKTt9KTsKICAvLyBjbHVzdGVyIGxhYmVscwogIENMVVNULmZvckVhY2goY2w9Pntjb25zdFtweCxweV09dG9QeChjbCk7Zy5maWxsU3R5bGU9Y2wuYztnLmZvbnQ9J2JvbGQgMTJweCBzYW5zLXNlcmlmJztnLnRleHRBbGlnbj0nY2VudGVyJzsKICAgIGcuZmlsbFRleHQoY2wubm0scHgscHktNTgpO30pOwogIC8vIG1hcmtlcgogIGNvbnN0W214LG15XT10b1B4KG16KTsKICBnLmZpbGxTdHlsZT0nIzZDNUNFNyc7Zy5iZWdpblBhdGgoKTtnLmFyYyhteCxteSw5LDAsNyk7Zy5maWxsKCk7CiAgZy5zdHJva2VTdHlsZT0nI2ZmZic7Zy5saW5lV2lkdGg9MztnLnN0cm9rZSgpOwogIGcuc3Ryb2tlU3R5bGU9JyM2QzVDRTcnO2cubGluZVdpZHRoPTI7Zy5iZWdpblBhdGgoKTtnLmFyYyhteCxteSwxNSwwLDcpO2cuc3Ryb2tlKCk7Cn0KCi8vIHNoYXBlIGdlbmVyYXRvcnMgaW4gWzAsMV0gb3ZlciBhIDI4eDI4IGdyaWQKZnVuY3Rpb24gc2hhcGVWYWwoa2luZCx4LHkpewogIGNvbnN0IGN4PTEzLjU7CiAgc3dpdGNoKGtpbmQpewogICAgY2FzZSAndHNoaXJ0Jzp7CiAgICAgIGNvbnN0IGJvZHk9KHk+OSYmeTwyNSYmTWF0aC5hYnMoeC1jeCk8Nyk7CiAgICAgIGNvbnN0IHNsPSh5PjkmJnk8MTUmJih4PjImJng8OHx8eD4yMCYmeDwyNikpOwogICAgICBjb25zdCBuZWNrPSh5Pj05JiZ5PDEyJiZNYXRoLmFicyh4LWN4KTwzKTsKICAgICAgcmV0dXJuIChib2R5fHxzbCkmJiFuZWNrPzAuODU6MC4wNTsKICAgIH0KICAgIGNhc2UgJ3Ryb3VzZXInOnsKICAgICAgY29uc3QgbGVnPSh5PjUmJnk8MjYmJihNYXRoLmFicyh4LTkpPDN8fE1hdGguYWJzKHgtMTgpPDMpKTsKICAgICAgY29uc3Qgd2Fpc3Q9KHk+PTUmJnk8OCYmeD42JiZ4PDIyKTsKICAgICAgcmV0dXJuIChsZWd8fHdhaXN0KT8wLjg1OjAuMDU7CiAgICB9CiAgICBjYXNlICdwdWxsb3Zlcic6ewogICAgICBjb25zdCBib2R5PSh5PjcmJnk8MjUmJk1hdGguYWJzKHgtY3gpPDgpOwogICAgICBjb25zdCBzbD0oeT43JiZ5PDE5JiYoeD4xJiZ4PDd8fHg+MjEmJng8MjcpKTsKICAgICAgcmV0dXJuIChib2R5fHxzbCk/MC44OjAuMDU7CiAgICB9CiAgICBjYXNlICdzbmVha2VyJzp7CiAgICAgIGNvbnN0IHNvbGU9KHk+MTkmJnk8MjQmJng+MyYmeDwyNSk7CiAgICAgIGNvbnN0IHRvcD0oeT4xMiYmeTwyMCYmeD4zJiZ4PDE5ICYmICh5LTEyKT4oeC0zKSotMC40KTsKICAgICAgcmV0dXJuIChzb2xlfHx0b3ApPzAuODI6MC4wNTsKICAgIH0KICAgIGNhc2UgJ2JhZyc6ewogICAgICBjb25zdCBib2R5PSh5PjExJiZ5PDI1JiZ4PjYmJng8MjIpOwogICAgICBjb25zdCBoYW5kbGU9KHk+NiYmeTwxMyYmKE1hdGguYWJzKHgtMTApPDEuNXx8TWF0aC5hYnMoeC0xOCk8MS41KSk7CiAgICAgIHJldHVybiAoYm9keXx8aGFuZGxlKT8wLjg6MC4wNTsKICAgIH0KICB9CiAgcmV0dXJuIDAuMDU7Cn0KLy8gd2VpZ2h0IG9mIGVhY2ggY2x1c3RlciBieSBpbnZlcnNlIGRpc3RhbmNlIChmb3Igc21vb3RoIG1vcnBoKQpmdW5jdGlvbiB3ZWlnaHRzKCl7CiAgY29uc3Qgdz1DTFVTVC5tYXAoY2w9Pntjb25zdCBkeD1jbC54LW16LngsZHk9Y2wueS1tei55O2NvbnN0IGQ9TWF0aC5zcXJ0KGR4KmR4K2R5KmR5KTtyZXR1cm4gMS9NYXRoLnBvdyhkKzAuMDIsNCk7fSk7CiAgY29uc3Qgcz13LnJlZHVjZSgoYSxiKT0+YStiLDApO3JldHVybiB3Lm1hcCh2PT52L3MpOwp9CmNvbnN0IHJjPSQoJ3JlY29uJykuZ2V0Q29udGV4dCgnMmQnKTsgcmMuaW1hZ2VTbW9vdGhpbmdFbmFibGVkPWZhbHNlOwpmdW5jdGlvbiBkcmF3UmVjb24oKXsKICBjb25zdCB3PXdlaWdodHMoKTsgY29uc3QgTj0yODsgY29uc3QgaW1nPXJjLmNyZWF0ZUltYWdlRGF0YShOLE4pLGQ9aW1nLmRhdGE7CiAgbGV0IGJsdXJBbXQ9MDsKICAvLyBibHVyIGdyb3dzIHdpdGggaG93ICJiZXR3ZWVuIiB3ZSBhcmUgKG1heCB3ZWlnaHQgbG93ID0gYW1iaWd1b3VzKQogIGNvbnN0IG1heHc9TWF0aC5tYXgoLi4udyk7IGJsdXJBbXQgPSBtYXh3PDAuNj8xOjA7CiAgZm9yKGxldCB5PTA7eTxOO3krKylmb3IobGV0IHg9MDt4PE47eCsrKXsKICAgIGxldCB2PTA7IENMVVNULmZvckVhY2goKGNsLGNpKT0+e3YrPXdbY2ldKnNoYXBlVmFsKGNsLmtpbmQseCx5KTt9KTsKICAgIGlmKGJsdXJBbXQpeyAvLyBjcnVkZSBibGVuZCBhbHJlYWR5IHNtb290aHM7IGFkZCBzbGlnaHQgbm9pc2UgdG8gcmVhZCBhcyAidW5jZXJ0YWluIgogICAgICB2Kz0oTWF0aC5yYW5kb20oKS0wLjUpKjAuMDU7CiAgICB9CiAgICB2PU1hdGgubWF4KDAsTWF0aC5taW4oMSx2KSk7CiAgICBjb25zdCBweD1NYXRoLnJvdW5kKDI1NS12KjIxMCk7CiAgICBkWyh5Kk4reCkqNF09cHg7ZFsoeSpOK3gpKjQrMV09cHg7ZFsoeSpOK3gpKjQrMl09cHg7ZFsoeSpOK3gpKjQrM109MjU1OwogIH0KICByYy5wdXRJbWFnZURhdGEoaW1nLDAsMCk7CiAgLy8gbGFiZWwgPSBkb21pbmFudCBjbHVzdGVyLCBvciAiYmxlbmQiCiAgY29uc3QgaWR4PXcuaW5kZXhPZihNYXRoLm1heCguLi53KSk7CiAgJCgncmxhYmVsJykudGV4dENvbnRlbnQgPSBtYXh3PjAuNjIgPyBDTFVTVFtpZHhdLm5tIDogJ2JsZW5kIG9mICcrQ0xVU1RbaWR4XS5ubSsnIOKApic7CiAgJCgncmxhYmVsJykuc3R5bGUuY29sb3IgPSBDTFVTVFtpZHhdLmM7CiAgJCgnY29vcmRzJykudGV4dENvbnRlbnQ9YHog4omIICgkeyhtei54KjItMSkudG9GaXhlZCgyKX0sICR7KDEtbXoueSoyKS50b0ZpeGVkKDIpfSlgOwp9CmZ1bmN0aW9uIHJlbmRlcigpeyBkcmF3TGF0ZW50KCk7IGRyYXdSZWNvbigpOyBwb3N0SCgpOyB9CgovLyBkcmFnCmZ1bmN0aW9uIHNldEZyb21FdmVudChlKXsKICBjb25zdCByPWN2LmdldEJvdW5kaW5nQ2xpZW50UmVjdCgpOwogIG16Lng9TWF0aC5tYXgoMCxNYXRoLm1pbigxLChlLmNsaWVudFgtci5sZWZ0KS9yLndpZHRoKSk7CiAgbXoueT1NYXRoLm1heCgwLE1hdGgubWluKDEsKGUuY2xpZW50WS1yLnRvcCkvci5oZWlnaHQpKTsKICByZW5kZXIoKTsKfQpsZXQgZHJhZz1mYWxzZTsKY3YuYWRkRXZlbnRMaXN0ZW5lcigncG9pbnRlcmRvd24nLGU9PntkcmFnPXRydWU7Y3Yuc2V0UG9pbnRlckNhcHR1cmUoZS5wb2ludGVySWQpO3NldEZyb21FdmVudChlKTt9KTsKY3YuYWRkRXZlbnRMaXN0ZW5lcigncG9pbnRlcm1vdmUnLGU9PntpZihkcmFnKXNldEZyb21FdmVudChlKTt9KTsKY3YuYWRkRXZlbnRMaXN0ZW5lcigncG9pbnRlcnVwJywoKT0+e2RyYWc9ZmFsc2U7fSk7CgovLyBsZWdlbmQgKyBqdW1wIGNoaXBzCmNvbnN0IGxlZz0kKCdsZWdlbmQnKSwgY2hpcHM9JCgnY2hpcHMnKTsKQ0xVU1QuZm9yRWFjaCgoY2wsaSk9PnsKICBjb25zdCBzPWRvY3VtZW50LmNyZWF0ZUVsZW1lbnQoJ3NwYW4nKTtzLmlubmVySFRNTD1gPGkgY2xhc3M9ImRvdCIgc3R5bGU9ImJhY2tncm91bmQ6JHtjbC5jfSI+PC9pPiR7Y2wubm19YDtsZWcuYXBwZW5kQ2hpbGQocyk7CiAgY29uc3QgYj1kb2N1bWVudC5jcmVhdGVFbGVtZW50KCdidXR0b24nKTtiLnRleHRDb250ZW50PWNsLm5tO2Iub25jbGljaz0oKT0+e216PXt4OmNsLngseTpjbC55fTtyZW5kZXIoKTt9O2NoaXBzLmFwcGVuZENoaWxkKGIpOwp9KTsKY29uc3QgYmI9ZG9jdW1lbnQuY3JlYXRlRWxlbWVudCgnYnV0dG9uJyk7YmIudGV4dENvbnRlbnQ9J+KGpiBiZXR3ZWVuJztiYi5zdHlsZS5iYWNrZ3JvdW5kPSd2YXIoLS10ZWFsLXNvZnQpJztiYi5zdHlsZS5jb2xvcj0ndmFyKC0tdGVhbCknOwpiYi5vbmNsaWNrPSgpPT57bXo9e3g6KENMVVNUWzBdLngrQ0xVU1RbMV0ueCkvMix5OihDTFVTVFswXS55K0NMVVNUWzFdLnkpLzJ9O3JlbmRlcigpO307Y2hpcHMuYXBwZW5kQ2hpbGQoYmIpOwoKY29uc3QgZ3JpZD0kKCdncmlkJyksIHdyYXA9ZG9jdW1lbnQucXVlcnlTZWxlY3RvcignLndyYXAnKTsKZnVuY3Rpb24gcmVsYXlvdXQoKXsgZ3JpZC5jbGFzc0xpc3QudG9nZ2xlKCdpcy1uYXJyb3cnLCB3aW5kb3cuaW5uZXJXaWR0aDw4MjApOyB9CmZ1bmN0aW9uIHBvc3RIKCl7IGNvbnN0IGg9d3JhcD9NYXRoLmNlaWwod3JhcC5nZXRCb3VuZGluZ0NsaWVudFJlY3QoKS5oZWlnaHQpKzM0OmRvY3VtZW50LmJvZHkub2Zmc2V0SGVpZ2h0OwogIGlmKHdpbmRvdy5wYXJlbnQhPT13aW5kb3cpIHdpbmRvdy5wYXJlbnQucG9zdE1lc3NhZ2Uoe3R5cGU6J2FlLWZyYW1lLWhlaWdodCcsaGVpZ2h0Omh9LCcqJyk7IH0KZnVuY3Rpb24gdXBkYXRlKCl7IHJlbGF5b3V0KCk7IHBvc3RIKCk7IH0Kd2luZG93LmFkZEV2ZW50TGlzdGVuZXIoJ2xvYWQnLHVwZGF0ZSk7IHdpbmRvdy5hZGRFdmVudExpc3RlbmVyKCdyZXNpemUnLHVwZGF0ZSk7CmlmKHdpbmRvdy5SZXNpemVPYnNlcnZlcikgbmV3IFJlc2l6ZU9ic2VydmVyKHBvc3RIKS5vYnNlcnZlKHdyYXB8fGRvY3VtZW50LmJvZHkpOwpyZWxheW91dCgpOyByZW5kZXIoKTsKc2V0VGltZW91dCh1cGRhdGUsMjAwKTsgc2V0VGltZW91dCh1cGRhdGUsNzAwKTsKfSkoKTsKPC9zY3JpcHQ+CjwvYm9keT4KPC9odG1sPgo="

HTML('''
<iframe id="latent-frame"
        src="data:text/html;base64,''' + _html_b64 + '''"
        style="width:100%; height:760px; border:1px solid #DDE5F2;
               border-radius:16px; box-shadow:0 8px 24px rgba(108,92,231,.12);
               display:block;"
        loading="lazy" title="Latent space explorer"></iframe>
<script>
(function(){
  function onMsg(e){
    if (e.data && e.data.type === "ae-frame-height") {
      var f = document.getElementById("latent-frame");
      if (f) f.style.height = (e.data.height) + "px";
    }
  }
  window.addEventListener("message", onMsg);
})();
</script>
''')


---

# Summary

---

## Key Concepts Learned

### What is an Autoencoder?
- A neural network that learns to **compress** and **reconstruct** data
- Consists of an **encoder** (compress) and **decoder** (expand)
- The **bottleneck** forces learning of essential features

### Architecture Components

| Component | Purpose | Implementation |
|-----------|---------|----------------|
| **Encoder** | Compress input | Conv layers with stride=2 |
| **Bottleneck** | Compressed representation | Smallest layer in the middle |
| **Decoder** | Reconstruct output | Upsample + Conv layers |
| **Sigmoid** | Bound output to [0,1] | Final activation |

### Deconvolution (Upsampling)
- Opposite of convolution with stride
- We use: **Upsample + Conv** (stable, no artifacts)
- Alternative: `ConvTranspose2d` (can cause checkerboard artifacts)

### Training Autoencoders
- **Loss function**: MSE (Mean Squared Error)
- **Target**: The input image itself!
- **Goal**: Minimize difference between input and reconstruction

### Applications
- **Dimensionality reduction** - Like PCA but non-linear
- **Denoising** - Train with noisy input, clean target
- **Anomaly detection** - High reconstruction error = unusual
- **Feature learning** - Encoder learns useful representations
- **Generative models** - VAEs extend this to generate new data

---

## What's Next?

In future notebooks, we'll explore:
- **Variational Autoencoders (VAEs)** - Generate new images
- **Deeper architectures** - Better reconstructions
- **Different loss functions** - Perceptual loss, adversarial loss
- **Applications** - Denoising, super-resolution, style transfer

---